# 01 · Technology Acceptance Model (TAM) with PLS-SEM

*SEM Lab Colab collection, notebook 1 of 5. Data: **SYNTHETIC** (generated in this
notebook). Estimation: SEM Lab's own engine (Python), no R required.*

**Attribution.** The model follows the core of the Technology Acceptance Model: Davis, F. D.
(1989). Perceived usefulness, perceived ease of use, and user acceptance of information
technology. *MIS Quarterly, 13*(3), 319–340. The data are **synthetic teaching data** generated
by SEM Lab from a stated population model; they do not reproduce any published study.
This notebook is the Python companion of SEM Lab's example *"Technology acceptance (TAM) —
mediation"* and uses byte-identical data.

### Learning objectives

After this notebook you can

1. specify a reflective measurement model and check indicator loadings, reliability
   (Cronbach's α, ρA, ρC), convergent validity (AVE) and discriminant validity (HTMT);
2. estimate structural paths with PLS-SEM and judge them with **bootstrap** confidence intervals;
3. decompose a total effect into a direct and an indirect (mediated) effect, and test the
   indirect effect with a bootstrap interval;
4. report the results in tables and figures, and export them.

### Theory and *illustrative* hypotheses

TAM proposes that people intend to use a technology when they find it useful (perceived
usefulness, **PU**) and easy to use (perceived ease of use, **PEOU**), and that ease of use also
makes a system appear more useful. In this notebook:

| | Hypothesis (illustrative) | Expected sign |
|---|---|---|
| H1 | PEOU → PU | + |
| H2 | PU → BI (behavioural intention) | + |
| H3 | PEOU → BI (direct effect) | + |
| H4 | PU mediates the effect of PEOU on BI (indirect effect PEOU → PU → BI) | + |

The hypotheses are illustrative: they give us something to test. Because the data are
synthetic and cross-sectional, the results can never confirm TAM, and the arrows express the
model's assumptions, **not** demonstrated causal effects.

## 1. Setup: dependencies and SEM Lab's engine

**Run every cell from top to bottom** (Colab: *Runtime → Run all*). The notebook needs no
uploads, no Google Drive, no tokens and no internet access beyond what Colab already has.

**Dependencies.** The notebook uses numpy, scipy, pandas and matplotlib. Google Colab
preinstalls all four, so this notebook deliberately does **not** `pip install` them: forcing a
reinstall of numpy or scipy on Colab replaces libraries that are already imported and makes
Colab ask for a runtime restart. Instead, the next cell prints the versions it finds and checks
them against the minimum versions SEM Lab's engine supports (the engine itself also warns at
import if scipy is too old).

Tested environment (CI runs this notebook headless on it): Python 3.12, numpy
2.5.5, scipy 1.18.1, pandas 3.1.1, matplotlib
3.11.2. To reproduce exactly that environment **outside Colab** (local Jupyter),
uncomment and run the `%pip` line in the next cell, then restart the kernel. On Colab, leave it
commented out.

**Estimation engine.** All estimates come from SEM Lab's own engine, the Python port of the
engine that runs in the SEM Lab app (`semlab_bundle.py`, version 0.1.0). It reproduces the
app's numbers (whole-result agreement about 1e-13 for PLS) and the R reference implementations
cSEM 0.7.1 and lavaan 0.7-3 within tested tolerances. The collapsed cell after the version check
writes the engine file; the cell after that verifies its SHA-256 against the version committed
in the SEM Lab repository and imports it. This is the **verified Python subset** of SEM
Studio: nothing in this notebook calls R.

In [ ]:
# Reproduce the tested environment exactly (local Jupyter only; on Colab this would force a restart):
# %pip install numpy==2.5.5 scipy==1.18.1 pandas==3.1.1 matplotlib==3.11.2

import re
import sys

import matplotlib
import numpy
import pandas
import scipy

MINIMUM = {"python": "3.10", "numpy": "1.22.4", "scipy": "1.12", "pandas": "1.5", "matplotlib": "3.5"}
TESTED = {"python": "3.12", "numpy": "2.5.5", "scipy": "1.18.1",
          "pandas": "3.1.1", "matplotlib": "3.11.2"}
FOUND = {"python": sys.version.split()[0], "numpy": numpy.__version__, "scipy": scipy.__version__,
         "pandas": pandas.__version__, "matplotlib": matplotlib.__version__}
IN_COLAB = "google.colab" in sys.modules


def version_tuple(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:3])


too_old = []
for pkg, have in FOUND.items():
    ok = version_tuple(have) >= version_tuple(MINIMUM[pkg])
    print(f"{pkg:<11} {have:<10} minimum {MINIMUM[pkg]:<7} tested {TESTED[pkg]:<8} {'ok' if ok else 'TOO OLD'}")
    if not ok:
        too_old.append(pkg)
print("Running on Google Colab" if IN_COLAB else "Running outside Colab")
if too_old:
    raise RuntimeError(f"Please upgrade {', '.join(too_old)} (see the minimum versions above).")

In [ ]:
%%writefile semlab_bundle.py
#@title SEM Lab engine: writes semlab_bundle.py (about 5,600 lines; click to expand)
"""semlab — SEM Lab's verified estimation engine in Python.

Mirrors @sem-lab/engine 0.1.0 (the TypeScript engine that runs in the SEM Lab app,
packages/engine): PLS-SEM and consistent PLS (cSEM 0.7.1 semantics: path / centroid /
factorial inner weighting, Mode A / Mode B, all quality criteria, effects incl. specific
indirect effects, IPMA, cSEM::verify admissibility), SEM Lab's seeded bootstrap (R's
Mersenne-Twister and sample.int(), bit-exact), and CB-SEM by ML / MLR, complete data or FIML
(lavaan 0.7 sem() defaults; Yuan-Bentler scaled and robust fit indices; RMSEA CI with R's
uniroot). Python scripts and notebooks exported from the app reproduce the app's numbers
with this package; the tested tolerances are in README.md.

Provenance: a line-by-line port of packages/engine/src (TypeScript) of the SEM Lab
repository, verified against it and against the committed R (cSEM 0.7.1, lavaan 0.7-3)
reference results; algorithms re-implemented from their documented definitions (no code from
cSEM, lavaan, plspm or semopy). Licence: MIT (see the LICENSE file of the SEM Lab
repository). Runtime dependencies: numpy, scipy (BSD-3-Clause).

    from semlab import estimate, build_analysis_data, read_csv, results_tables
    columns, rows = read_csv("data.csv")
    data = build_analysis_data(model, columns, rows, analysis["missingData"])
    out = estimate(model, analysis, data)          # {'ok': True, 'result': {...}}
    tables = results_tables(out["result"], model)  # plain dict tables

Single-file bundle of the semlab package (generated by
packages/python-engine/scripts/bundle.py; do not edit). Import it as a module next to
an exported script: ``import semlab_bundle as semlab``.
Package source sha256: 86e7c204946429a9ad966e12cdc8904b344b814e80488318a6b56cce28884b6e

MIT License

Copyright (c) 2026 Darko Etinger

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""
# ruff: noqa
# fmt: off
from datetime import datetime, timezone
from decimal import ROUND_HALF_UP, Decimal, localcontext
import math
import platform
import re
import warnings
from scipy.linalg import solve_triangular
from scipy.special import betainc as _betainc
from scipy.stats import t as _student_t
import numpy as np
import scipy

# =============================================================================================
# semlab/meta.py
# =============================================================================================
# Package version and result provenance shared by the PLS and CB engines.

__version__ = "0.1.0"
#: The TypeScript engine version this implementation mirrors.
MIRRORS_ENGINE_VERSION = "0.1.0"


def _version_tuple(v):
    out = []
    for part in v.split(".")[:2]:
        digits = "".join(ch for ch in part if ch.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


#: Tested lower bounds (pyproject.toml). The single-file bundle bypasses pip, so check here.
MIN_NUMPY = (1, 22)
MIN_SCIPY = (1, 12)
if _version_tuple(np.__version__) < MIN_NUMPY or _version_tuple(scipy.__version__) < MIN_SCIPY:
    warnings.warn(
        f"semlab is tested with numpy >= 1.22.4 and scipy >= 1.12 (found numpy {np.__version__}, "
        f"scipy {scipy.__version__}); with older scipy, Student-t p values can differ from the app by "
        "up to ~1e-11. Upgrade with: pip install -U numpy scipy",
        RuntimeWarning,
        stacklevel=2,
    )


def iso_now():
    """new Date().toISOString()."""
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def provenance(call, started_at, package_versions):
    versions = dict(package_versions)
    versions["semlab"] = __version__
    versions["numpy"] = np.__version__
    versions["scipy"] = scipy.__version__
    return {
        "engine": "browser",
        "packageVersions": versions,
        "pythonVersion": platform.python_version(),
        "call": call,
        "startedAt": started_at,
        "finishedAt": iso_now(),
    }


# =============================================================================================
# semlab/errors.py
# =============================================================================================
# Stable engine error codes and the handled EngineError (port of packages/engine/src/shared/errors.ts).

ENGINE_ERROR_CODES = (
    "REQUEST_INVALID",
    "DATA_COLUMN_MISSING",
    "DATA_NOT_NUMERIC",
    "DATA_INSUFFICIENT_ROWS",
    "DATA_ZERO_VARIANCE",
    "MODEL_NOT_IDENTIFIED",
    "ENGINE_NONCONVERGENCE",
    "ENGINE_INADMISSIBLE",
    "CAPABILITY_UNSUPPORTED",
    "CANCELLED",
    "TIMEOUT",
    "ENGINE_INTERNAL",
)


class EngineError(Exception):
    """A handled engine error. Messages and details may name model ids, column names and
    counts, never a data value."""

    def __init__(self, code, message, details=None):
        super().__init__(message)
        self.code = code
        self.message = message
        self.details = details

    def to_info(self):
        if self.details:
            return {"code": self.code, "message": self.message, "details": self.details}
        return {"code": self.code, "message": self.message}


def invalid(message, details=None):
    return EngineError("REQUEST_INVALID", message, details)


# =============================================================================================
# semlab/jsnum.py
# =============================================================================================
# JavaScript number formatting (String(x), toExponential, toFixed, toPrecision), so that
# messages and the rounded bootstrap draws are character-for-character those of the
# TypeScript engine. JS rounds exact ties away from zero ("pick the larger n"); Python's
# format() rounds half-even, hence the Decimal implementation.

def js_str(x):
    """Number.prototype.toString() / String(x) for a finite or non-finite double."""
    if isinstance(x, bool):
        return "true" if x else "false"
    if isinstance(x, int):
        return str(x)
    if x != x:
        return "NaN"
    if x == math.inf:
        return "Infinity"
    if x == -math.inf:
        return "-Infinity"
    if x == 0:
        return "0"
    sign = "-" if x < 0 else ""
    # shortest round-trip digits (repr) -> digits d1..dk and decimal exponent n: 0.d1..dk x 10^n
    t = Decimal(repr(abs(float(x)))).normalize().as_tuple()
    digits = "".join(str(d) for d in t.digits)
    k = len(digits)
    n = t.exponent + k
    if k <= n <= 21:
        return sign + digits + "0" * (n - k)
    if 0 < n <= 21:
        return sign + digits[:n] + "." + digits[n:]
    if -6 < n <= 0:
        return sign + "0." + "0" * (-n) + digits
    e = n - 1
    es = ("+" if e >= 0 else "-") + str(abs(e))
    if k == 1:
        return sign + digits + "e" + es
    return sign + digits[0] + "." + digits[1:] + "e" + es


def _nonfinite(x):
    if x != x:
        return "NaN"
    return "Infinity" if x > 0 else "-Infinity"


def js_to_exponential(x, f):
    """Number.prototype.toExponential(f)."""
    x = float(x)
    if not math.isfinite(x):
        return _nonfinite(x)
    sign = "-" if x < 0 else ""
    a = abs(x)
    if a == 0:
        return sign + "0" + ("." + "0" * f if f > 0 else "") + "e+0"
    with localcontext() as ctx:
        ctx.prec = 2000
        d = Decimal(a)
        e = d.adjusted()
        q = d.scaleb(-e).quantize(Decimal(1).scaleb(-f), rounding=ROUND_HALF_UP)
        if q >= 10:
            e += 1
            q = d.scaleb(-e).quantize(Decimal(1).scaleb(-f), rounding=ROUND_HALF_UP)
        s = format(q, "f")
    return sign + s + "e" + ("+" if e >= 0 else "-") + str(abs(e))


def js_to_fixed(x, f):
    """Number.prototype.toFixed(f) for |x| < 1e21."""
    x = float(x)
    if not math.isfinite(x):
        return _nonfinite(x)
    if abs(x) >= 1e21:
        return js_str(x)
    sign = "-" if x < 0 else ""
    with localcontext() as ctx:
        ctx.prec = 2000
        q = Decimal(abs(x)).quantize(Decimal(1).scaleb(-f), rounding=ROUND_HALF_UP)
        s = format(q, "f")
    if sign and float(q) == 0:
        # JS: (-0.0001).toFixed(2) === '-0.00'
        return "-" + s
    return sign + s


def js_signif(x, digits):
    """Number(x.toPrecision(digits)) (0 stays 0): round to `digits` significant digits."""
    if x == 0 or not math.isfinite(x):
        return 0.0 if x == 0 else x
    a = abs(x)
    with localcontext() as ctx:
        ctx.prec = 2000
        d = Decimal(a)
        e = d.adjusted()
        q = d.quantize(Decimal(1).scaleb(e - digits + 1), rounding=ROUND_HALF_UP)
        v = float(q)
    return -v if x < 0 else v


# =============================================================================================
# semlab/linalg.py
# =============================================================================================
# Small dense linear-algebra helpers for the PLS engine (port of packages/engine/src/linalg).
#
# Numerics follow the R functions the reference engine uses:
#  - ``inverse_r`` / ``solve_r`` mirror base::solve(): LU with partial pivoting and the same
#    refusal rule (error when the reciprocal 1-norm condition number is below
#    .Machine$double.eps, "system is computationally singular");
#  - ``symmetric_eigenvalues`` mirrors eigen(symmetric = TRUE);
#  - ``is_positive_semi_definite`` mirrors cSEM:::isPositiveSemiDefinite (eigenvalues >= -1e-8);
#  - ``correlation`` mirrors stats::cor() for complete numeric data (Pearson, two-pass means).

DOUBLE_EPS = 2.220446049250313e-16


class NumericalError(Exception):
    """Raised where R itself would stop with an error (singular system, NaN input ...)."""


def _assert_finite(a):
    if not np.all(np.isfinite(a)):
        raise NumericalError("non-finite value in matrix")


def _norm1(a):
    # max column sum of |a|; NaN propagates (JS: if (!(s <= best)) best = s)
    s = np.sum(np.abs(a), axis=0)
    if np.any(np.isnan(s)):
        return math.nan
    return float(np.max(s)) if s.size else 0.0


def inverse_r(a):
    """Inverse with base::solve()'s singularity rule; raises NumericalError."""
    a = np.asarray(a, dtype=float)
    n = a.shape[0]
    if n == 0:
        return np.zeros((0, 0))
    _assert_finite(a)
    try:
        inv = np.linalg.inv(a)
    except np.linalg.LinAlgError as e:
        raise NumericalError("system is exactly singular") from e
    _assert_finite(inv)
    with np.errstate(all="ignore"):
        rcond = 1.0 / (_norm1(a) * _norm1(inv))
    if not (rcond >= DOUBLE_EPS):
        raise NumericalError("system is computationally singular")
    return inv


def solve_r(a, b):
    """solve(a, b) for a vector right-hand side, with base::solve()'s singularity rule."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = a.shape[0]
    if n == 0:
        return np.zeros(0)
    if n == 1:
        d = float(a[0, 0])
        if not math.isfinite(d) or d == 0 or not math.isfinite(float(b[0])):
            raise NumericalError("singular 1x1 system")
        return np.array([float(b[0]) / d])
    _assert_finite(a)
    if not np.all(np.isfinite(b)):
        raise NumericalError("non-finite right-hand side")
    try:
        lu_inv = np.linalg.inv(a)
        x = np.linalg.solve(a, b)
    except np.linalg.LinAlgError as e:
        raise NumericalError("system is exactly singular") from e
    with np.errstate(all="ignore"):
        rcond = 1.0 / (_norm1(a) * _norm1(lu_inv))
    if not (rcond >= DOUBLE_EPS):
        raise NumericalError("system is computationally singular")
    if not np.all(np.isfinite(x)):
        raise NumericalError("non-finite solution")
    return x


def symmetric_eigenvalues(a):
    """Eigenvalues of a symmetric matrix, descending."""
    a = np.asarray(a, dtype=float)
    if a.shape[0] == 0:
        return np.zeros(0)
    try:
        ev = np.linalg.eigvalsh(a)
    except np.linalg.LinAlgError as e:
        raise NumericalError("eigen decomposition failed") from e
    return ev[::-1]


def is_positive_semi_definite(a, tolerance=1e-8):
    a = np.asarray(a, dtype=float)
    if not np.all(np.isfinite(a)):
        return False
    try:
        ev = symmetric_eigenvalues(a)
    except NumericalError:
        return False
    return not bool(np.any(~np.isfinite(ev) | (ev < -tolerance)))


def symmetrize_from_upper(a):
    """R: x[lower.tri(x)] <- t(x)[lower.tri(x)] (in place; returns a)."""
    il = np.tril_indices(a.shape[0], -1)
    a[il] = a.T[il]
    return a


def r_mean(x):
    """mean(x) with R's two-pass refinement (s/n, then + mean(x - s/n))."""
    x = np.asarray(x, dtype=float)
    n = x.shape[0]
    m = np.sum(x) / n
    return float(m + np.sum(x - m) / n)


def r_sd(x):
    """stats::sd (divisor n - 1)."""
    x = np.asarray(x, dtype=float)
    n = x.shape[0]
    m = r_mean(x)
    d = x - m
    return math.sqrt(float(np.sum(d * d)) / (n - 1))


def correlation(X, rows=None):
    """Pearson correlation of the columns of X (n x p), over `rows` (0-based, repetitions
    allowed: a bootstrap resample). Returns (S, zero_variance_column_indices); S has an exact
    unit diagonal and off-diagonal values clamped to [-1, 1] (stats::cor)."""
    V = X if rows is None else X[rows]
    n, p = V.shape
    m = np.sum(V, axis=0) / n
    m = m + np.sum(V - m, axis=0) / n
    C = V - m
    ss = np.sum(C * C, axis=0)
    all_same = np.all(V == V[0:1, :], axis=0) if n > 0 else np.ones(p, dtype=bool)
    zero = [j for j in range(p) if all_same[j] or not (ss[j] > 0)]
    with np.errstate(all="ignore"):
        S = (C.T @ C) / np.sqrt(np.outer(ss, ss))
    np.clip(S, -1.0, 1.0, out=S)
    np.fill_diagonal(S, 1.0)
    # exact symmetry, as the reference computes the upper triangle once
    symmetrize_from_upper(S)
    return S, zero


def qr_rank(a, tol=1e-10):
    """Numerical rank with base::qr(x, tol)'s rule (Gram-Schmidt with limited pivoting)."""
    a = np.asarray(a, dtype=float)
    n = a.shape[0]
    if n == 0:
        return 0
    kept = []
    for j in range(a.shape[1]):
        c = a[:, j].copy()
        orig = math.hypot(*c)
        v = c.copy()
        for q in kept:
            d = float(np.dot(q, v))
            v = v - d * q
        nv = math.hypot(*v)
        if orig > 0 and nv >= tol * orig:
            kept.append(v / nv)
    return len(kept)


# =============================================================================================
# semlab/stats.py
# =============================================================================================
# Distribution and summary functions with R's definitions (port of packages/engine/src/shared/stats.ts):
#
#  - ``pt(t, df)`` / ``pt_lower_tail_abs(t, df)``: Student t CDF, P(T <= -|t|) computed in the
#    tail without cancellation: scipy.stats.t.sf, except near t = 0 (tail > 1/4), where scipy's
#    cephes stdtr loses ~1e-11 (df = 1, |t| = 1e-6) and the tail is taken as
#    1/2 - 1/2 I_y(1/2, df/2), y = t^2/(df + t^2) (scipy.special.betainc, no cancellation in y).
#    Equals R's stats::pt to 2.2e-16 absolute, 1.5e-13 relative in the tail (tests/test_stats.py);
#  - ``quantile7``: stats::quantile(x, probs, type = 7) including R's (1 - h)*x_lo + h*x_hi form;
#  - ``sample_sd``: stats::sd (divisor n - 1).

def _t_upper_tail(t_abs, df):
    """P(T > |t|) for T ~ t(df)."""
    s = float(_student_t.sf(t_abs, df))
    if s > 0.25:
        t2 = t_abs * t_abs
        return 0.5 - 0.5 * float(_betainc(0.5, df / 2, t2 / (df + t2)))
    return s


def pt(t, df):
    """Lower-tail Student t CDF P(T <= t), df > 0 (stats::pt(t, df))."""
    if t != t or not (df > 0):
        return math.nan
    if t == math.inf:
        return 1.0
    if t == -math.inf:
        return 0.0
    tail = _t_upper_tail(abs(t), df)
    return 1.0 - tail if t > 0 else tail


def pt_lower_tail_abs(t, df):
    """P(T <= -|t|) without cancellation (the bootstrap p-value building block)."""
    if t != t or not (df > 0):
        return math.nan
    if not math.isfinite(t):
        return 0.0
    return _t_upper_tail(abs(t), df)


def quantile7(x, probs):
    """stats::quantile(x, probs, type = 7, names = FALSE) for finite x."""
    xs = np.sort(np.asarray(x, dtype=float))
    n = xs.shape[0]
    out = []
    for p in probs:
        index = 1 + max(n - 1, 0) * p
        lo = math.floor(index)
        hi = math.ceil(index)
        qs = float(xs[lo - 1])
        xh = float(xs[hi - 1])
        if index > lo and xh != qs:
            h = index - lo
            out.append((1 - h) * qs + h * xh)
        else:
            out.append(qs)
    return out


def sample_sd(x):
    """stats::sd (divisor n - 1)."""
    x = np.asarray(x, dtype=float)
    n = x.shape[0]
    m = r_mean(x)
    d = x - m
    return math.sqrt(float(np.sum(d * d)) / (n - 1))


# =============================================================================================
# semlab/rng.py
# =============================================================================================
# Bit-exact port of R's default uniform generator and integer sampler, as selected by
#
#     set.seed(seed, kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
#
# (port of packages/engine/src/rng/r-mt.ts). Mirrors R's src/main/RNG.c: RNG_Init with the
# 50-fold LCG scramble, MT19937 MT_genrand, fixup() to the open interval (0, 1), and
# R_unif_index() with rbits() rejection sampling (R >= 3.6.0) for
# ``sample.int(n, size, replace = TRUE)``.
#
# The MT19937 recurrence and tempering are numpy's ``MT19937`` bit generator, whose state is set
# to exactly the state R's seeding produces (key = mt[0..623], pos = 624), so the 32-bit outputs
# are R's. ``unif_rand`` = y * 2^-32 with R's fixup. Integer sampling is vectorised: every
# rejection-sampling attempt consumes the same number of 16-bit chunks (one uniform each), so
# the uniform stream partitions into fixed-size attempts; attempts are drawn in rounds of exactly
# the number still known to be needed, so the generator never runs ahead of R's.
# Verified against packages/fixtures/reference-results/rng/r-mersenne-twister.json.

_N = 624
#: 1 / (2^32 - 1), R's i2_32m1
_I2_32M1 = 2.328306437080797e-10
#: 2^-32 as written in R's MT_genrand
_TWO_M32 = 2.3283064365386963e-10
_MASK32 = 0xFFFFFFFF


def _is_r_integer(seed):
    if isinstance(seed, bool):
        return False
    if isinstance(seed, (int, np.integer)):
        return -2147483647 <= int(seed) <= 2147483647
    if isinstance(seed, float) and math.isfinite(seed) and seed == math.floor(seed):
        return -2147483647 <= seed <= 2147483647
    return False


class RMersenneTwister:
    """R's Mersenne-Twister after set.seed(seed) (sample.kind = "Rejection")."""

    def __init__(self, seed):
        self._bg = np.random.MT19937(0)
        self.set_seed(seed)

    def set_seed(self, seed):
        """R's set.seed(seed) for kind "Mersenne-Twister". `seed` is an R integer (may be negative)."""
        if not _is_r_integer(seed):
            raise ValueError("seed must be an R integer")
        s = int(seed) & _MASK32
        # initial scrambling
        for _ in range(50):
            s = (69069 * s + 1) & _MASK32
        # dummy[0] (= mti) is seeded too and then overwritten by FixupSeeds(initial = TRUE)
        s = (69069 * s + 1) & _MASK32
        mt = np.empty(_N, dtype=np.uint32)
        for j in range(_N):
            s = (69069 * s + 1) & _MASK32
            mt[j] = s
        # FixupSeeds: an all-zero state cannot come from the LCG; mti = N forces a regeneration
        self._bg.state = {"bit_generator": "MT19937", "state": {"key": mt, "pos": _N}}

    # ---- raw stream ----------------------------------------------------------------------
    def _raw32(self, count):
        """`count` tempered 32-bit MT_genrand outputs (int64 array)."""
        if count <= 0:
            return np.zeros(0, dtype=np.int64)
        return np.asarray(self._bg.random_raw(int(count)), dtype=np.int64)

    def unif_rand_array(self, count):
        """unif_rand() `count` times: uniform draws in the open interval (0, 1), exactly as R."""
        x = self._raw32(count).astype(np.float64) * _TWO_M32
        x[x <= 0] = 0.5 * _I2_32M1
        x[1 - x <= 0] = 1 - 0.5 * _I2_32M1
        return x

    def unif_rand(self):
        return float(self.unif_rand_array(1)[0])

    def unif_index(self, dn):
        """R_unif_index(dn) for sample.kind = "Rejection": uniform integer in [0, dn) (scalar,
        written exactly as R's rbits()/R_unif_index())."""
        if dn <= 0:
            return 0
        bits = 0
        while 2**bits < dn:
            bits += 1
        while True:
            v = 0
            n = 0
            while n <= bits:
                v1 = math.floor(self.unif_rand() * 65536)
                v = 65536 * v + v1
                n += 16
            dv = v % (2**bits)
            if not (dn <= dv):
                return dv

    def sample_int_replace(self, n, size):
        """sample.int(n, size, replace = TRUE): 1-based indices (int32 array)."""
        if isinstance(n, bool) or not isinstance(n, (int, np.integer)) or n < 1 or n > 2147483647:
            raise ValueError("invalid n")
        return self._indices0(int(n), int(size)) + 1

    def _indices0(self, n, size):
        """R_unif_index(n) `size` times, vectorised (0-based)."""
        out = np.empty(size, dtype=np.int32)
        if size == 0:
            return out
        bits = 0
        while 2**bits < n:
            bits += 1
        chunks = bits // 16 + 1  # rbits(): for (n = 0; n <= bits; n += 16)
        mod = 1 << bits
        filled = 0
        while filled < size:
            need = size - filled  # every remaining index needs at least one more attempt
            raw = self._raw32(need * chunks).reshape(need, chunks)
            # floor(unif_rand() * 65536) == y >> 16 exactly (the fixup only touches y == 0)
            v = np.zeros(need, dtype=np.int64)
            for c in range(chunks):
                v = v * 65536 + (raw[:, c] >> 16)
            v %= mod
            acc = v[v < n]
            out[filled : filled + acc.shape[0]] = acc
            filled += acc.shape[0]
        return out


def r_set_seed(seed):
    """set.seed(seed, kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")."""
    return RMersenneTwister(seed)


def runif(rng, count):
    """runif(count) after the generator's current state."""
    return [float(v) for v in rng.unif_rand_array(count)]


def bootstrap_indices(seed, n, B, batch_rows=4_000_000):
    """The bootstrap resamples in R's order: draw b is ``sample.int(n, n, replace = TRUE) - 1``
    (0-based row indices) after set.seed(seed, "Mersenne-Twister", "Inversion", "Rejection").
    Yields int32 arrays; generated in batches of draws (one RNG stream, drawn sequentially)."""
    rng = RMersenneTwister(seed)
    per = max(1, batch_rows // max(n, 1))
    done = 0
    while done < B:
        m = min(per, B - done)
        block = rng._indices0(n, m * n).reshape(m, n)
        for b in range(m):
            yield block[b]
        done += m


# =============================================================================================
# semlab/effects.py
# =============================================================================================
# Structural-model graph and effects (port of packages/engine/src/pls/effects.ts):
# total = (I - B)^-1 - I, total indirect = total - B, specific indirect = product of the path
# coefficients along one directed chain of length >= 2 (at most 500 chains).

MAX_SPECIFIC_INDIRECT = 500


def _adjacency(k, edges):
    """Adjacency lists in node order; each list keeps the order of the paths."""
    adj = [[] for _ in range(k)]
    for e in edges:
        adj[e.source].append(e.target)
    return adj


def find_cycle(k, edges):
    """One directed cycle (node indices, first node repeated at the end) or None."""
    adj = _adjacency(k, edges)
    state = [0] * k
    stack = []
    found = []

    def visit(v):
        state[v] = 1
        stack.append(v)
        for w in adj[v]:
            if found:
                return
            if state[w] == 1:
                found.append(stack[stack.index(w) :] + [w])
                return
            if state[w] == 0:
                visit(w)
        stack.pop()
        state[v] = 2

    for v in range(k):
        if not found and state[v] == 0:
            visit(v)
    return found[0] if found else None


def all_paths(k, edges, src, tgt, limit=float("inf")):
    """All directed paths src -> tgt (node lists, source first) in a DAG, depth-first."""
    adj = _adjacency(k, edges)
    out = []

    def walk(v, trail):
        if len(out) >= limit:
            return
        for w in adj[v]:
            if w == tgt:
                out.append(trail + [w])
            else:
                walk(w, trail + [w])

    walk(src, [src])
    return out


class ConnectedPair:
    __slots__ = ("source", "target", "direct", "has_indirect", "specific")

    def __init__(self, source, target, direct, has_indirect, specific):
        self.source = source
        self.target = target
        self.direct = direct
        self.has_indirect = has_indirect
        self.specific = specific


class SpecificChain:
    __slots__ = ("nodes", "source", "target", "via")

    def __init__(self, nodes):
        self.nodes = nodes
        self.source = nodes[0]
        self.target = nodes[-1]
        self.via = nodes[1:-1]


class EffectsSpec:
    __slots__ = ("pairs", "specific", "truncated")

    def __init__(self, pairs, specific, truncated):
        self.pairs = pairs
        self.specific = specific
        self.truncated = truncated


def effects_spec(k, edges, max_specific=MAX_SPECIFIC_INDIRECT):
    """Every ordered pair with at least one directed path, in construct order."""
    pairs = []
    n_specific = 0
    truncated = False
    for s in range(k):
        for t in range(k):
            if s == t:
                continue
            ps = all_paths(k, edges, s, t)
            if not ps:
                continue
            long = [p for p in ps if len(p) >= 3]
            if n_specific + len(long) > max_specific:
                keep = max(0, max_specific - n_specific)
                truncated = truncated or keep < len(long)
                long = long[:keep]
            n_specific += len(long)
            pairs.append(
                ConnectedPair(
                    s,
                    t,
                    any(e.source == s and e.target == t for e in edges),
                    any(len(p) >= 3 for p in ps),
                    long,
                )
            )
    specific = [SpecificChain(nodes) for p in pairs for nodes in p.specific]
    return EffectsSpec(pairs, specific, truncated)


class EffectValues:
    __slots__ = ("total", "indirect", "specific")

    def __init__(self, total, indirect, specific):
        self.total = total
        self.indirect = indirect
        self.specific = specific


def compute_effects(k, edges, path_values, spec):
    """Effects for one vector of path coefficients (`path_values[i]` belongs to `edges[i]`)."""
    B = np.zeros((k, k))
    for e, v in zip(edges, path_values):
        B[e.target, e.source] = v
    eye = np.eye(k)
    inv = inverse_r(eye - B)
    T = inv - eye
    total = [float(T[p.target, p.source]) for p in spec.pairs]
    indirect = [float(T[p.target, p.source] - B[p.target, p.source]) for p in spec.pairs]
    specific = []
    for sp in spec.specific:
        v = 1.0
        for i in range(len(sp.nodes) - 1):
            v *= float(B[sp.nodes[i + 1], sp.nodes[i]])
        specific.append(v)
    return EffectValues(total, indirect, specific)


# =============================================================================================
# semlab/pls_core.py
# =============================================================================================
# PLS request validation and the PLS-PM estimation core (port of packages/engine/src/pls/spec.ts
# and core.ts).
#
# Validation: the checks, error codes and reasons of the R engine's validate_request /
# validate_ipma / load_job_data, applied to the in-memory analysis matrix the web app builds
# (indicator columns in model order, reverse scoring min + max - x ALREADY applied,
# listwise-complete rows).
#
# Estimation: the semantics of cSEM 0.7.1 ``csem()`` with .approach_weights = "PLS-PM",
# .approach_paths = "OLS", .conv_criterion = "diff_absolute", inner scheme path / centroid /
# factorial, .disattenuate = FALSE (classic) / TRUE with .PLS_approach_cf =
# "dist_squared_euclid" (PLSc), no dominant indicators, no starting values. Everything is
# computed from the Pearson indicator correlation matrix S.

ENGINE_ID = "browser"
_ID_PATTERN = re.compile(r"[A-Za-z0-9_][A-Za-z0-9_.:-]*")
_ID_PATTERN_SOURCE = "^[A-Za-z0-9_][A-Za-z0-9_.:-]*$"


# ---------------------------------------------------------------------------------------------
# spec


class SpecConstruct:
    __slots__ = ("id", "reflective", "mode_b", "indicator_ids", "indicators")

    def __init__(self, id, reflective, mode_b, indicator_ids, indicators):
        self.id = id
        self.reflective = reflective
        self.mode_b = mode_b
        self.indicator_ids = indicator_ids
        #: positions of the block's indicators in PlsSpec.indicators
        self.indicators = indicators


class SpecIndicator:
    __slots__ = ("id", "column", "reverse", "scale")

    def __init__(self, id, column, reverse, scale):
        self.id = id
        self.column = column
        self.reverse = reverse
        self.scale = scale  # (min, max) or None


class SpecPath:
    __slots__ = ("id", "source", "target")

    def __init__(self, id, source, target):
        self.id = id
        self.source = source
        self.target = target


class BootstrapSpec:
    __slots__ = ("reps", "level", "sides", "seed")

    def __init__(self, reps, level, sides, seed):
        self.reps = reps
        self.level = level
        self.sides = sides
        self.seed = seed


class IpmaSpec:
    __slots__ = ("target", "predecessors", "direct")

    def __init__(self, target, predecessors, direct):
        self.target = target
        self.predecessors = predecessors
        self.direct = direct


class PlsSpec:
    __slots__ = (
        "method",
        "scheme",
        "max_iter",
        "tol",
        "constructs",
        "indicators",
        "paths",
        "bootstrap",
        "ipma",
        "warnings",
    )

    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)


def _is_number(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool)


def _is_whole(x):
    return _is_number(x) and math.isfinite(x) and abs(x - round(x)) < 1e-9


def _check_id(x, what):
    if not isinstance(x, str) or len(x) > 128 or not _ID_PATTERN.fullmatch(x):
        raise invalid(f"{what} must be an id matching {_ID_PATTERN_SOURCE} (max 128 characters).")
    return x


def _get(d, key):
    """`d[key]` treating an absent key and JSON null alike (undefined)."""
    return d.get(key) if isinstance(d, dict) else None


def validate_request(model, an):
    """validate_request(): model + analysis -> PlsSpec, or a stable EngineError."""
    if an.get("engine") != ENGINE_ID:
        raise invalid(
            f"This is the {ENGINE_ID} engine; the request asks for engine `{an.get('engine')}`."
        )
    method = an.get("method")
    if method == "cb-sem":
        raise EngineError(
            "CAPABILITY_UNSUPPORTED",
            "CB-SEM is not estimated by the PLS engine (use the CB-SEM entry point).",
            {"method": method, "engine": ENGINE_ID},
        )
    if method not in ("pls-sem", "plsc"):
        raise invalid("analysis.method must be `pls-sem` or `plsc`.")
    if an.get("estimator") != "pls-pm":
        raise EngineError(
            "CAPABILITY_UNSUPPORTED",
            "The browser PLS engine implements estimator `pls-pm` only.",
            {"engine": ENGINE_ID},
        )
    if an.get("missingData") != "listwise":
        raise invalid(
            "analysis.missingData must be `listwise` for PLS methods (FIML is defined for CB-SEM only)."
        )
    if _get(an, "cb") is not None:
        raise invalid("analysis.cb is only valid for CB-SEM.")
    ipma_req = _get(an, "ipma")
    if ipma_req is not None and method != "pls-sem":
        raise EngineError(
            "CAPABILITY_UNSUPPORTED",
            "IPMA uses a classic PLS-SEM fit (method `pls-sem`); it is not computed from a PLSc fit (in Phase 0 a PLSc fit shifted importance by 0.086).",
            {"capabilityId": "ipma.construct", "method": method},
        )
    pls = _get(an, "pls")
    if not pls:
        raise invalid(
            "analysis.pls (innerWeighting, maxIterations, tolerance) is required for PLS methods."
        )
    scheme = pls.get("innerWeighting")
    if scheme not in ("path", "centroid", "factorial"):
        raise invalid("analysis.pls.innerWeighting must be `path`, `centroid` or `factorial`.")
    mi = pls.get("maxIterations")
    if not _is_whole(mi) or mi < 1 or mi > 100000:
        raise invalid("analysis.pls.maxIterations must be an integer in [1, 100000].")
    tol = pls.get("tolerance")
    if not (_is_number(tol) and math.isfinite(tol)) or tol <= 0 or tol > 0.01:
        raise invalid("analysis.pls.tolerance must be a number in (0, 0.01].")

    bootstrap = None
    b = _get(an, "bootstrap")
    if b is not None and b.get("enabled") is True:
        if b.get("ci") == "bca":
            raise EngineError(
                "CAPABILITY_UNSUPPORTED",
                "Bootstrap `bca` intervals are not available in the browser engine (only percentile intervals are verified).",
                {"capabilityId": "resampling.bootstrap-bca", "engine": ENGINE_ID},
            )
        if b.get("ci") != "percentile":
            raise invalid("analysis.bootstrap.ci must be `percentile` or `bca`.")
        reps = b.get("repetitions")
        if not _is_whole(reps) or reps < 1 or reps > 100000:
            raise invalid("analysis.bootstrap.repetitions must be an integer in [1, 100000].")
        lvl = b.get("confidenceLevel")
        if not (_is_number(lvl) and lvl > 0.5 and lvl < 1):
            raise invalid("analysis.bootstrap.confidenceLevel must lie strictly between 0.5 and 1.")
        if b.get("sides") not in ("two-sided", "one-sided"):
            raise invalid("analysis.bootstrap.sides must be `two-sided` or `one-sided`.")
        seed = an.get("seed")
        if not _is_whole(seed) or seed < 0 or seed > 2147483647:
            raise invalid(
                "analysis.seed (integer in [0, 2147483647]) is required when bootstrapping."
            )
        bootstrap = BootstrapSpec(int(round(reps)), float(lvl), b["sides"], int(round(seed)))

    # ---- model ------------------------------------------------------------------------
    cons = model.get("constructs")
    if not isinstance(cons, list) or len(cons) == 0:
        raise invalid("model.constructs must be a non-empty array.")
    ind_by_id = {}
    for i in model.get("indicators") or []:
        iid = _check_id(i.get("id"), "indicator id")
        if iid in ind_by_id:
            raise invalid(f"Duplicate indicator id `{iid}`.")
        sc = _get(i, "scale")
        if sc is not None and not (
            _is_number(sc.get("min"))
            and _is_number(sc.get("max"))
            and math.isfinite(sc["min"])
            and math.isfinite(sc["max"])
            and sc["min"] < sc["max"]
        ):
            raise invalid(f"Indicator `{iid}`: scale needs numeric min < max.")
        ind_by_id[iid] = i
    constructs = []
    indicators = []
    used = set()
    cons_index = {}
    for cc in cons:
        cid = _check_id(cc.get("id"), "construct id")
        if cid in cons_index or cid in ind_by_id:
            raise invalid(f"Duplicate id `{cid}`.")
        meas = cc.get("measurement")
        if meas not in ("reflective", "composite-formative"):
            raise invalid(
                f"Construct `{cid}`: measurement must be `reflective` or `composite-formative`."
            )
        if _get(cc, "higherOrder") is not None:
            raise EngineError(
                "CAPABILITY_UNSUPPORTED",
                "Higher-order constructs are not available in the browser PLS engine.",
                {"capabilityId": "pls.higher-order", "engine": ENGINE_ID, "constructId": cid},
            )
        ids = list(cc.get("indicatorIds") or [])
        if len(ids) == 0:
            raise invalid(f"Construct `{cid}` has no indicators.", {"constructId": cid})
        for ii in ids:
            if ii not in ind_by_id:
                raise invalid(f"Construct `{cid}` lists unknown indicator `{ii}`.")
        if len(set(ids)) != len(ids):
            raise invalid(f"Construct `{cid}` lists an indicator twice.")
        dup = [ii for ii in ids if ii in used]
        if dup:
            raise invalid(
                f"Indicator `{dup[0]}` is assigned to more than one construct.",
                {"indicatorIds": dup},
            )
        default_mode = "mode-a" if meas == "reflective" else "mode-b"
        mode = _get(cc, "plsWeightingMode") or default_mode
        if mode != default_mode:
            raise EngineError(
                "CAPABILITY_UNSUPPORTED",
                f"Construct `{cid}`: overriding the weighting mode of a {meas} block ({mode}) is not offered; the mode follows the measurement kind.",
                {"constructId": cid},
            )
        positions = []
        for ii in ids:
            used.add(ii)
            ind = ind_by_id[ii]
            positions.append(len(indicators))
            sc = _get(ind, "scale")
            col = ind.get("column")
            indicators.append(
                SpecIndicator(
                    ii,
                    col if col is not None else "",
                    ind.get("reverseScored") is True,
                    (sc["min"], sc["max"]) if sc is not None else None,
                )
            )
            if col is None:
                raise invalid(
                    f"Indicator `{ii}` is not mapped to a data column.", {"indicatorIds": [ii]}
                )
            if ind.get("reverseScored") is True and sc is None:
                raise invalid(
                    f"Indicator `{ii}` is reverse scored but has no scale endpoints.",
                    {"indicatorIds": [ii]},
                )
        cons_index[cid] = len(constructs)
        constructs.append(
            SpecConstruct(cid, meas == "reflective", mode == "mode-b", list(ids), positions)
        )
    cols = [i.column for i in indicators]
    dup_cols = []
    for idx, c in enumerate(cols):
        if cols.index(c) != idx and c not in dup_cols:
            dup_cols.append(c)
    if dup_cols:
        raise invalid("A data column is bound to more than one indicator.", {"columns": dup_cols})

    # ---- relationships --------------------------------------------------------------------
    rel_ids = set()
    paths = []
    rels = model.get("relationships") or []
    for r in rels:
        rid = _check_id(r.get("id"), "relationship id")
        if rid in rel_ids:
            raise invalid(f"Duplicate relationship id `{rid}`.")
        rel_ids.add(rid)
        if r.get("kind") != "structural":
            continue
        s = cons_index.get(r.get("sourceId"))
        t = cons_index.get(r.get("targetId"))
        if s is None or t is None:
            raise invalid(f"Structural relationship `{rid}` must connect two constructs.")
        if s == t:
            raise invalid(f"Structural relationship `{rid}` is a self-loop.")
        if any(p.source == s and p.target == t for p in paths):
            raise invalid(
                f"Structural path {r.get('sourceId')} -> {r.get('targetId')} is listed twice."
            )
        if _get(r, "fixedValue") is not None:
            raise EngineError(
                "CAPABILITY_UNSUPPORTED",
                f"Relationship `{rid}`: fixed path values are not supported in PLS-SEM.",
                {"relationshipId": rid},
            )
        paths.append(SpecPath(rid, s, t))
    targets = {p.target for p in paths}
    warnings = []
    for r in rels:
        if r.get("kind") != "covariance":
            continue
        s = cons_index.get(r.get("sourceId"))
        t = cons_index.get(r.get("targetId"))
        if s is not None and t is not None and s != t and s not in targets and t not in targets:
            ids = []
            for x in (r["id"], r["sourceId"], r["targetId"]):
                if x not in ids:
                    ids.append(x)
            warnings.append(
                {
                    "code": "EXOGENOUS_CORRELATION_IMPLIED",
                    "severity": "info",
                    "message": "PLS-SEM leaves the correlation between exogenous constructs free; this covariance relationship is not a separate parameter. The estimate is in the construct-correlations matrix.",
                    "elementIds": ids,
                }
            )
        else:
            raise EngineError(
                "CAPABILITY_UNSUPPORTED",
                f"Covariance relationship `{r.get('id')}`: PLS-SEM does not estimate indicator residual covariances or covariances involving endogenous constructs.",
                {"relationshipId": r.get("id")},
            )
    if not paths:
        raise EngineError(
            "MODEL_NOT_IDENTIFIED", "A PLS path model needs at least one structural path.", {}
        )
    isolated = [
        c.id
        for i, c in enumerate(constructs)
        if not any(p.source == i or p.target == i for p in paths)
    ]
    if isolated:
        raise EngineError(
            "MODEL_NOT_IDENTIFIED",
            "Every construct must take part in at least one structural path (the PLS inner weighting has no neighbour for an isolated construct).",
            {"constructIds": isolated},
        )
    k = len(constructs)
    cyc = find_cycle(k, paths)
    if cyc:
        ids = [constructs[i].id for i in cyc]
        uniq = []
        for x in ids:
            if x not in uniq:
                uniq.append(x)
        raise invalid(
            f"The structural model contains a cycle ({' -> '.join(ids)}). The PLS engine estimates recursive models only.",
            {"constructIds": uniq},
        )

    ipma = None
    if ipma_req is not None:
        target = cons_index.get(ipma_req.get("targetConstructId"))
        if target is None:
            raise invalid(
                "analysis.ipma.targetConstructId is not a construct of the model.",
                {"reason": "ipma-target-unknown"},
            )
        preds = [c for c in range(k) if c != target and len(all_paths(k, paths, c, target, 1)) > 0]
        if not preds:
            raise invalid(
                f"The IPMA target `{constructs[target].id}` has no predecessor construct (no structural path leads to it).",
                {"reason": "ipma-no-predecessors", "constructIds": [constructs[target].id]},
            )
        no_scale = [i.id for i in indicators if i.scale is None]
        if no_scale:
            raise invalid(
                f"IPMA needs the original scale endpoints (scale.min, scale.max) of every model indicator; {len(no_scale)} indicator(s) have none.",
                {"reason": "ipma-scale-missing", "indicatorIds": no_scale},
            )
        ipma = IpmaSpec(
            target,
            preds,
            [any(p.source == c and p.target == target for p in paths) for c in preds],
        )

    return PlsSpec(
        method=method,
        scheme=scheme,
        max_iter=int(round(mi)),
        tol=tol,
        constructs=constructs,
        indicators=indicators,
        paths=paths,
        bootstrap=bootstrap,
        ipma=ipma,
        warnings=warnings,
    )


class PreparedData:
    __slots__ = ("X", "n", "n_rows", "n_dropped", "reversed")

    def __init__(self, X, n, n_rows, n_dropped, reversed):
        #: analysis matrix (n x p) in model indicator order
        self.X = X
        self.n = n
        self.n_rows = n_rows
        self.n_dropped = n_dropped
        self.reversed = reversed


def _is_int(x):
    return isinstance(x, (int, np.integer)) and not isinstance(x, bool) or (
        isinstance(x, float) and math.isfinite(x) and x == math.floor(x)
    )


def pls_prepare_data(spec, data):
    """load_job_data(): the numeric matrix in model indicator order, with the R engine's checks."""
    rows = data.get("rows") if isinstance(data, dict) else None
    hdr = data.get("columns") if isinstance(data, dict) else None
    if not isinstance(rows, (list, tuple)) or not isinstance(hdr, (list, tuple)):
        raise invalid("data needs `columns` and `rows`.")
    n_rows = data.get("nRows")
    n_dropped = data.get("nDropped")
    if (
        not (_is_int(n_rows) and _is_int(n_dropped))
        or n_rows < 0
        or n_dropped < 0
        or n_rows - n_dropped != len(rows)
    ):
        raise invalid(
            "The data preparation is inconsistent (nRows - nDropped must equal the number of rows passed).",
            {"reason": "data-preparation-mismatch", "uploadedRows": len(rows)},
        )
    hdr = list(hdr)
    missing_cols = [i for i in spec.indicators if i.column not in hdr]
    if missing_cols:
        raise EngineError(
            "DATA_COLUMN_MISSING",
            f"{len(missing_cols)} mapped column(s) are not in the data: {', '.join('`' + i.column + '`' for i in missing_cols)}.",
            {
                "columns": [i.column for i in missing_cols],
                "indicatorIds": [i.id for i in missing_cols],
            },
        )
    ambiguous = [i for i in spec.indicators if hdr.count(i.column) > 1]
    if ambiguous:
        raise invalid(
            "A mapped column name occurs more than once in the data header.",
            {"columns": [i.column for i in ambiguous]},
        )
    n = len(rows)
    p = len(spec.indicators)
    col_idx = [hdr.index(i.column) for i in spec.indicators]
    X = np.zeros((n, p))
    non_numeric = {}
    missing = {}
    for r in range(n):
        row = rows[r]
        for j in range(p):
            ci = col_idx[j]
            v = row[ci] if row is not None and ci < len(row) else None
            col = spec.indicators[j].column
            if v is None or (isinstance(v, float) and v != v):
                missing[col] = missing.get(col, 0) + 1
            elif not _is_number(v) or not math.isfinite(v):
                non_numeric[col] = non_numeric.get(col, 0) + 1
            else:
                X[r, j] = v
    if non_numeric:
        raise EngineError(
            "DATA_NOT_NUMERIC",
            f"{len(non_numeric)} mapped column(s) contain non-numeric cells: {', '.join(f'`{c}` ({k} cells)' for c, k in non_numeric.items())}.",
            {"columns": list(non_numeric), "nonNumericCells": list(non_numeric.values())},
        )
    if missing:
        raise invalid(
            f"{len(missing)} mapped column(s) contain missing cells: {', '.join(f'`{c}` ({k} cells)' for c, k in missing.items())}. The engine expects the listwise-complete data set.",
            {
                "reason": "missing-values",
                "columns": list(missing),
                "missingCells": list(missing.values()),
            },
        )
    if n < 3:
        raise EngineError(
            "DATA_INSUFFICIENT_ROWS",
            f"The data have {n} row(s); at least 3 are required.",
            {"nRows": n, "minimum": 3},
        )
    constant = [ind for j, ind in enumerate(spec.indicators) if np.all(X[:, j] == X[0, j])]
    if constant:
        raise EngineError(
            "DATA_ZERO_VARIANCE",
            f"{len(constant)} mapped column(s) have zero variance: {', '.join('`' + i.column + '`' for i in constant)}.",
            {"columns": [i.column for i in constant], "indicatorIds": [i.id for i in constant]},
        )
    if spec.ipma is not None:
        bad = []
        counts = []
        for j, ind in enumerate(spec.indicators):
            lo, hi = ind.scale
            c = int(np.sum((X[:, j] < lo) | (X[:, j] > hi)))
            if c > 0:
                bad.append(ind.id)
                counts.append(c)
        if bad:
            raise invalid(
                f"IPMA: {len(bad)} indicator(s) have observed values outside their declared scale endpoints.",
                {"reason": "ipma-out-of-scale", "indicatorIds": bad, "cellsOutOfScale": counts},
            )
    for c in spec.constructs:
        if not c.mode_b or len(c.indicators) < 2:
            continue
        R, _ = correlation(X[:, c.indicators])
        if qr_rank(R, 1e-10) < R.shape[0]:
            raise EngineError(
                "MODEL_NOT_IDENTIFIED",
                f"The indicators of Mode B construct `{c.id}` are perfectly collinear; its weights are not identified.",
                {"constructId": c.id},
            )
    return PreparedData(
        X, n, int(n_rows), int(n_dropped), [i.id for i in spec.indicators if i.reverse]
    )


# ---------------------------------------------------------------------------------------------
# core


class CoreModel:
    """The model as the estimation core sees it (indices, no ids)."""

    __slots__ = (
        "k",
        "p",
        "blocks",
        "mode_b",
        "common_factor",
        "structural",
        "scheme",
        "max_iter",
        "tol",
        "preds",
        "endo_mask",
    )

    def __init__(self, k, p, blocks, mode_b, common_factor, structural, scheme, max_iter, tol):
        self.k = k
        self.p = p
        self.blocks = blocks
        self.mode_b = mode_b
        self.common_factor = common_factor
        #: structural[t, s] = 1 when there is a path s -> t
        self.structural = structural
        self.scheme = scheme
        self.max_iter = max_iter
        self.tol = tol
        self.preds = [[s for s in range(k) if structural[t, s] == 1] for t in range(k)]
        self.endo_mask = [bool(np.any(structural[t] == 1)) for t in range(k)]


def core_model(spec):
    k = len(spec.constructs)
    structural = np.zeros((k, k), dtype=np.int8)
    for p in spec.paths:
        structural[p.target, p.source] = 1
    return CoreModel(
        k,
        len(spec.indicators),
        [list(c.indicators) for c in spec.constructs],
        [c.mode_b for c in spec.constructs],
        [c.reflective for c in spec.constructs],
        structural,
        spec.scheme,
        spec.max_iter,
        spec.tol,
    )


class CoreFit:
    __slots__ = (
        "converged",
        "iterations",
        "W",
        "Lambda",
        "Q",
        "C",
        "P",
        "B",
        "R2",
        "R2adj",
        "vif",
        "disattenuated",
    )

    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)


def _composite_vcv(WS, W):
    """C = W S W' with the upper triangle copied to the lower (cSEM calculateCompositeVCV)."""
    C = WS @ W.T
    return symmetrize_from_upper(C)


def scale_weights(S, W, blocks):
    """cSEM:::scaleWeights: each row divided by the composite's sd sqrt(w'Sw)."""
    out = np.empty_like(W)
    with np.errstate(all="ignore"):
        for j, b in enumerate(blocks):
            w = W[j, b]
            v = float(w @ S[np.ix_(b, b)] @ w)
            f = 1.0 / np.sqrt(v) if v >= 0 else math.nan
            if not math.isfinite(f):
                raise NumericalError("composite variance is not positive")
            out[j] = W[j] * f
    return out


def _inner_weights(C, model):
    """cSEM:::calculateInnerWeightsPLS (with .PLS_ignore_structural_model = FALSE)."""
    k = model.k
    st = model.structural
    if model.scheme == "path":
        E = np.where(st.T == 1, C, 0.0)
        for j in range(k):
            f = model.preds[j]
            if not f:
                continue
            coef = solve_r(C[np.ix_(f, f)], C[j, f])
            E[j, f] = coef
        return E
    adj = (st == 1) | (st.T == 1)
    if model.scheme == "centroid":
        return np.where(adj, np.sign(C), 0.0)
    return np.where(adj, C, 0.0)


def _outer_weights(S, W, E, model):
    """cSEM:::calculateOuterWeightsPLS for Mode A / Mode B blocks."""
    WS = W @ S
    PIC = E @ WS
    out = np.zeros((model.k, model.p))
    for j, b in enumerate(model.blocks):
        if len(b) == 1:
            out[j, b[0]] = 1.0
        elif model.mode_b[j]:
            out[j, b] = solve_r(S[np.ix_(b, b)], PIC[j, b])
        else:
            out[j, b] = PIC[j, b]
    return out


class WeightResult:
    __slots__ = ("W", "converged", "iterations")

    def __init__(self, W, converged, iterations):
        self.W = W
        self.converged = converged
        self.iterations = iterations


def fit_weights(S, model):
    """cSEM:::calculateWeightsPLS with .conv_criterion = "diff_absolute"."""
    W = np.zeros((model.k, model.p))
    for j, b in enumerate(model.blocks):
        W[j, b] = 1.0
    W = scale_weights(S, W, model.blocks)
    W_iter = W
    it = 0
    converged = False
    while True:
        it += 1
        C = _composite_vcv(W_iter @ S, W_iter)
        E = _inner_weights(C, model)
        W = scale_weights(S, _outer_weights(S, W_iter, E, model), model.blocks)
        max_diff = float(np.max(np.abs(W_iter - W)))
        if not math.isfinite(max_diff):
            raise NumericalError("non-finite weights")
        if max_diff < model.tol:
            converged = True
            break
        # iter_max == 1 gives cSEM status NULL, which the engine treats as not converged
        if it >= model.max_iter:
            break
        W_iter = W
    return WeightResult(W, converged, it)


def correction_factor(S, w, block):
    """cSEM:::calculateCorrectionFactors, .PLS_approach_cf = "dist_squared_euclid"."""
    w = np.asarray(w, dtype=float)
    WW = np.outer(w, w)
    np.fill_diagonal(WW, 0.0)
    num = float(np.sum(WW * S[np.ix_(block, block)]))
    den = float(np.sum(WW * WW))
    with np.errstate(all="ignore"):
        return float(np.sqrt(np.abs(np.divide(num, den))))


def loadings_and_q(S, W, model, disattenuate):
    """Loadings and Q (square roots of reliabilities) as cSEM:::calculateReliabilities."""
    WS = W @ S
    Lambda = np.zeros((model.k, model.p))
    Q = np.ones(model.k)
    for j, b in enumerate(model.blocks):
        Lambda[j, b] = WS[j, b]
    if disattenuate:
        for j, b in enumerate(model.blocks):
            if not model.common_factor[j] or model.mode_b[j]:
                continue
            w = W[j, b]
            c = 1.0 if len(b) == 1 else correction_factor(S, w, b)
            lam = c * w
            Lambda[j, b] = lam
            Q[j] = float(np.sum(w * lam))
    return Lambda, Q, WS


def construct_vcv(C, Q):
    """cSEM:::calculateConstructVCV."""
    with np.errstate(all="ignore"):
        P = C / np.outer(Q, Q)
    np.fill_diagonal(P, 1.0)
    return P


class PathResult:
    __slots__ = ("B", "R2", "R2adj", "vif")

    def __init__(self, B, R2, R2adj, vif):
        self.B = B
        self.R2 = R2
        self.R2adj = R2adj
        self.vif = vif


def estimate_paths(P, structural, n):
    """cSEM:::estimatePath (linear, OLS): coef = P_XX^-1 P_Xy per endogenous construct."""
    k = P.shape[0]
    B = np.zeros((k, k))
    R2 = np.full(k, math.nan)
    R2adj = np.full(k, math.nan)
    vif = np.full((k, k), math.nan)
    for y in range(k):
        X = [s for s in range(k) if structural[y][s] == 1]
        if not X:
            continue
        Pxx = P[np.ix_(X, X)]
        inv = inverse_r(Pxx)
        pxy = P[X, y]
        coef = inv @ pxy
        r2 = float(coef @ Pxx @ coef)
        B[y, X] = coef
        R2[y] = r2
        with np.errstate(all="ignore"):
            R2adj[y] = 1 - np.divide((1 - r2) * (n - 1), float(n - len(X) - 1))
        if len(X) > 1:
            vif[y, X] = np.diag(inv)
    return PathResult(B, R2, R2adj, vif)


def fit_pls(S, model, disattenuate, n):
    """The complete csem() fit on a correlation matrix. Raises NumericalError where cSEM errors."""
    return complete_fit(S, model, fit_weights(S, model), disattenuate, n)


def complete_fit(S, model, wr, disattenuate, n):
    W = wr.W
    Lambda, Q, WS = loadings_and_q(S, W, model, disattenuate)
    C = _composite_vcv(WS, W)
    P = construct_vcv(C, Q)
    pr = estimate_paths(P, model.structural, n)
    return CoreFit(
        converged=wr.converged,
        iterations=wr.iterations,
        W=W,
        Lambda=Lambda,
        Q=Q,
        C=C,
        P=P,
        B=pr.B,
        R2=pr.R2,
        R2adj=pr.R2adj,
        vif=pr.vif,
        disattenuated=disattenuate,
    )


def model_implied_construct_cor(fit, model):
    """Model-implied construct correlations of cSEM:::fit(.saturated = FALSE)."""
    k = model.k
    endo = [c for c in range(k) if model.endo_mask[c]]
    exo = [c for c in range(k) if not model.endo_mask[c]]
    Path = fit.B
    P = fit.P
    ne = len(endo)
    IB = inverse_r(np.eye(ne) - Path[np.ix_(endo, endo)])
    Gamma = Path[np.ix_(endo, exo)]
    Phi = P[np.ix_(exo, exo)]
    zeta = np.array([1 - float(np.sum(Path[t] * P[t])) for t in endo])
    PhiGt = Phi @ Gamma.T
    corr_exo_endo = PhiGt @ IB.T
    inner = Gamma @ PhiGt + np.diag(zeta)
    cor_endo = IB @ inner @ IB.T
    np.fill_diagonal(cor_endo, 1.0)
    nx = len(exo)
    V0 = np.zeros((k, k))
    V0[:nx, :nx] = Phi
    V0[:nx, nx:] = corr_exo_endo
    V0[nx:, :nx] = corr_exo_endo.T
    V0[nx:, nx:] = cor_endo
    symmetrize_from_upper(V0)
    order = exo + endo
    V = np.zeros((k, k))
    V[np.ix_(order, order)] = V0
    return V


def model_implied_indicator_cor(S, fit, model, treat_as_common_factor=False):
    """Model-implied indicator correlations of cSEM:::fit(.type_vcv = "indicator")."""
    V = model_implied_construct_cor(fit, model)
    p = model.p
    owner = np.full(p, -1)
    for j, b in enumerate(model.blocks):
        owner[b] = j
    lam = np.sum(fit.Lambda**2, axis=0)  # diag(Lambda' Lambda)
    L = fit.Lambda[owner, np.arange(p)]
    Sigma = (L[:, None] * V[np.ix_(owner, owner)]) * L[None, :]
    Sigma[np.diag_indices(p)] += np.diag(S) - lam
    Sigma = np.triu(Sigma)
    symmetrize_from_upper(Sigma)
    if not treat_as_common_factor:
        for j, b in enumerate(model.blocks):
            if model.common_factor[j]:
                continue
            Sigma[np.ix_(b, b)] = S[np.ix_(b, b)]
    return Sigma


VERIFY_CHECKS = {
    "1": "Convergence achieved",
    "2": "All absolute standardized loading estimates <= 1",
    "3": "Construct VCV is positive semi-definite",
    "4": "All reliability estimates <= 1",
    "5": "Model-implied indicator VCV is positive semi-definite",
}


def verify(S, fit, model):
    """cSEM::verify(): the checks that FAILED."""
    failed = []
    if not fit.converged:
        failed.append("1")
    absL = np.abs(fit.Lambda)
    if np.any(np.isnan(absL)) or float(np.max(absL)) > 1:
        failed.append("2")
    if not is_positive_semi_definite(fit.P):
        failed.append("3")
    if any(not (q * q <= 1) for q in fit.Q):
        failed.append("4")
    try:
        sigma_psd = is_positive_semi_definite(model_implied_indicator_cor(S, fit, model))
    except NumericalError:
        sigma_psd = False
    if not sigma_psd:
        failed.append("5")
    return failed


# =============================================================================================
# semlab/assess.py
# =============================================================================================
# Quality criteria with the definitions of cSEM 0.7.1 ``assess()`` (port of
# packages/engine/src/pls/assess.ts):
#
#     cronbach_alpha   cSEM:::calculateRhoT (unweighted, correlation-based)
#     rho_c            cSEM:::calculateRhoC (.model_implied = TRUE, .weighted = FALSE)
#     ave              cSEM:::calculateAVE
#     rho_a            Estimates$Reliabilities of a PLSc pass (cSEM:::calculateReliabilities)
#     htmt_pair        cSEM:::calculateHTMTcore (.absolute = TRUE, htmt / htmt2)
#     cross_loadings   stats::cor(data, Estimates$Construct_scores)
#     f2               cSEM:::calculatef2 (drop the predictor, same construct correlations)
#     vif_mode_b       cSEM:::calculateVIFModeB
#     fit_indices      cSEM:::calculateSRMR / calculateDL / calculateDG (log base 10)

def cronbach_alpha(S, block):
    """Standardised Cronbach's alpha: rbar * (sum w)^2, w = unit weights scaled to w'Sw = 1."""
    Sb = S[np.ix_(block, block)]
    total = float(np.sum(Sb))
    iu = np.triu_indices(len(block), 1)
    rbar = float(np.sum(Sb[iu])) / len(iu[0])
    with np.errstate(all="ignore"):
        w = 1 / np.sqrt(total)
    return float(rbar * (len(block) * w) ** 2)


def rho_c(sigma_cf, fit, j, block):
    """rho_C on the reported loadings with the model-implied block correlations."""
    total = float(np.sum(sigma_cf[np.ix_(block, block)]))
    with np.errstate(all="ignore"):
        w = 1 / np.sqrt(total)
    s = float(np.sum(w * fit.Lambda[j, block]))
    return s * s


def ave(fit, j, block):
    """AVE = sum l^2 / (sum l^2 + sum (1 - l^2)) over the block's non-zero loadings."""
    s2 = 0.0
    s1 = 0.0
    for a in block:
        lam = float(fit.Lambda[j, a])
        if lam == 0:
            continue
        s2 += lam * lam
        s1 += 1 - lam * lam
    with np.errstate(all="ignore"):
        return float(np.divide(s2, s2 + s1))


def rho_a(S, W, block, j):
    """rho_A = Q^2 of the PLSc correction applied to these (identical) weights."""
    w = W[j, block]
    c = 1.0 if len(block) == 1 else correction_factor(S, w, block)
    q = float(np.sum(w * c * w))
    return q * q


def _nth_root(x, n):
    if x < 0 and n % 2 == 1:
        return -((-x) ** (1 / n))
    if x < 0:
        return math.nan
    return x ** (1 / n)


def _lower_tri(S, block):
    # column-major lower triangle, as S[lower.tri(S)]
    out = []
    for c in range(len(block)):
        for r in range(c + 1, len(block)):
            out.append(float(S[block[r], block[c]]))
    return out


def htmt_pair(S, b1, b2, kind):
    """HTMT / HTMT2 of two blocks (absolute value), as cSEM:::calculateHTMTcore."""
    mono1 = [1.0] if len(b1) == 1 else _lower_tri(S, b1)
    mono2 = [1.0] if len(b2) == 1 else _lower_tri(S, b2)
    het = [float(S[r, c]) for c in b2 for r in b1]
    sign_id = 1
    if all(v < 0 for v in het):
        het = [abs(v) for v in het]
        sign_id = -1
    if kind == "htmt":
        a1 = sum(mono1) / len(mono1)
        a2 = sum(mono2) / len(mono2)
        a3 = sum(het) / len(het)
    else:

        def prod(x):
            p = 1.0
            for v in x:
                p *= v
            return p

        a1 = _nth_root(prod(mono1), len(mono1))
        a2 = _nth_root(prod(mono2), len(mono2))
        a3 = _nth_root(prod(het), len(het))
    with np.errstate(all="ignore"):
        return float(abs((a3 / np.sqrt(a1 * a2)) * sign_id))


def cross_loadings(fit, S, model):
    """Pearson correlation of every indicator (rows) with every construct score (columns)."""
    with np.errstate(all="ignore"):
        return (fit.W @ S).T / np.sqrt(np.diag(fit.C))[None, :]


def f2(fit, model, n):
    """f2[t, s] for every path s -> t (NaN elsewhere)."""
    k = model.k
    out = np.full((k, k), math.nan)
    for t in range(k):
        for s in range(k):
            if model.structural[t, s] != 1:
                continue
            reduced = model.structural.copy()
            reduced[t, s] = 0
            r2excl = (
                float(estimate_paths(fit.P, reduced, n).R2[t]) if np.any(reduced[t] == 1) else 0.0
            )
            r2 = float(fit.R2[t])
            with np.errstate(all="ignore"):
                out[t, s] = np.divide(r2 - r2excl, 1 - r2)
    return out


def vif_mode_b(S, block):
    """VIF of each indicator of a Mode B block: diagonal of the inverse block correlation matrix."""
    inv = inverse_r(S[np.ix_(block, block)])
    return np.diag(inv).copy()


def _geodesic(S, Sigma):
    """0.5 sum (log10 l_i)^2, l eigenvalues of S^-1 Sigma (symmetric form L^-1 Sigma L^-T)."""
    try:
        L = np.linalg.cholesky(S)
    except np.linalg.LinAlgError as e:
        raise NumericalError("S is not positive definite") from e
    X = solve_triangular(L, Sigma, lower=True)
    M = solve_triangular(L, X.T, lower=True)
    M = 0.5 * (M + M.T)
    ev = symmetric_eigenvalues(M)
    with np.errstate(all="ignore"):
        return float(0.5 * np.sum(np.log10(ev) ** 2))


class FitIndices:
    __slots__ = ("srmr", "d_uls", "d_g")

    def __init__(self, srmr, d_uls, d_g):
        self.srmr = srmr
        self.d_uls = d_uls
        self.d_g = d_g


def fit_indices(S, fit, model):
    """SRMR, d_ULS and d_G (base-10 logarithm) against the model-implied indicator correlations."""
    Sigma = model_implied_indicator_cor(S, fit, model)
    p = S.shape[0]
    with np.errstate(all="ignore"):
        ds = np.sqrt(np.diag(S))
        dh = np.sqrt(np.diag(Sigma))
        D = S - Sigma
        dl = float(np.sum(D * D))
        cs = S / np.outer(ds, ds)
        ch = Sigma / np.outer(dh, dh)
        np.fill_diagonal(cs, 1.0)
        np.fill_diagonal(ch, 1.0)
        il = np.tril_indices(p)
        c = cs[il] - ch[il]
        srmr = math.sqrt(float(np.sum(c * c)) / len(il[0]))
    try:
        dg = _geodesic(S, Sigma)
    except NumericalError:
        dg = math.nan
    return FitIndices(srmr, 0.5 * dl, dg)


def reliability_eligible(model):
    """Common-factor blocks with >= 2 indicators (eligibility for alpha, rho, AVE, HTMT, FL)."""
    return [model.common_factor[j] and len(b) >= 2 for j, b in enumerate(model.blocks)]


def rho_c_sigma(S, fit, model):
    """Model-implied correlations with composites converted to common factors (calculateRhoC)."""
    return model_implied_indicator_cor(S, fit, model, True)


# =============================================================================================
# semlab/bootstrap.py
# =============================================================================================
# SEM Lab's bootstrap definition (port of packages/engine/src/pls/bootstrap.ts):
#
#     set.seed(seed, kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
#     for (b in 1:B) {
#       idx_b <- sample.int(n, n, replace = TRUE)   # one RNG stream, drawn sequentially
#       refit the full model on data[idx_b, ]         # same csem() options as the main fit
#       failed       if the fit errors, does not converge or gives a non-finite estimate
#       inadmissible if cSEM::verify() flags it
#     }
#     valid draws only: se = sd (divisor B_valid - 1); t = est / se;
#     p = 2 P(T <= -|t|), T ~ t(B_valid - 1) (one-sided: P(T <= -|t|), direction of the estimate);
#     CI = quantile(type = 7) percentile; one-sided: est >= 0 -> lower = q(1 - L), upper null;
#     est < 0 -> upper = q(L), lower null. Total / indirect / specific effects per draw.

class ThetaSlot:
    """Where an estimated parameter lives in a CoreFit: B / Lambda / W [row, col]."""

    __slots__ = ("kind", "row", "col")

    def __init__(self, kind, row, col):
        self.kind = kind
        self.row = row
        self.col = col


class BootstrapProblem:
    """Everything needed to fit one draw."""

    __slots__ = ("core", "disattenuate", "X", "theta")

    def __init__(self, core, disattenuate, X, theta):
        self.core = core
        self.disattenuate = disattenuate
        #: analysis matrix (n x p), model indicator order
        self.X = X
        self.theta = theta


def extract_theta(fit, slots):
    out = []
    for s in slots:
        if s.kind == "path":
            out.append(float(fit.B[s.row, s.col]))
        elif s.kind == "loading":
            out.append(float(fit.Lambda[s.row, s.col]))
        else:
            out.append(float(fit.W[s.row, s.col]))
    return out


class DrawResult:
    __slots__ = ("status", "reason", "checks", "theta")

    def __init__(self, status, reason=None, checks=None, theta=None):
        self.status = status  # 'ok' | 'failed' | 'inadmissible'
        self.reason = reason  # failed: 'error' | 'nonconvergence' | 'non-finite'
        self.checks = checks  # inadmissible: failed verify() checks
        self.theta = theta


def fit_draw(problem, idx):
    """Refit the model on the rows `idx` (0-based, with repetitions) and classify the draw."""
    S, zero = correlation(problem.X, idx)
    if zero:
        return DrawResult("failed", reason="error")
    try:
        with np.errstate(all="ignore"):
            fit = fit_pls(S, problem.core, problem.disattenuate, len(idx))
    except NumericalError:
        return DrawResult("failed", reason="error")
    if not fit.converged:
        return DrawResult("failed", reason="nonconvergence")
    theta = extract_theta(fit, problem.theta)
    if any(not math.isfinite(v) for v in theta):
        return DrawResult("failed", reason="non-finite")
    with np.errstate(all="ignore"):
        checks = verify(S, fit, problem.core)
    if checks:
        return DrawResult("inadmissible", checks=checks, theta=theta)
    return DrawResult("ok", theta=theta)


class BootstrapCounts:
    __slots__ = ("requested", "completed", "failed", "inadmissible", "valid")

    def __init__(self, requested, completed, failed, inadmissible, valid):
        self.requested = requested
        self.completed = completed
        self.failed = failed
        self.inadmissible = inadmissible
        #: valid draws in draw order: valid[d][i] belongs to theta slot i
        self.valid = valid


def summarise_bootstrap(draws_in_order):
    valid = []
    failed = 0
    inadmissible = 0
    for d in draws_in_order:
        if d.status == "ok":
            valid.append(d.theta)
        elif d.status == "failed":
            failed += 1
        else:
            inadmissible += 1
    return BootstrapCounts(len(draws_in_order), len(valid), failed, inadmissible, valid)


class Inference:
    __slots__ = ("n_valid", "se", "statistic", "p_value", "ci_lower", "ci_upper", "reason")

    def __init__(self, n_valid, se, statistic, p_value, ci_lower, ci_upper, reason=None):
        self.n_valid = n_valid
        self.se = se
        self.statistic = statistic
        self.p_value = p_value
        self.ci_lower = ci_lower
        self.ci_upper = ci_upper
        self.reason = reason

    def get(self, field):
        return {
            "se": self.se,
            "statistic": self.statistic,
            "pValue": self.p_value,
            "ciLower": self.ci_lower,
            "ciUpper": self.ci_upper,
        }[field]


def infer_from_draws(est, draws_in, level, sides):
    """Inference for one parameter from its valid draws (R engine infer_from_draws)."""
    draws = [v for v in draws_in if math.isfinite(v)]
    nv = len(draws)
    if not math.isfinite(est) or nv < 2:
        return Inference(
            nv, None, None, None, None, None, "failed" if math.isfinite(est) else "non-finite"
        )
    se = sample_sd(draws)
    with np.errstate(all="ignore"):
        t = float(np.divide(est, se))
    df = nv - 1
    if sides == "two-sided":
        p = 2 * pt_lower_tail_abs(t, df)
        q = quantile7(draws, [(1 - level) / 2, 1 - (1 - level) / 2])
        lo, hi = q[0], q[1]
    else:
        p = pt_lower_tail_abs(t, df)
        if est >= 0:
            lo, hi = quantile7(draws, [1 - level])[0], None
        else:
            lo, hi = None, quantile7(draws, [level])[0]
    if math.isfinite(p):
        p = min(1.0, max(0.0, p))
    return Inference(nv, se, t, p, lo, hi)


# =============================================================================================
# semlab/ipma.py
# =============================================================================================
# Importance-performance map analysis (port of packages/engine/src/pls/ipma.ts; semantics of
# the R engine's pls_ipma.R, which matched cSEM::doIPMA() on rescaled data):
#
#  1. reverse scoring first (already in the analysis matrix), then x100 = (x - min)/(max - min)*100
#     with the DECLARED endpoints;
#  2. classic PLS (same scheme / iteration cap / tolerance) on the rescaled data;
#  3. unstandardised weight = standardised weight / sd(x100), normalised to sum to 1 per block;
#  4. construct score = weighted sum of rescaled indicators; performance = mean score;
#  5. importance = unstandardised total effect: OLS (with intercept) of each endogenous score on
#     its predictors' scores gives B, total = (I - B)^-1 - I; the sign is kept;
#  6. indicator importance = normalised weight x construct importance; indicator performance =
#     mean rescaled indicator.

IPMA_CONVENTION = " ".join(
    [
        "Each indicator is first reverse scored where flagged ((min + max) - x), then rescaled to 0-100 as (x - min)/(max - min)*100",
        "with its declared original scale endpoints (never the observed minimum/maximum).",
        "Classic PLS-SEM (no consistency correction; same inner weighting scheme, iteration cap and tolerance as the main fit) is re-estimated on the rescaled data.",
        "Unstandardized outer weights = standardized weight / sd(rescaled indicator), normalized to sum to 1 within each block;",
        "construct score = weighted sum of the rescaled indicators (0-100 when no weight is negative).",
        "Performance = mean construct score (constructs) or mean rescaled indicator (indicators), on the 0-100 scale.",
        "Importance = unstandardized total effect of the construct on the target: OLS of each endogenous score on its predictors' scores gives B, total effects = (I - B)^-1 - I.",
        "Indicator importance = normalized weight x the construct's importance. Negative importance keeps its sign.",
        "Constructs listed are all predecessors of the target (any directed path); 'direct' means a direct structural path exists.",
        "Point estimates only (no bootstrap intervals).",
    ]
)


def _fin(x):
    return float(x) if math.isfinite(x) else None


def compute_ipma(spec, data):
    ip = spec.ipma
    model = core_model(spec)
    n = data.n
    lo = np.array([ind.scale[0] for ind in spec.indicators], dtype=float)
    hi = np.array([ind.scale[1] for ind in spec.indicators], dtype=float)
    X100 = ((data.X - lo[None, :]) / (hi - lo)[None, :]) * 100
    S, _ = correlation(X100)
    wr = fit_weights(S, model)
    if not wr.converged:
        raise EngineError(
            "ENGINE_NONCONVERGENCE",
            "The classic PLS fit on the 0-100 rescaled data (IPMA) did not converge.",
            {"stage": "ipma"},
        )
    k = model.k
    wn = []
    scores = []
    with np.errstate(all="ignore"):
        for j in range(k):
            block = model.blocks[j]
            wu = [float(wr.W[j, a]) / r_sd(X100[:, a]) for a in block]
            total = 0.0
            for v in wu:
                total += v
            w = [v / total if total != 0 else float(np.divide(v, total)) for v in wu]
            wn.append(w)
            sc = np.zeros(n)
            for idx, a in enumerate(block):
                sc = sc + X100[:, a] * w[idx]
            scores.append(sc)
        means = [r_mean(s) for s in scores]
        Bu = np.zeros((k, k))
        for t in range(k):
            src = [p.source for p in spec.paths if p.target == t]
            if not src:
                continue

            def cov(a, b):
                return float(np.sum((scores[a] - means[a]) * (scores[b] - means[b])))

            A = np.array([[cov(a, b) for b in src] for a in src])
            coef = solve_r(A, np.array([cov(a, t) for a in src]))
            Bu[t, src] = coef
        eye = np.eye(k)
        inv = inverse_r(eye - Bu)
        Tu = inv - eye
        imp = [float(Tu[ip.target, c]) for c in ip.predecessors]

    warnings = []
    constructs = [
        {
            "constructId": spec.constructs[c].id,
            "importance": _fin(imp[j]),
            "performance": _fin(means[c]),
            "predecessor": "direct" if ip.direct[j] else "indirect",
        }
        for j, c in enumerate(ip.predecessors)
    ]
    indicators = []
    for j, c in enumerate(ip.predecessors):
        for idx, a in enumerate(spec.constructs[c].indicators):
            indicators.append(
                {
                    "indicatorId": spec.indicators[a].id,
                    "constructId": spec.constructs[c].id,
                    "importance": _fin(wn[c][idx] * imp[j]),
                    "performance": _fin(r_mean(X100[:, a])),
                }
            )
    neg = [
        spec.indicators[a].id
        for c in ip.predecessors
        for idx, a in enumerate(spec.constructs[c].indicators)
        if wn[c][idx] < 0
    ]
    if neg:
        warnings.append(
            {
                "code": "IPMA_NEGATIVE_WEIGHT",
                "severity": "warning",
                "message": f"{len(neg)} normalized IPMA weight(s) are negative; the construct score is then not a weighted mean and its performance can fall outside 0-100.",
                "elementIds": neg,
            }
        )
    if any(c["importance"] is None or c["performance"] is None for c in constructs):
        warnings.append(
            {
                "code": "IPMA_NON_FINITE",
                "severity": "error",
                "message": "Some IPMA values could not be computed (non-finite; e.g. weights of a block summing to zero) and are null.",
            }
        )
    if spec.bootstrap is not None:
        warnings.append(
            {
                "code": "IPMA_NO_INTERVALS",
                "severity": "info",
                "message": "IPMA importance and performance are point estimates in this version (no bootstrap intervals).",
            }
        )
    return (
        {
            "targetConstructId": spec.constructs[ip.target].id,
            "constructs": constructs,
            "indicators": indicators,
            "convention": IPMA_CONVENTION,
        },
        warnings,
    )


# =============================================================================================
# semlab/pls.py
# =============================================================================================
# estimate_pls(): the PLS-SEM / PLSc job (port of packages/engine/src/pls/estimate.ts and
# results.ts). Numbers follow cSEM 0.7.1 (pls_core.py), the normalisation follows the R engine's
# pls_results.R (keys, kinds, ordering, definitions, missing reasons), the bootstrap is SEM
# Studio's definition (bootstrap.py).

ENGINE_VERSION = "0.1.0"
#: Draws stored in result.bootstrap.draws are rounded to this many significant digits.
DRAW_SIGNIF = 6

SOURCE_PLS = "@sem-lab/engine pls-pm (cSEM 0.7.1 semantics)"
SOURCE_EFFECTS = "@sem-lab/engine effects from the path estimates"
INFERENCE_NOTE = "se = sd(valid draws, divisor B_valid-1); t = est/se; p from t distribution, df = B_valid-1; CI = percentile, quantile(type 7)"
_INF_FIELDS = ("se", "statistic", "pValue", "ciLower", "ciUpper")


# ---- theta ----------------------------------------------------------------------------------


class ThetaEntry(ThetaSlot):
    __slots__ = ("key", "source_id", "target_id", "relationship_id", "single", "raw_path")

    def __init__(self, kind, row, col, key, source_id, target_id, relationship_id, single, raw):
        super().__init__(kind, row, col)
        self.key = key
        self.source_id = source_id
        self.target_id = target_id
        self.relationship_id = relationship_id
        self.single = single
        self.raw_path = raw


def theta_entries(spec):
    """Paths in relationship order, then every block's (loading, weight) per indicator."""
    out = []
    for p in spec.paths:
        s = spec.constructs[p.source].id
        t = spec.constructs[p.target].id
        out.append(
            ThetaEntry(
                "path",
                p.target,
                p.source,
                f"path:{s}->{t}",
                s,
                t,
                p.id,
                False,
                f"path coefficient B[{t}, {s}] (OLS on the construct correlations)",
            )
        )
    for j, c in enumerate(spec.constructs):
        single = len(c.indicators) == 1
        for a in c.indicators:
            ii = spec.indicators[a].id
            out.append(
                ThetaEntry(
                    "loading",
                    j,
                    a,
                    f"loading:{c.id}:{ii}",
                    c.id,
                    ii,
                    None,
                    single,
                    f"loading Lambda[{c.id}, {ii}]",
                )
            )
            out.append(
                ThetaEntry(
                    "weight",
                    j,
                    a,
                    f"weight:{c.id}:{ii}",
                    c.id,
                    ii,
                    None,
                    single,
                    f"weight W[{c.id}, {ii}] (scaled to unit composite variance)",
                )
            )
    return out


# ---- parameters ------------------------------------------------------------------------------


def _num_or_reason(x, reason):
    if x is None:
        return None, reason
    if not math.isfinite(x):
        return None, "non-finite"
    return float(x), None


def param_entry(
    key,
    kind,
    est,
    source_id=None,
    target_id=None,
    via_ids=None,
    relationship_id=None,
    inf=None,
    boot=None,
    fixed_reason=None,
    source_method="",
    raw_path="",
):
    """R param_entry()."""
    missing = {}
    status = "fixed" if fixed_reason else "estimated"
    e_val, e_reason = _num_or_reason(est, "non-finite")
    if e_reason:
        missing["est"] = e_reason
        status = "failed"
    vals = {f: None for f in _INF_FIELDS}
    use_boot = boot is not None and not fixed_reason
    for f in _INF_FIELDS:
        if fixed_reason:
            missing[f] = fixed_reason
        elif not use_boot:
            missing[f] = "not-requested"
        else:
            v = inf.get(f)
            if v is None:
                missing[f] = inf.reason or (
                    "not-applicable" if f in ("ciLower", "ciUpper") else "failed"
                )
            else:
                nv, nr = _num_or_reason(v, "failed")
                vals[f] = nv
                if nr:
                    missing[f] = nr
    p = {"key": key, "kind": kind}
    if source_id is not None:
        p["sourceId"] = source_id
    if target_id is not None:
        p["targetId"] = target_id
    if via_ids is not None:
        p["viaIds"] = via_ids
    if relationship_id is not None:
        p["relationshipId"] = relationship_id
    p["scale"] = "standardized"
    p["est"] = e_val
    p.update(vals)
    if use_boot:
        p["statisticType"] = "t"
    if missing:
        p["missing"] = missing
    p["status"] = status
    if use_boot:
        p["inference"] = {
            "method": "bootstrap-percentile",
            "level": boot.level,
            "sides": boot.sides,
            "nValid": inf.n_valid,
        }
    p["sourceMethod"] = source_method
    p["rawPath"] = raw_path[:400]
    return p


class BootstrapInference:
    __slots__ = ("boot", "theta", "total", "indirect", "specific")

    def __init__(self, boot, theta, total, indirect, specific):
        self.boot = boot
        self.theta = theta
        self.total = total
        self.indirect = indirect
        self.specific = specific


def build_parameters(spec, theta, theta_values, es, eff, bs):
    ids = [c.id for c in spec.constructs]
    out = []
    for i, t in enumerate(theta):
        fixed = t.single
        raw = t.raw_path
        if bs is not None and not fixed:
            raw += f"; {INFERENCE_NOTE}"
        if fixed:
            raw += "; single-indicator block: fixed at 1 by construction"
        out.append(
            param_entry(
                t.key,
                t.kind,
                theta_values[i],
                source_id=t.source_id,
                target_id=t.target_id,
                relationship_id=t.relationship_id,
                inf=bs.theta[i] if bs is not None and not fixed else None,
                boot=bs.boot if bs is not None and not fixed else None,
                fixed_reason="not-applicable" if fixed else None,
                source_method=SOURCE_PLS,
                raw_path=raw,
            )
        )
    note = f"; per draw; {INFERENCE_NOTE}" if bs is not None else ""
    for j, pr in enumerate(es.pairs):
        out.append(
            param_entry(
                f"total:{ids[pr.source]}->{ids[pr.target]}",
                "total-effect",
                eff.total[j],
                source_id=ids[pr.source],
                target_id=ids[pr.target],
                inf=bs.total[j] if bs is not None else None,
                boot=bs.boot if bs is not None else None,
                source_method=SOURCE_EFFECTS,
                raw_path=f"(I - B)^-1 - I, B = path coefficients{note}",
            )
        )
    for j, pr in enumerate(es.pairs):
        if not pr.has_indirect:
            continue
        out.append(
            param_entry(
                f"indirect:{ids[pr.source]}->{ids[pr.target]}",
                "indirect-effect",
                eff.indirect[j],
                source_id=ids[pr.source],
                target_id=ids[pr.target],
                inf=bs.indirect[j] if bs is not None else None,
                boot=bs.boot if bs is not None else None,
                source_method=SOURCE_EFFECTS,
                raw_path=f"(I - B)^-1 - I - B, B = path coefficients{note}",
            )
        )
    for j, sp in enumerate(es.specific):
        key = "specific-indirect:" + "->".join(ids[c] for c in sp.nodes)
        if len(key) > 400:
            continue
        out.append(
            param_entry(
                key,
                "specific-indirect",
                eff.specific[j],
                source_id=ids[sp.source],
                target_id=ids[sp.target],
                via_ids=[ids[c] for c in sp.via],
                inf=bs.specific[j] if bs is not None else None,
                boot=bs.boot if bs is not None else None,
                source_method=SOURCE_EFFECTS,
                raw_path=f"product of the path coefficients along the chain{note}",
            )
        )
    return out


# ---- construct metrics ---------------------------------------------------------------------

DEF = {
    "r2": "R²: share of the endogenous construct's variance explained by its predictors (OLS on the construct correlation matrix; cSEM Estimates$R2 definition). Defined for endogenous constructs only.",
    "r2adj": "Adjusted R² = 1 - (1 - R²)(n - 1)/(n - k - 1), n = rows used, k = number of predictors (cSEM Estimates$R2adj definition). Defined for endogenous constructs only.",
    "alpha": "Cronbach's alpha, standardized (correlation-based): k·r̄/(1 + (k - 1)·r̄), r̄ = mean inter-item correlation of the block (cSEM assess() RhoT definition). Reflective blocks with at least 2 indicators only.",
    "rhoa": "Dijkstra–Henseler rho_A of the Mode A weights: the PLSc (consistent PLS) reliability c²·(w′w)² of the same weights (cSEM Estimates$Reliabilities of a PLSc fit; the weights equal those of the classic fit). Reflective blocks with at least 2 indicators only.",
    "rhoc": "Composite reliability rho_C = (Σλ)² / ((Σλ)² + Σ(1 - λ²)) on the reported standardized loadings (cSEM assess() RhoC definition). Reflective blocks with at least 2 indicators only.",
    "ave": "Average variance extracted: mean of the squared reported standardized loadings (cSEM assess() AVE definition). Reflective blocks with at least 2 indicators only.",
}


def _metric_entry(construct_id, metric, value, reason, definition):
    v, r = _num_or_reason(value, reason)
    if r:
        return {
            "constructId": construct_id,
            "metric": metric,
            "value": None,
            "missing": r,
            "definition": definition,
        }
    return {"constructId": construct_id, "metric": metric, "value": v, "definition": definition}


def _safe(f):
    try:
        with np.errstate(all="ignore"):
            return f()
    except NumericalError:
        return None


def build_construct_metrics(spec, model, fit, S, rho_available):
    eligible = reliability_eligible(model)
    sigma_cf = _safe(lambda: rho_c_sigma(S, fit, model))
    out = []
    with np.errstate(all="ignore"):
        for j, c in enumerate(spec.constructs):
            endo = model.endo_mask[j]
            out.append(
                _metric_entry(
                    c.id,
                    "r2",
                    float(fit.R2[j]) if endo else None,
                    "failed" if endo else "not-applicable",
                    DEF["r2"],
                )
            )
            out.append(
                _metric_entry(
                    c.id,
                    "r2-adjusted",
                    float(fit.R2adj[j]) if endo else None,
                    "failed" if endo else "not-applicable",
                    DEF["r2adj"],
                )
            )
            el = eligible[j]
            block = c.indicators
            na = "not-applicable"
            out.append(
                _metric_entry(
                    c.id,
                    "cronbach-alpha",
                    cronbach_alpha(S, block) if el else None,
                    "failed" if el else na,
                    DEF["alpha"],
                )
            )
            ra = None
            if el and rho_available:
                ra = float(fit.Q[j]) ** 2 if fit.disattenuated else rho_a(S, fit.W, block, j)
            out.append(_metric_entry(c.id, "rho-a", ra, "failed" if el else na, DEF["rhoa"]))
            out.append(
                _metric_entry(
                    c.id,
                    "rho-c",
                    rho_c(sigma_cf, fit, j, block) if el and sigma_cf is not None else None,
                    "failed" if el else na,
                    DEF["rhoc"],
                )
            )
            out.append(
                _metric_entry(
                    c.id, "ave", ave(fit, j, block) if el else None, "failed" if el else na, DEF["ave"]
                )
            )
    return out


# ---- matrices ------------------------------------------------------------------------------


def _clean_row(row):
    return [float(v) if math.isfinite(v) else None for v in row]


def _matrix_entry(id, title, row_ids, col_ids, M, symmetric, definition, missing=None):
    m = {
        "id": id,
        "title": title,
        "rowIds": row_ids,
        "colIds": col_ids,
        "values": [_clean_row(r) for r in M] if row_ids and M is not None else [],
        "symmetric": symmetric,
        "definition": definition,
    }
    if missing:
        m["missing"] = missing
    return m


def build_matrices(spec, model, fit, S, n):
    ids = [c.id for c in spec.constructs]
    k = model.k
    eligible_idx = [j for j, e in enumerate(reliability_eligible(model)) if e]
    eligible = [ids[j] for j in eligible_idx]
    out = []
    for kind in ("htmt", "htmt2"):
        title = (
            "Heterotrait–monotrait ratio (HTMT)" if kind == "htmt" else "HTMT2 (geometric-mean HTMT)"
        )
        d = (
            "HTMT (Henseler, Ringle & Sarstedt, 2015): mean heterotrait correlation / sqrt(product of the mean monotrait correlations)"
            if kind == "htmt"
            else "HTMT2 (Roemer, Schuberth & Henseler, 2021): as HTMT with geometric instead of arithmetic means"
        ) + ", on Pearson indicator correlations, absolute value (cSEM assess() definition, .absolute = TRUE). Reflective constructs with at least 2 indicators only; other constructs are omitted (not applicable). Diagonal is null (not applicable)."
        if len(eligible) < 2:
            out.append(_matrix_entry(kind, title, [], [], None, True, d, "not-applicable"))
            continue
        M = [
            [
                math.nan if a == b else htmt_pair(S, model.blocks[a], model.blocks[b], kind)
                for b in eligible_idx
            ]
            for a in eligible_idx
        ]
        out.append(_matrix_entry(kind, title, eligible, eligible, M, True, d))

    fl_def = "Fornell–Larcker matrix (cSEM assess() fl_criterion definition): diagonal = AVE, off-diagonal = squared construct correlations. Reflective constructs with at least 2 indicators only."
    if not eligible:
        out.append(
            _matrix_entry(
                "fornell-larcker",
                "Fornell–Larcker criterion",
                [],
                [],
                None,
                True,
                fl_def,
                "not-applicable",
            )
        )
    else:
        M = [
            [ave(fit, a, model.blocks[a]) if a == b else float(fit.P[a, b]) ** 2 for b in eligible_idx]
            for a in eligible_idx
        ]
        out.append(
            _matrix_entry(
                "fornell-larcker", "Fornell–Larcker criterion", eligible, eligible, M, True, fl_def
            )
        )

    out.append(
        _matrix_entry(
            "cross-loadings",
            "Cross-loadings",
            [i.id for i in spec.indicators],
            ids,
            cross_loadings(fit, S, model),
            False,
            "Pearson correlation of each indicator with each construct score (standardized composites, as cSEM Estimates$Construct_scores)."
            + (
                " For PLSc the own-block entries are composite (proxy) correlations and therefore differ from the consistency-corrected loadings."
                if spec.method == "plsc"
                else ""
            ),
        )
    )
    out.append(
        _matrix_entry(
            "construct-correlations",
            "Construct correlations",
            ids,
            ids,
            fit.P,
            True,
            "Construct correlations (cSEM Estimates$Construct_VCV definition): consistency-corrected for reflective constructs (PLSc), composite correlations for Mode B constructs."
            if spec.method == "plsc"
            else "Construct correlations (cSEM Estimates$Construct_VCV definition): correlations of the standardized composites (classic PLS, no consistency correction).",
        )
    )

    endo = [j for j in range(k) if model.endo_mask[j]]
    preds = [j for j in range(k) if np.any(model.structural[:, j] == 1)]
    F = _safe(lambda: f2(fit, model, n))
    M = [
        [float(F[t, s]) if F is not None and model.structural[t, s] == 1 else math.nan for t in endo]
        for s in preds
    ]
    out.append(
        _matrix_entry(
            "f2",
            "Effect size f²",
            [ids[j] for j in preds],
            [ids[j] for j in endo],
            M,
            False,
            "f² = (R²_incl - R²_excl) / (1 - R²_incl) per structural path (row = predictor, column = endogenous construct), cSEM definition (assess() F2): R²_excl drops the predictor from the regression on the SAME construct correlations; the model is not re-estimated. Null where there is no path.",
            None if F is not None else "failed",
        )
    )

    def vif_cell(s, t):
        if model.structural[t, s] != 1:
            return math.nan
        n_pred = int(np.sum(model.structural[t] == 1))
        return 1.0 if n_pred == 1 else float(fit.vif[t, s])

    V = [[vif_cell(s, t) for t in endo] for s in preds]
    out.append(
        _matrix_entry(
            "vif-structural",
            "Structural VIF",
            [ids[j] for j in preds],
            [ids[j] for j in endo],
            V,
            False,
            "Variance inflation factor of each predictor in the regression of the endogenous construct (row = predictor, column = endogenous construct): diagonal of the inverse predictor correlation matrix (cSEM Estimates$VIF definition). Equals 1 by definition for a single predictor. Null where there is no path.",
        )
    )

    mode_b = [j for j in range(k) if model.mode_b[j]]
    vb_def = "Indicator VIF within each Mode B block: diagonal of the inverse indicator correlation matrix of the block (cSEM assess() VIF_modeB definition). Rows = indicators of Mode B blocks, columns = Mode B constructs; null outside the indicator's own block and for single-indicator blocks."
    if not mode_b:
        out.append(
            _matrix_entry(
                "vif-indicator-mode-b",
                "Indicator VIF (Mode B blocks)",
                [],
                [],
                None,
                False,
                vb_def,
                "not-applicable",
            )
        )
    else:
        rows = [a for j in mode_b for a in model.blocks[j]]
        Mv = [[math.nan] * len(mode_b) for _ in rows]
        failed = False
        for col, j in enumerate(mode_b):
            block = model.blocks[j]
            if len(block) < 2:
                continue
            v = _safe(lambda: vif_mode_b(S, block))
            if v is None:
                failed = True
                continue
            for idx, a in enumerate(block):
                Mv[rows.index(a)][col] = float(v[idx])
        out.append(
            _matrix_entry(
                "vif-indicator-mode-b",
                "Indicator VIF (Mode B blocks)",
                [spec.indicators[a].id for a in rows],
                [ids[j] for j in mode_b],
                Mv,
                False,
                vb_def,
                "failed" if failed else None,
            )
        )
    return out


# ---- fit ---------------------------------------------------------------------------------------

_FIT_DEFS = {
    "srmr": "Standardized root mean square residual: sqrt(mean of the squared differences between the empirical and the model-implied indicator correlations over the lower triangle INCLUDING the diagonal) (cSEM assess() SRMR definition; model-implied matrix as cSEM::fit()). No CB-SEM cutoffs apply.",
    "d_uls": "Unweighted least-squares discrepancy d_ULS = 0.5 · Σ_ij (s_ij - σ̂_ij)² over all elements of the empirical (S) and model-implied (Σ̂) indicator correlation matrices (cSEM assess() DL definition). No CB-SEM cutoffs apply.",
    "d_g": "Geodesic discrepancy d_G = 0.5 · Σ_i (log10 λ_i)², λ_i the eigenvalues of S⁻¹Σ̂ — base-10 logarithm as cSEM; the natural-log definition is (ln 10)² ≈ 5.30 times larger (cSEM assess() DG definition). No CB-SEM cutoffs apply.",
}


def build_fit(model, fit, S):
    fi = _safe(lambda: fit_indices(S, fit, model))
    values = {
        "srmr": fi.srmr if fi is not None else None,
        "d_uls": fi.d_uls if fi is not None else None,
        "d_g": fi.d_g if fi is not None else None,
    }
    out = []
    for name in ("srmr", "d_uls", "d_g"):
        v, r = _num_or_reason(values[name], "failed")
        if r:
            out.append(
                {
                    "name": name,
                    "value": None,
                    "missing": r,
                    "variant": "standard",
                    "definition": _FIT_DEFS[name],
                }
            )
        else:
            out.append(
                {"name": name, "value": v, "variant": "standard", "definition": _FIT_DEFS[name]}
            )
    return out


# ---- warnings ----------------------------------------------------------------------------------


def collect_warnings(spec, reversed_ids, verify_failed, theta, theta_values, rho_message, truncated, boot):
    """R collect_warnings(), preceded by the validation warnings."""
    out = list(spec.warnings)

    def add(code, severity, message, ids=None):
        w = {"code": code, "severity": severity, "message": message[:2000]}
        if ids:
            uniq = []
            for x in ids:
                if x not in uniq:
                    uniq.append(x)
            w["elementIds"] = uniq
        out.append(w)

    if reversed_ids:
        add(
            "REVERSE_SCORED_APPLIED",
            "info",
            f"{len(reversed_ids)} indicator(s) were reverse scored before estimation as (scale min + scale max) - x.",
            reversed_ids,
        )
    if verify_failed:
        add(
            "INADMISSIBLE_SOLUTION",
            "warning",
            f"The estimate is not admissible according to the cSEM::verify() checks: {'; '.join(VERIFY_CHECKS[c] for c in verify_failed)} failed. The numbers are reported but should not be interpreted without resolving this.",
        )
    neg = [
        t
        for i, t in enumerate(theta)
        if t.kind == "loading"
        and not t.single
        and math.isfinite(theta_values[i])
        and theta_values[i] < 0
    ]
    if neg:
        add(
            "NEGATIVE_LOADING",
            "warning",
            f"{len(neg)} loading(s) are negative. A construct's sign is arbitrary in PLS (the weights start from unit weights, so each block is oriented so that most loadings are positive); a negative loading usually means a reverse-worded item or a block whose indicators do not share one direction. Check item coding before interpreting path signs.",
            [t.source_id for t in neg] + [t.target_id for t in neg],
        )
    if rho_message:
        add("RHO_A_UNAVAILABLE", "warning", rho_message)
    if truncated:
        add(
            "SPECIFIC_INDIRECT_TRUNCATED",
            "warning",
            f"The structural model has more than {MAX_SPECIFIC_INDIRECT} indirect chains; only the first {MAX_SPECIFIC_INDIRECT} specific indirect effects are reported (total and total indirect effects are complete).",
        )
    b = boot
    if b is not None:
        if b.inadmissible > 0:
            add(
                "BOOTSTRAP_INADMISSIBLE_DROPPED",
                "warning" if b.inadmissible / b.requested > 0.05 else "info",
                f"{b.inadmissible} of {b.requested} bootstrap samples were inadmissible (cSEM::verify() checks) and were excluded from the standard errors and intervals.",
            )
        if b.failed > 0:
            add(
                "BOOTSTRAP_FAILED_DRAWS",
                "warning",
                f"{b.failed} of {b.requested} bootstrap samples could not be estimated (e.g. an indicator without variance in the resample, or no convergence) and were excluded.",
            )
        if b.completed < 2:
            add(
                "BOOTSTRAP_TOO_FEW_DRAWS",
                "error",
                f"Only {b.completed} valid bootstrap draw(s); standard errors, p values and intervals are not reported.",
            )
        if spec.bootstrap is not None and spec.bootstrap.sides == "one-sided":
            add(
                "ONE_SIDED_DIRECTION",
                "info",
                "One-sided inference is taken in the direction of each point estimate's sign: for est >= 0 the lower bound (quantile 1 - level) and p = P(T <= -|t|); for est < 0 the upper bound (quantile level).",
            )
    return out


# ---- estimate --------------------------------------------------------------------------------


def _cancelled(stage):
    return EngineError("CANCELLED", f"The job was cancelled {stage}.")


def estimate_pls(model, analysis, data, on_progress=None, is_cancelled=None, execute_draws=None, batch_size=25):
    """Run a PLS-SEM or PLSc analysis. Never raises: errors are returned as {'ok': False, 'error'}.

    on_progress(event: dict) receives {'stage', 'done'?, 'total'?, 'failed'?, 'inadmissible'?};
    is_cancelled() is polled between bootstrap draws; execute_draws(problem, batch) may fit a
    batch of resamples in parallel (results in the same order)."""
    try:
        with np.errstate(all="ignore"):
            return {
                "ok": True,
                "result": _run(
                    model, analysis, data, on_progress, is_cancelled, execute_draws, batch_size
                ),
            }
    except EngineError as e:
        return {"ok": False, "error": e.to_info()}
    except NumericalError:
        return {
            "ok": False,
            "error": {
                "code": "ENGINE_INTERNAL",
                "message": "A numerical step of the PLS estimation failed (singular or non-finite system).",
            },
        }
    except Exception:  # noqa: BLE001 - mirrors the TS engine's catch-all
        return {
            "ok": False,
            "error": {
                "code": "ENGINE_INTERNAL",
                "message": "Unexpected internal error in the browser PLS engine.",
            },
        }


def _run(model, analysis, data, on_progress, is_cancelled, execute_draws, batch_size):
    started_at = iso_now()

    def progress(e):
        if on_progress is not None:
            on_progress(e)

    def cancelled():
        return is_cancelled is not None and is_cancelled() is True

    progress({"stage": "validating"})
    spec = validate_request(model, analysis)
    prepared = pls_prepare_data(spec, data)
    if cancelled():
        raise _cancelled("before estimation")

    progress({"stage": "estimating"})
    cm = core_model(spec)
    plsc = spec.method == "plsc"
    S, _ = correlation(prepared.X)
    n = prepared.n
    try:
        fit = fit_pls(S, cm, plsc, n)
    except NumericalError:
        raise EngineError(
            "ENGINE_INTERNAL",
            "The PLS estimation failed (a singular or non-finite system occurred during estimation).",
            {"stage": "estimation"},
        ) from None
    if not fit.converged:
        raise EngineError(
            "ENGINE_NONCONVERGENCE",
            f"The PLS weight algorithm did not converge within {spec.max_iter} iterations (tolerance {js_str(spec.tol)}).",
            {"maxIterations": spec.max_iter, "tolerance": spec.tol, "iterations": fit.iterations},
        )
    verify_failed = verify(S, fit, cm)
    rho_available = True

    theta = theta_entries(spec)
    theta_values = extract_theta(fit, theta)
    es = effects_spec(cm.k, spec.paths)
    path_idx = [i for i, t in enumerate(theta) if t.kind == "path"]

    def effects_of(vals):
        return compute_effects(cm.k, spec.paths, [vals[i] for i in path_idx], es)

    eff = effects_of(theta_values)

    # ---- bootstrap -----------------------------------------------------------------------
    bs_inference = None
    boot_counts = None
    valid_effects = []
    if spec.bootstrap is not None:
        if cancelled():
            raise _cancelled("before bootstrapping")
        B = spec.bootstrap.reps
        problem = BootstrapProblem(cm, plsc, prepared.X, theta)
        bsz = max(1, batch_size)
        draws = []
        failed = 0
        inadm = 0
        progress({"stage": "bootstrapping", "done": 0, "total": B, "failed": 0, "inadmissible": 0})
        gen = bootstrap_indices(spec.bootstrap.seed, n, B)
        while len(draws) < B:
            batch = []
            while len(batch) < bsz and len(draws) + len(batch) < B:
                batch.append(next(gen))
            if execute_draws is not None:
                if cancelled():
                    raise _cancelled("during bootstrapping")
                results = list(execute_draws(problem, batch))
                if len(results) != len(batch):
                    raise EngineError(
                        "ENGINE_INTERNAL", "The bootstrap executor returned a wrong number of draws."
                    )
            else:
                results = []
                for idx in batch:
                    if cancelled():
                        raise _cancelled("during bootstrapping")
                    results.append(fit_draw(problem, idx))
            for r in results:
                draws.append(r)
                if r.status == "failed":
                    failed += 1
                elif r.status == "inadmissible":
                    inadm += 1
            progress(
                {
                    "stage": "bootstrapping",
                    "done": len(draws),
                    "total": B,
                    "failed": failed,
                    "inadmissible": inadm,
                }
            )
        if cancelled():
            raise _cancelled("during bootstrapping")
        boot_counts = summarise_bootstrap(draws)
        valid_effects = [effects_of(v) for v in boot_counts.valid]
        level = spec.bootstrap.level
        sides = spec.bootstrap.sides
        V = boot_counts.valid
        bs_inference = BootstrapInference(
            spec.bootstrap,
            [infer_from_draws(est, [v[i] for v in V], level, sides) for i, est in enumerate(theta_values)],
            [
                infer_from_draws(est, [e.total[j] for e in valid_effects], level, sides)
                for j, est in enumerate(eff.total)
            ],
            [
                infer_from_draws(est, [e.indirect[j] for e in valid_effects], level, sides)
                for j, est in enumerate(eff.indirect)
            ],
            [
                infer_from_draws(est, [e.specific[j] for e in valid_effects], level, sides)
                for j, est in enumerate(eff.specific)
            ],
        )
    if cancelled():
        raise _cancelled("before summarizing")

    # ---- summarise --------------------------------------------------------------------------
    progress({"stage": "summarizing"})
    parameters = build_parameters(spec, theta, theta_values, es, eff, bs_inference)
    construct_metrics = build_construct_metrics(spec, cm, fit, S, rho_available)
    matrices = build_matrices(spec, cm, fit, S, n)
    fit_idx = build_fit(cm, fit, S)
    warnings = collect_warnings(
        spec,
        prepared.reversed,
        verify_failed,
        theta,
        theta_values,
        None,
        es.truncated,
        boot_counts,
    )
    ipma = None
    estimator_text = (
        'consistent PLS (PLSc: Mode A weights + consistency correction, correction factor "dist_squared_euclid")'
        if plsc
        else "classic PLS-SEM (no consistency correction)"
    )
    calls = [
        f'PLS-PM (cSEM 0.7.1 semantics) on the Pearson indicator correlation matrix: {estimator_text}; reflective blocks Mode A, composite-formative blocks Mode B; starting weights = unit weights scaled to unit composite variance; inner weighting scheme "{spec.scheme}"; convergence: max |W_old - W_new| < {js_str(spec.tol)} within {spec.max_iter} iterations; paths by OLS on the construct correlations; no sign re-orientation.'
    ]
    if not plsc and any(cm.common_factor):
        calls.append("rho_A: PLSc correction (dist_squared_euclid) applied to the same weights.")
    if spec.bootstrap is not None:
        calls.append(
            f'Bootstrap: set.seed({spec.bootstrap.seed}, kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection"); for b in 1..{spec.bootstrap.reps}: idx <- sample.int({n}, {n}, replace = TRUE), refit the full model; failed (error / no convergence / non-finite) and inadmissible (verify) draws excluded; percentile CI (quantile type 7), p from t(df = B_valid - 1).'
        )
    if spec.ipma is not None:
        ipma, ipma_warnings = compute_ipma(spec, prepared)
        warnings.extend(ipma_warnings)
        calls.append(
            f'IPMA: classic PLS-SEM (scheme "{spec.scheme}", same tolerance and iteration cap) on the indicators rescaled to 0-100 with their declared endpoints.'
        )

    admissible = len(verify_failed) == 0
    messages = [
        f"PLS weights converged after {fit.iterations} iteration(s) (tolerance {js_str(spec.tol)}, criterion max |W_old - W_new|, at most {spec.max_iter} iterations).",
        "Estimator: "
        + ("consistent PLS (PLSc)" if plsc else "classic PLS-SEM (no consistency correction)")
        + f'; inner weighting scheme "{spec.scheme}".',
    ]
    if not admissible:
        messages.append(
            f"Admissibility (cSEM::verify() checks) failed: {'; '.join(VERIFY_CHECKS[c] for c in verify_failed)}."
        )
    messages = [m[:2000] for m in messages]

    bootstrap = None
    if spec.bootstrap is not None and boot_counts is not None:
        draw_map = {}
        for i in path_idx:
            draw_map[theta[i].key] = [js_signif(v[i], DRAW_SIGNIF) for v in boot_counts.valid]
        ids = [c.id for c in spec.constructs]
        for j, sp in enumerate(es.specific):
            key = "specific-indirect:" + "->".join(ids[c] for c in sp.nodes)
            if len(key) <= 400:
                draw_map[key] = [js_signif(e.specific[j], DRAW_SIGNIF) for e in valid_effects]
        bootstrap = {
            "requested": boot_counts.requested,
            "completed": boot_counts.completed,
            "failed": boot_counts.failed,
            "inadmissible": boot_counts.inadmissible,
            "seed": spec.bootstrap.seed,
            "ciMethod": "percentile",
            "confidenceLevel": spec.bootstrap.level,
            "chunkSize": 1,
            "draws": draw_map,
        }

    call = "\n".join(calls)
    if len(call) > 4000:
        call = call[:3980] + "\n[truncated]"
    result = {
        "schemaVersion": 1,
        "sample": {
            "nRows": prepared.n_rows,
            "nUsed": n,
            "nDropped": prepared.n_rows - n,
            "missingData": "listwise",
        },
        "estimation": {
            "converged": True,
            "iterations": fit.iterations,
            "admissible": admissible,
            "messages": messages,
        },
        "parameters": parameters,
        "constructMetrics": construct_metrics,
        "matrices": matrices,
        "fit": fit_idx,
    }
    if bootstrap is not None:
        result["bootstrap"] = bootstrap
    if ipma is not None:
        result["ipma"] = ipma
    result["warnings"] = warnings
    result["provenance"] = provenance(call, started_at, {"@sem-lab/engine": ENGINE_VERSION})
    return result


__all__ = [
    "ENGINE_VERSION",
    "estimate_pls",
    "theta_entries",
    "DrawResult",
]


# =============================================================================================
# semlab/distributions.py
# =============================================================================================
# Distribution functions for CB-SEM inference and fit (port of
# packages/engine/src/cb/distributions.ts; the same textbook algorithms, so that the RMSEA
# confidence limits follow the identical root-finder path):
#
#  - log Gamma by the Lanczos approximation (g = 7, 9 terms);
#  - regularized incomplete gamma P(a, x) / Q(a, x): power series for x < a + 1, modified Lentz
#    continued fraction otherwise;
#  - central chi^2 = incomplete gamma; noncentral chi^2 CDF as the Poisson(lambda/2) mixture of
#    central chi^2 CDFs, summed outwards from the Poisson mode;
#  - standard normal tail via Q(1/2, z^2/2);
#  - Brent's zeroin exactly as R's ``uniroot()`` (R_zeroin2), so lavaan's RMSEA confidence limits
#    (uniroot results with R's default tolerance) are reproduced, not merely approximated.

_EPS = 2.220446049250313e-16
_LANCZOS = (
    0.99999999999980993,
    676.5203681218851,
    -1259.1392167224028,
    771.32342877765313,
    -176.61502916214059,
    12.507343278686905,
    -0.13857109526572012,
    9.9843695780195716e-6,
    1.5056327351493116e-7,
)


def lgamma_lanczos(x):
    if x < 0.5:
        return math.log(math.pi / abs(math.sin(math.pi * x))) - lgamma_lanczos(1 - x)
    z = x - 1
    a = _LANCZOS[0]
    t = z + 7.5
    for i in range(1, 9):
        a += _LANCZOS[i] / (z + i)
    return 0.5 * math.log(2 * math.pi) + (z + 0.5) * math.log(t) - t + math.log(a)


def _gamma_prefix_log(a, x):
    """log of x^a e^-x / Gamma(a)."""
    return a * math.log(x) - x - lgamma_lanczos(a)


def _gamma_series(a, x):
    term = 1 / a
    s = term
    for n in range(1, 100000):
        term *= x / (a + n)
        s += term
        if abs(term) < abs(s) * 1e-17:
            break
    return s * math.exp(_gamma_prefix_log(a, x))


def _gamma_cf(a, x):
    tiny = 1e-300
    b = x + 1 - a
    c = 1 / tiny
    d = 1 / b
    h = d
    for i in range(1, 100000):
        an = -i * (i - a)
        b += 2
        d = an * d + b
        if abs(d) < tiny:
            d = tiny
        c = b + an / c
        if abs(c) < tiny:
            c = tiny
        d = 1 / d
        delta = d * c
        h *= delta
        if abs(delta - 1) < 1e-16:
            break
    return math.exp(_gamma_prefix_log(a, x)) * h


def gamma_p(a, x):
    """Regularized lower incomplete gamma P(a, x)."""
    if x <= 0:
        return 0.0
    if not math.isfinite(x):
        return 1.0
    if x < a + 1:
        return _gamma_series(a, x)
    return 1 - _gamma_cf(a, x)


def gamma_q(a, x):
    """Regularized upper incomplete gamma Q(a, x) = 1 - P(a, x), accurate in the far tail."""
    if x <= 0:
        return 1.0
    if not math.isfinite(x):
        return 0.0
    if x < a + 1:
        return 1 - _gamma_series(a, x)
    return _gamma_cf(a, x)


def pchisq(q, df):
    """Central chi^2 CDF (lower tail); df may be non-integer."""
    return gamma_p(df / 2, q / 2)


def pchisq_upper(q, df):
    """Central chi^2 upper tail P(X > q)."""
    return gamma_q(df / 2, q / 2)


def pnchisq(q, df, ncp):
    """Noncentral chi^2 CDF (lower tail) with noncentrality ncp >= 0."""
    if q <= 0:
        return 0.0
    if ncp <= 0:
        return pchisq(q, df)
    half = ncp / 2
    x = q / 2
    mode = math.floor(half)
    log_w0 = -half + mode * math.log(half) - lgamma_lanczos(mode + 1)
    s = 0.0
    # upward from the mode
    w = math.exp(log_w0)
    w_sum = 0.0
    j = mode
    while j < mode + 100000:
        if j > mode:
            w *= half / j
        term = w * gamma_p(df / 2 + j, x)
        s += term
        w_sum += w
        if j > mode + 5 and (w < 1e-18 or term < 1e-18 * s):
            break
        j += 1
    # downward from the mode
    w = math.exp(log_w0)
    j = mode - 1
    while j >= 0:
        w *= (j + 1) / half
        term = w * gamma_p(df / 2 + j, x)
        s += term
        w_sum += w
        if w < 1e-18 * max(w_sum, 1e-300):
            break
        j -= 1
    return min(1.0, s)


def pnorm_upper(z):
    """Upper tail of the standard normal, P(Z > z), accurate far into the tail."""
    if z == 0:
        return 0.5
    if z > 0:
        return 0.5 * gamma_q(0.5, (z * z) / 2)
    return 1 - 0.5 * gamma_q(0.5, (z * z) / 2)


def p_two_sided_normal(z):
    """Two-sided normal p value 2 P(Z > |z|), computed in the tail (no 1 - Phi cancellation)."""
    return min(1.0, 2 * pnorm_upper(abs(z)))


#: qnorm(0.975) to full double precision
Z_975 = 1.959963984540054


def zeroin(f, ax, bx, tol=_EPS**0.25, maxit=1000):
    """Brent's zeroin as R's uniroot() (R_zeroin2). None when the sign condition fails."""
    fa = f(ax)
    fb = f(bx)
    if not math.isfinite(fa) or not math.isfinite(fb) or fa * fb > 0:
        return None
    a = ax
    b = bx
    c = a
    fc = fa
    if fa == 0:
        return a
    if fb == 0:
        return b
    it = maxit + 1
    while it > 0:
        it -= 1
        prev_step = b - a
        if abs(fc) < abs(fb):
            a = b
            b = c
            c = a
            fa = fb
            fb = fc
            fc = fa
        tol_act = 2 * _EPS * abs(b) + tol / 2
        new_step = (c - b) / 2
        if abs(new_step) <= tol_act or fb == 0:
            return b
        if abs(prev_step) >= tol_act and abs(fa) > abs(fb):
            cb = c - b
            if a == c:
                t1 = fb / fa
                p = cb * t1
                q = 1 - t1
            else:
                qq = fa / fc
                t1 = fb / fc
                t2 = fb / fa
                p = t2 * (cb * qq * (qq - t1) - (b - a) * (t1 - 1))
                q = (qq - 1) * (t1 - 1) * (t2 - 1)
            if p > 0:
                q = -q
            else:
                p = -p
            if p < 0.75 * cb * q - abs(tol_act * q) / 2 and p < abs((prev_step * q) / 2):
                new_step = p / q
        if abs(new_step) < tol_act:
            new_step = tol_act if new_step > 0 else -tol_act
        a = b
        fa = fb
        b += new_step
        fb = f(b)
        if (fb > 0 and fc > 0) or (fb < 0 and fc < 0):
            c = a
            fc = fa
    return None


def _sqrt(v):
    """Math.sqrt: NaN (not an exception) for negative or NaN input."""
    return math.sqrt(v) if v >= 0 else math.nan


def rmsea_point(x2, df, n, c_hat=1.0):
    """RMSEA point estimate as lavaan::lav_fit_rmsea (single group)."""
    if not (df > 0):
        return 0.0
    return _sqrt(max(x2 / n / df - c_hat / n, 0))


def rmsea_ci(x2, df, n, c_hat=1.0, level=0.9):
    """RMSEA confidence interval as lavaan::lav_fit_rmsea_ci (uniroot on the noncentrality)."""
    upper_perc = 1 - (1 - level) / 2
    lower_perc = (1 - level) / 2

    def lower_fn(lam):
        return pnchisq(x2, df, lam) - upper_perc

    def upper_fn(lam):
        return pnchisq(x2, df, lam) - lower_perc

    if df < 1 or lower_fn(0) < 0:
        lower = 0.0
    else:
        root = zeroin(lower_fn, 0, x2)
        lower = None if root is None else _sqrt((c_hat * root) / (n * df))
    n_rmsea = max(n, x2 * 4)
    if df < 1 or upper_fn(n_rmsea) > 0 or upper_fn(0) < 0:
        upper = 0.0
    else:
        root = zeroin(upper_fn, 0, n_rmsea)
        upper = None if root is None else _sqrt((c_hat * root) / (n * df))
    return lower, upper


def rmsea_close_pvalue(x2, df, n, h0, c_hat=1.0):
    """lavaan::lav_fit_rmsea_closefit: P(RMSEA <= h0) test p value."""
    if not (df > 0):
        return None
    ncp = n * df * (1 / c_hat) * h0 * h0
    return 1 - pnchisq(x2, df, ncp)


# =============================================================================================
# semlab/cb.py
# =============================================================================================
# CB-SEM engine: ML / MLR estimation of reflective SEM models (complete data or FIML),
# reproducing lavaan 0.7 ``sem()`` defaults (port of packages/engine/src/cb; conventions in
# packages/engine/src/cb/README.md).
#
# Model (LISREL all-y): Sigma = Lambda (I - B)^-1 Psi (I - B)^-T Lambda' + Theta,
# mu = nu + Lambda (I - B)^-1 alpha, with lavaan's sem() free/fixed defaults (marker or std-lv
# identification, auto.fix.single, auto.cov.lv.x, auto.cov.y, equality labels).
#
# Estimation: f = sum_k (n_k/N)[log|Sigma_oo| + tr(Sigma_oo^-1 (S_k + d_k d_k'))] over
# missingness patterns; Fisher scoring with analytic gradient and backtracking line search,
# Newton steps on the finite-difference Hessian near the optimum. Inference: expected
# information (ML, complete), observed information (FIML), Huber-White sandwich (MLR);
# Yuan-Bentler (Mplus) scaled test, robust CFI/TLI/RMSEA, RMSEA CI by noncentral chi^2 inversion
# with R's uniroot.

CB_ENGINE_VERSION = "0.1.0"
_CB_EPS = 2.220446049250313e-16
_LOG_2PI = math.log(2 * math.pi)


# =============================================================================================
# spec


class CbEntry:
    __slots__ = (
        "key",
        "kind",
        "mat",
        "row",
        "col",
        "free",
        "value",
        "source_id",
        "target_id",
        "relationship_id",
        "label",
        "where",
    )

    def __init__(self, key, kind, mat, row, col, free, value, source_id=None, target_id=None,
                 relationship_id=None, label=None, where=""):
        self.key = key
        self.kind = kind
        self.mat = mat
        #: matrix position; beta: row = target, col = source; nu/alpha: col = 0
        self.row = row
        self.col = col
        #: index into the free-parameter vector, or -1 when fixed
        self.free = free
        self.value = value
        self.source_id = source_id
        self.target_id = target_id
        self.relationship_id = relationship_id
        self.label = label
        #: short location string, e.g. "lambda[3,1]" (1-based), used as rawPath
        self.where = where


class CbSpec:
    __slots__ = ("ov_ids", "lv_ids", "p", "m", "entries", "npar", "mean_structure",
                 "identification", "endogenous")

    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)


def _cb_find_cycle(adj):
    n = len(adj)
    state = [0] * n
    stack = []

    def visit(u):
        state[u] = 1
        stack.append(u)
        for v in adj[u]:
            if state[v] == 1:
                return stack[stack.index(v):] + [v]
            if state[v] == 0:
                c = visit(v)
                if c:
                    return c
        stack.pop()
        state[u] = 2
        return None

    for u in range(n):
        if state[u] == 0:
            c = visit(u)
            if c:
                return c
    return None


def _spec_fail(code, message, details=None):
    e = {"code": code, "message": message}
    if details is not None:
        e["details"] = details
    return None, e


def build_cb_spec(model, identification, mean_structure):
    """SemModel -> LISREL-style parameter table with lavaan's sem() default free/fixed pattern.
    Returns (spec, None) or (None, error)."""
    constructs = model.get("constructs") or []
    if len(constructs) == 0:
        return _spec_fail("REQUEST_INVALID", "The model has no constructs.", {"reason": "empty-model"})
    for c in constructs:
        if c.get("measurement") != "reflective":
            return _spec_fail(
                "CAPABILITY_UNSUPPORTED",
                f"Construct {c.get('id')} is composite-formative; CB-SEM estimates common factors only and does not convert composites.",
                {"reason": "composite-in-cb-sem", "elementIds": [c.get("id")]},
            )
        if c.get("higherOrder"):
            return _spec_fail(
                "CAPABILITY_UNSUPPORTED",
                f"Higher-order construct {c.get('id')} is not supported.",
                {"reason": "higher-order", "elementIds": [c.get("id")]},
            )
        if len(c.get("indicatorIds") or []) == 0:
            return _spec_fail(
                "REQUEST_INVALID",
                f"Construct {c.get('id')} has no indicators.",
                {"reason": "empty-block", "elementIds": [c.get("id")]},
            )

    lv_ids = [c["id"] for c in constructs]
    lv_index = {cid: i for i, cid in enumerate(lv_ids)}
    ov_ids = []
    for c in constructs:
        for i in c["indicatorIds"]:
            if i not in ov_ids:
                ov_ids.append(i)
    ov_index = {iid: i for i, iid in enumerate(ov_ids)}
    known = {i["id"] for i in model.get("indicators") or []}
    for iid in ov_ids:
        if iid not in known:
            return _spec_fail(
                "REQUEST_INVALID",
                f"Indicator {iid} is referenced but not defined.",
                {"reason": "unknown-indicator", "elementIds": [iid]},
            )
    p = len(ov_ids)
    m = len(lv_ids)
    rels = model.get("relationships") or []

    structural = [r for r in rels if r.get("kind") == "structural"]
    is_target = [False] * m
    is_source = [False] * m
    adj = [[] for _ in range(m)]
    for r in structural:
        s = lv_index.get(r.get("sourceId"))
        t = lv_index.get(r.get("targetId"))
        if s is None or t is None:
            return _spec_fail(
                "REQUEST_INVALID",
                f"Structural path {r.get('id')} must connect two constructs.",
                {"reason": "structural-endpoint", "elementIds": [r.get("id")]},
            )
        if s == t:
            return _spec_fail(
                "REQUEST_INVALID",
                f"Structural path {r.get('id')} is a self-loop.",
                {"reason": "self-loop", "elementIds": [r.get("id")]},
            )
        is_target[t] = True
        is_source[s] = True
        adj[s].append(t)
    cycle = _cb_find_cycle(adj)
    if cycle:
        return _spec_fail(
            "REQUEST_INVALID",
            f"The structural model contains a cycle ({' → '.join(lv_ids[i] for i in cycle)}); only recursive models are supported.",
            {"reason": "structural-cycle", "elementIds": [lv_ids[i] for i in cycle]},
        )

    entries = []

    def push(key, kind, mat, row, col, is_free, value, **kw):
        entries.append(CbEntry(key, kind, mat, row, col, 0 if is_free else -1, value, **kw))

    # loadings
    load_count = {}
    for c in constructs:
        for i in c["indicatorIds"]:
            load_count[i] = load_count.get(i, 0) + 1
    for j, c in enumerate(constructs):
        for t, iid in enumerate(c["indicatorIds"]):
            i = ov_index[iid]
            fixed = identification == "marker" and t == 0
            push(
                f"loading:{c['id']}:{iid}",
                "loading",
                "lambda",
                i,
                j,
                not fixed,
                1.0 if fixed else 0.0,
                source_id=c["id"],
                target_id=iid,
                where=f"lambda[{i + 1},{j + 1}]",
            )

    # structural paths
    for r in structural:
        s = lv_index[r["sourceId"]]
        t = lv_index[r["targetId"]]
        fv = r.get("fixedValue")
        push(
            f"path:{r['sourceId']}->{r['targetId']}",
            "path",
            "beta",
            t,
            s,
            fv is None,
            float(fv) if fv is not None else 0.0,
            source_id=r["sourceId"],
            target_id=r["targetId"],
            relationship_id=r["id"],
            label=r.get("parameterLabel") or None,
            where=f"beta[{t + 1},{s + 1}]",
        )

    # indicator residual variances
    single = {c["indicatorIds"][0] for c in constructs if len(c["indicatorIds"]) == 1}
    for i, iid in enumerate(ov_ids):
        fix_zero = iid in single and load_count.get(iid) == 1
        push(
            f"residual-variance:{iid}",
            "residual-variance",
            "theta",
            i,
            i,
            not fix_zero,
            0.0,
            source_id=iid,
            target_id=iid,
            where=f"theta[{i + 1},{i + 1}]",
        )

    # latent (residual) variances
    for j, cid in enumerate(lv_ids):
        endo = is_target[j]
        push(
            f"{'residual-variance' if endo else 'variance'}:{cid}",
            "residual-variance" if endo else "variance",
            "psi",
            j,
            j,
            identification == "marker",
            0.0 if identification == "marker" else 1.0,
            source_id=cid,
            target_id=cid,
            where=f"psi[{j + 1},{j + 1}]",
        )

    # covariances: explicit relationships first (they override automatic ones)
    def cov_key(a, b):
        return "\u0000".join(sorted([a, b]))

    explicit = {}
    for r in rels:
        if r.get("kind") != "covariance":
            continue
        sl = lv_index.get(r.get("sourceId"))
        tl = lv_index.get(r.get("targetId"))
        so = ov_index.get(r.get("sourceId"))
        to = ov_index.get(r.get("targetId"))
        if sl is not None and tl is not None:
            mat = "psi"
            a, b = max(sl, tl), min(sl, tl)
        elif so is not None and to is not None:
            mat = "theta"
            a, b = max(so, to), min(so, to)
        else:
            return _spec_fail(
                "REQUEST_INVALID",
                f"Covariance {r.get('id')} must connect two constructs or two indicators of the model.",
                {"reason": "covariance-endpoint", "elementIds": [r.get("id")]},
            )
        if a == b:
            return _spec_fail(
                "REQUEST_INVALID",
                f"Covariance {r.get('id')} connects an element with itself.",
                {"reason": "self-covariance", "elementIds": [r.get("id")]},
            )
        k = cov_key(r["sourceId"], r["targetId"])
        if k in explicit:
            return _spec_fail(
                "REQUEST_INVALID",
                f"Covariance {r.get('id')} duplicates another covariance.",
                {"reason": "duplicate-covariance", "elementIds": [r.get("id")]},
            )
        fv = r.get("fixedValue")
        explicit[k] = CbEntry(
            f"covariance:{r['sourceId']}~~{r['targetId']}",
            "covariance",
            mat,
            a,
            b,
            0 if fv is None else -1,
            float(fv) if fv is not None else 0.0,
            source_id=r["sourceId"],
            target_id=r["targetId"],
            relationship_id=r["id"],
            label=r.get("parameterLabel") or None,
            where=f"{mat}[{a + 1},{b + 1}]",
        )
    exo = [not is_target[j] for j in range(m)]
    pure_y = [is_target[j] and not is_source[j] for j in range(m)]
    for a in range(m):
        for b in range(a + 1, m):
            if (exo[a] and exo[b]) or (pure_y[a] and pure_y[b]):
                if cov_key(lv_ids[a], lv_ids[b]) in explicit:
                    continue
                entries.append(
                    CbEntry(
                        f"covariance:{lv_ids[a]}~~{lv_ids[b]}",
                        "covariance",
                        "psi",
                        b,
                        a,
                        0,
                        0.0,
                        source_id=lv_ids[a],
                        target_id=lv_ids[b],
                        where=f"psi[{b + 1},{a + 1}]",
                    )
                )
    # explicit covariances in model order, after the automatic ones
    entries.extend(explicit.values())

    if mean_structure:
        for i, iid in enumerate(ov_ids):
            entries.append(
                CbEntry(f"intercept:{iid}", "intercept", "nu", i, 0, 0, 0.0, source_id=iid,
                        where=f"nu[{i + 1}]")
            )
        for j, cid in enumerate(lv_ids):
            entries.append(
                CbEntry(f"intercept:{cid}", "intercept", "alpha", j, 0, -1, 0.0, source_id=cid,
                        where=f"alpha[{j + 1}]")
            )

    # number the free parameters; equal labels share one parameter
    npar = 0
    by_label = {}
    for e in entries:
        if e.free < 0:
            continue
        if e.label is not None:
            prev = by_label.get(e.label)
            if prev is not None:
                e.free = prev
                continue
            by_label[e.label] = npar
        e.free = npar
        npar += 1

    return (
        CbSpec(
            ov_ids=ov_ids,
            lv_ids=lv_ids,
            p=p,
            m=m,
            entries=entries,
            npar=npar,
            mean_structure=mean_structure,
            identification=identification,
            endogenous=is_target,
        ),
        None,
    )


def sample_moments(p, mean_structure):
    """Number of observed sample moments: p(p+1)/2 (+ p with a mean structure)."""
    return (p * (p + 1)) // 2 + (p if mean_structure else 0)


def cb_parameter_count(model, identification="marker", mean_structure=False):
    """{moments, parameters, df} exactly as the CB engine (and lavaan sem()) uses them, or None."""
    spec, err = build_cb_spec(model, identification, mean_structure)
    if spec is None:
        return None
    moments = sample_moments(spec.p, mean_structure)
    return {"moments": moments, "parameters": spec.npar, "df": moments - spec.npar}


# =============================================================================================
# small matrix kernel (exact counterparts of cb/matrix.ts where the algorithm matters)


def gj_inverse(a):
    """General inverse by Gauss-Jordan with partial pivoting (cb/matrix.ts ``inverse``); None
    when (numerically) singular: a pivot <= max|a| * n * eps."""
    a = np.asarray(a, dtype=float)
    n = a.shape[0]
    if n == 0:
        return np.zeros((0, 0))
    if not np.all(np.isfinite(a)):
        return None
    m = a.copy()
    inv = np.eye(n)
    tiny = float(np.max(np.abs(a))) * n * _CB_EPS
    for col in range(n):
        piv = col + int(np.argmax(np.abs(m[col:, col])))
        best = abs(m[piv, col])
        if not (best > tiny):
            return None
        if piv != col:
            m[[col, piv]] = m[[piv, col]]
            inv[[col, piv]] = inv[[piv, col]]
        d = m[col, col]
        m[col] = m[col] / d
        inv[col] = inv[col] / d
        f = m[:, col].copy()
        f[col] = 0.0
        m -= np.outer(f, m[col])
        inv -= np.outer(f, inv[col])
    return inv


def gj_solve(a, b):
    inv = gj_inverse(a)
    return None if inv is None else inv @ b


def cholesky_lower(a):
    """Lower Cholesky factor, or None when not (numerically) positive definite."""
    try:
        L = np.linalg.cholesky(a)
    except np.linalg.LinAlgError:
        return None
    if not np.all(np.isfinite(L)):
        return None
    return L


def spd_inverse(a):
    """(inverse, log-determinant) of a symmetric positive-definite matrix; None if not PD."""
    L = cholesky_lower(a)
    if L is None:
        return None
    logdet = 2.0 * float(np.sum(np.log(np.diag(L))))
    Li = solve_triangular(L, np.eye(a.shape[0]), lower=True)
    inv = Li.T @ Li
    iu = np.triu_indices(a.shape[0], 1)
    inv[iu] = inv.T[iu]
    return inv, logdet


def sym_eigen(a):
    """Eigen-decomposition of a symmetric matrix: values descending, vectors as columns."""
    w, v = np.linalg.eigh(a)
    return w[::-1].copy(), v[:, ::-1].copy()


def sym_eigenvalues(a):
    return np.linalg.eigvalsh(a)[::-1].copy()


# =============================================================================================
# data


class Pattern:
    __slots__ = ("obs", "n", "mean", "cov", "rows", "full")

    def __init__(self, obs, n, mean, cov, rows, full):
        #: observed variable indices (into the model's ov order)
        self.obs = obs
        self.n = n
        self.mean = mean
        #: divisor-n covariance of the pattern's observed values
        self.cov = cov
        #: observed values of each case (n x len(obs))
        self.rows = rows
        self.full = full


class PatternGroup:
    """Missingness patterns with the same number q of observed variables, stacked so that the
    per-pattern likelihood terms are evaluated in one batched (LAPACK) call; the arithmetic per
    pattern is unchanged."""

    __slots__ = ("q", "idx", "rows_ix", "cols_ix", "w", "mean", "cov")

    def __init__(self, q, patterns, n_total):
        self.q = q
        self.idx = np.array([pt.obs for pt in patterns], dtype=int)  # k x q
        self.rows_ix = self.idx[:, :, None]
        self.cols_ix = self.idx[:, None, :]
        self.w = np.array([pt.n / n_total for pt in patterns])
        self.mean = np.array([pt.mean for pt in patterns])
        self.cov = np.array([pt.cov for pt in patterns])


class SampleData:
    __slots__ = ("p", "n", "patterns", "complete", "mean", "cov", "groups")

    def __init__(self, p, n, patterns, complete, mean, cov):
        self.p = p
        self.n = n
        self.patterns = patterns
        self.complete = complete
        self.mean = mean
        self.cov = cov
        by_q = {}
        for pt in patterns:
            by_q.setdefault(pt.obs.shape[0], []).append(pt)
        self.groups = [PatternGroup(q, by_q[q], n) for q in sorted(by_q)]


def _is_finite_number(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool) and math.isfinite(x)


def cb_prepare_data(model, ov_ids, inp, missing):
    """Column resolution, listwise deletion or FIML pattern grouping, divisor-N moments.
    Returns ((data, n_used, n_excluded), None) or (None, error)."""

    def fail(code, message, details=None):
        e = {"code": code, "message": message}
        if details is not None:
            e["details"] = details
        return None, e

    rows_in = inp.get("rows") or []
    columns = list(inp.get("columns") or [])
    if inp.get("nRows") - inp.get("nDropped") != len(rows_in):
        return fail(
            "REQUEST_INVALID",
            f"nRows − nDropped ({inp.get('nRows')} − {inp.get('nDropped')}) does not equal the number of rows supplied ({len(rows_in)}).",
            {"reason": "data-preparation-mismatch"},
        )
    by_id = {i["id"]: i for i in model.get("indicators") or []}
    col_index = {}
    for i, c in enumerate(columns):
        col_index[c] = i  # Map semantics: the last occurrence wins
    unmapped = []
    absent = []
    cols = []
    for iid in ov_ids:
        col = (by_id.get(iid) or {}).get("column")
        if col is None:
            unmapped.append(iid)
            continue
        k = col_index.get(col)
        if k is None:
            absent.append(col)
        else:
            cols.append(k)
    if unmapped or absent:
        parts = []
        if unmapped:
            parts.append(f"indicators without a bound column: {', '.join(unmapped)}")
        if absent:
            parts.append(f"columns not in the data: {', '.join(absent)}")
        return fail("DATA_COLUMN_MISSING", "; ".join(parts), {"elementIds": unmapped, "columns": absent})

    p = len(ov_ids)
    used = []
    for r, row in enumerate(rows_in):
        if len(row) != len(columns):
            return fail(
                "REQUEST_INVALID",
                f"Row {r + 1} has {len(row)} values for {len(columns)} columns.",
                {"reason": "ragged-rows"},
            )
        v = np.empty(p)
        n_obs = 0
        for j in range(p):
            x = row[cols[j]]
            if x is None:
                v[j] = math.nan
            elif not _is_finite_number(x):
                return fail(
                    "DATA_NOT_NUMERIC",
                    f"Column {columns[cols[j]]} contains a non-numeric or non-finite value.",
                    {"columns": [columns[cols[j]]], "elementIds": [ov_ids[j]]},
                )
            else:
                v[j] = float(x)
                n_obs += 1
        if (n_obs == p) if missing == "listwise" else (n_obs > 0):
            used.append(v)
    n = len(used)
    if n < 2:
        return fail(
            "DATA_INSUFFICIENT_ROWS",
            f"Only {n} usable row(s) after {'listwise deletion' if missing == 'listwise' else 'removing empty rows'}.",
            {"nUsed": n},
        )
    U = np.array(used)
    zero = []
    for j in range(p):
        col = U[:, j]
        obs = col[~np.isnan(col)]
        if obs.shape[0] < 2 or float(np.min(obs)) == float(np.max(obs)):
            zero.append(ov_ids[j])
    if zero:
        return fail(
            "DATA_ZERO_VARIANCE",
            f"Indicator(s) without variance in the analysed rows: {', '.join(zero)}.",
            {"elementIds": zero},
        )

    # group into missingness patterns (first-appearance order)
    groups = {}
    for v in used:
        obs = tuple(j for j in range(p) if not math.isnan(v[j]))
        g = groups.get(obs)
        if g is None:
            g = []
            groups[obs] = g
        g.append(v[list(obs)])
    patterns = []
    for obs, rows in groups.items():
        R = np.array(rows)
        k = R.shape[0]
        mean = np.sum(R, axis=0) / k
        D = R - mean
        cov = (D.T @ D) / k
        il = np.tril_indices(len(obs), -1)
        cov[il[1], il[0]] = cov[il]
        patterns.append(Pattern(np.array(obs, dtype=int), k, mean, cov, R, len(obs) == p))
    complete = len(patterns) == 1 and patterns[0].full
    data = SampleData(
        p,
        n,
        patterns,
        complete,
        patterns[0].mean if complete else None,
        patterns[0].cov if complete else None,
    )
    return (data, n, len(rows_in) - n), None


def em_saturated(data, tol=1e-13, max_iter=20000):
    """Saturated ML moments under MAR (EM). Start: available-case means and divisor-n
    variances (as lavaan); iterate to max |delta| < tol. None if a conditional covariance
    becomes singular. Returns (mu, sigma, iterations, converged)."""
    p = data.p
    if data.complete:
        return data.mean, data.cov, 0, True
    s = np.zeros(p)
    cnt = np.zeros(p)
    for pat in data.patterns:
        s[pat.obs] += pat.n * pat.mean
        cnt[pat.obs] += pat.n
    mu = s / cnt
    ss = np.zeros(p)
    for pat in data.patterns:
        ss[pat.obs] += np.sum((pat.rows - mu[pat.obs]) ** 2, axis=0)
    sigma = np.diag(ss / cnt)
    n = data.n
    for it in range(1, max_iter + 1):
        T1 = np.zeros(p)
        T2 = np.zeros((p, p))
        for pat in data.patterns:
            o = pat.obs
            nk = pat.n
            sy = nk * pat.mean
            syy = nk * (pat.cov + np.outer(pat.mean, pat.mean))
            T1[o] += sy
            T2[np.ix_(o, o)] += syy
            if pat.full:
                continue
            mis = np.array([j for j in range(p) if j not in set(o.tolist())], dtype=int)
            res = spd_inverse(sigma[np.ix_(o, o)])
            if res is None:
                return None
            inv = res[0]
            Bm = sigma[np.ix_(mis, o)] @ inv
            C = sigma[np.ix_(mis, mis)] - Bm @ sigma[np.ix_(o, mis)]
            a = mu[mis] - Bm @ mu[o]
            bsy = Bm @ sy
            bSyy = Bm @ syy
            T1[mis] += nk * a + bsy
            cross = np.outer(a, sy) + bSyy
            T2[np.ix_(mis, o)] += cross
            T2[np.ix_(o, mis)] += cross.T
            T2[np.ix_(mis, mis)] += (
                nk * C + nk * np.outer(a, a) + np.outer(a, bsy) + np.outer(bsy, a) + bSyy @ Bm.T
            )
        mu_new = T1 / n
        sig_new = (0.5 * (T2 + T2.T)) / n - np.outer(mu_new, mu_new)
        il = np.tril_indices(p, -1)
        sig_new[il[1], il[0]] = sig_new[il]
        delta = max(float(np.max(np.abs(mu_new - mu))), float(np.max(np.abs(sig_new - sigma))))
        mu = mu_new
        sigma = sig_new
        if delta < tol:
            return mu, sigma, it, True
    return mu, sigma, max_iter, False


# =============================================================================================
# parameterizations


class ModelState:
    __slots__ = ("lam", "beta", "psi", "theta", "nu", "alpha", "a", "phi", "sigma", "mu", "la",
                 "a_alpha")


def fill_matrices(spec, x):
    p, m = spec.p, spec.m
    st = ModelState()
    st.lam = np.zeros((p, m))
    st.beta = np.zeros((m, m))
    st.psi = np.zeros((m, m))
    st.theta = np.zeros((p, p))
    st.nu = np.zeros(p)
    st.alpha = np.zeros(m)
    for e in spec.entries:
        v = x[e.free] if e.free >= 0 else e.value
        if e.mat == "lambda":
            st.lam[e.row, e.col] = v
        elif e.mat == "beta":
            st.beta[e.row, e.col] = v
        elif e.mat == "psi":
            st.psi[e.row, e.col] = v
            st.psi[e.col, e.row] = v
        elif e.mat == "theta":
            st.theta[e.row, e.col] = v
            st.theta[e.col, e.row] = v
        elif e.mat == "nu":
            st.nu[e.row] = v
        elif e.mat == "alpha":
            st.alpha[e.row] = v
    return st


def model_state(spec, x):
    st = fill_matrices(spec, x)
    m = spec.m
    a = np.zeros((0, 0)) if m == 0 else gj_inverse(np.eye(m) - st.beta)
    if a is None:
        return None
    st.a = a
    st.phi = (a @ st.psi) @ a.T
    st.la = st.lam @ a
    sigma = (st.lam @ st.phi) @ st.lam.T + st.theta
    sigma = 0.5 * (sigma + sigma.T)  # exact symmetry
    st.sigma = sigma
    st.a_alpha = a @ st.alpha
    st.mu = st.nu + st.la @ st.alpha if spec.mean_structure else None
    return st


class ModelParameterization:
    """theta -> (Sigma, mu) of the LISREL all-y model with the analytic Jacobian."""

    def __init__(self, spec):
        self.spec = spec
        self.npar = spec.npar
        self.p = spec.p
        self.mean_structure = spec.mean_structure
        self._free = [e for e in spec.entries if e.free >= 0]
        idx = np.array([e.free for e in self._free], dtype=int)
        self._idx = idx
        self._dup = len(set(idx.tolist())) != len(idx)

    def moments(self, x):
        st = model_state(self.spec, x)
        return None if st is None else (st.sigma, st.mu)

    def jacobian(self, x):
        st = model_state(self.spec, x)
        if st is None:
            return None
        p = self.p
        lphi = st.lam @ st.phi  # Lambda Phi (p x m)
        ne = len(self._free)
        U = np.zeros((ne, p))
        V = np.zeros((ne, p))
        Mu = np.zeros((ne, p)) if self.mean_structure else None
        for t, e in enumerate(self._free):
            r, c = e.row, e.col
            if e.mat == "lambda":
                U[t, r] = 1.0
                V[t] = lphi[:, c]
                if Mu is not None:
                    Mu[t, r] += st.a_alpha[c]
            elif e.mat == "beta":
                U[t] = st.la[:, r]
                V[t] = lphi[:, c]
                if Mu is not None:
                    Mu[t] += st.la[:, r] * st.a_alpha[c]
            elif e.mat == "psi":
                U[t] = st.la[:, r]
                V[t] = 0.5 * st.la[:, r] if r == c else st.la[:, c]
            elif e.mat == "theta":
                U[t, r] = 1.0
                if r == c:
                    V[t, r] = 0.5
                else:
                    V[t, c] = 1.0
            elif e.mat == "nu":
                if Mu is not None:
                    Mu[t, r] += 1.0
            elif e.mat == "alpha":
                if Mu is not None:
                    Mu[t] += st.la[:, r]
        D = U[:, :, None] * V[:, None, :]
        D = D + D.transpose(0, 2, 1)
        if self._dup:
            dS = np.zeros((self.npar, p, p))
            np.add.at(dS, self._idx, D)
            dM = None
            if Mu is not None:
                dM = np.zeros((self.npar, p))
                np.add.at(dM, self._idx, Mu)
        else:
            dS = np.zeros((self.npar, p, p))
            dS[self._idx] = D
            dM = None
            if Mu is not None:
                dM = np.zeros((self.npar, p))
                dM[self._idx] = Mu
        return dS, dM


class SaturatedParameterization:
    """theta = (sigma_ij for i >= j, row-wise lower triangle; then mu when modelled)."""

    def __init__(self, p, mean_structure):
        self.p = p
        self.mean_structure = mean_structure
        self.pairs = [(i, j) for i in range(p) for j in range(i + 1)]
        n_cov = len(self.pairs)
        self.n_cov = n_cov
        self.npar = n_cov + (p if mean_structure else 0)
        dS = np.zeros((self.npar, p, p))
        dM = np.zeros((self.npar, p)) if mean_structure else None
        for k, (i, j) in enumerate(self.pairs):
            dS[k, i, j] = 1.0
            dS[k, j, i] = 1.0
        if mean_structure:
            for i in range(p):
                dM[n_cov + i, i] = 1.0
        self._jac = (dS, dM)
        self._ri = np.array([i for i, _ in self.pairs], dtype=int)
        self._ci = np.array([j for _, j in self.pairs], dtype=int)

    def moments(self, x):
        sigma = np.zeros((self.p, self.p))
        sigma[self._ri, self._ci] = x[: self.n_cov]
        sigma[self._ci, self._ri] = x[: self.n_cov]
        return sigma, (np.array(x[self.n_cov :], dtype=float) if self.mean_structure else None)

    def jacobian(self, x):
        return self._jac


def saturated_theta(sigma, mu):
    p = sigma.shape[0]
    out = [sigma[i, j] for i in range(p) for j in range(i + 1)]
    if mu is not None:
        out.extend(mu.tolist())
    return np.array(out, dtype=float)


class IndependenceParameterization:
    """theta = (sigma_11 ... sigma_pp; then mu when modelled)."""

    def __init__(self, p, mean_structure):
        self.p = p
        self.mean_structure = mean_structure
        self.npar = p + (p if mean_structure else 0)
        dS = np.zeros((self.npar, p, p))
        dM = np.zeros((self.npar, p)) if mean_structure else None
        for k in range(p):
            dS[k, k, k] = 1.0
        if mean_structure:
            for i in range(p):
                dM[p + i, i] = 1.0
        self._jac = (dS, dM)

    def moments(self, x):
        return np.diag(np.asarray(x[: self.p], dtype=float)), (
            np.array(x[self.p :], dtype=float) if self.mean_structure else None
        )

    def jacobian(self, x):
        return self._jac


# =============================================================================================
# objective


def _effective_mu(data, mu):
    if mu is not None:
        return mu
    if data.mean is None:
        raise RuntimeError("a mean structure is required for incomplete data")
    return data.mean


def _sub(M, pat):
    return M if pat.full else M[np.ix_(pat.obs, pat.obs)]


def evaluate(data, sigma, mu):
    """f and its derivatives G = df/dSigma, g = df/dmu (None without mean structure). None when
    some Sigma_oo is not PD. Patterns are evaluated in stacks of equal size (PatternGroup)."""
    p = data.p
    mu_e = _effective_mu(data, mu)
    G = np.zeros((p, p))
    g = np.zeros(p) if mu is not None else None
    f = 0.0
    for grp in data.groups:
        S = sigma[grp.rows_ix, grp.cols_ix]  # k x q x q
        if not np.all(np.isfinite(S)):
            return None
        try:
            L = np.linalg.cholesky(S)
        except np.linalg.LinAlgError:
            return None
        diag = np.diagonal(L, axis1=1, axis2=2)
        if not np.all(diag > 0):
            return None
        logdet = 2.0 * np.sum(np.log(diag), axis=1)
        Li = np.linalg.inv(L)
        inv = np.matmul(np.swapaxes(Li, 1, 2), Li)
        inv = 0.5 * (inv + np.swapaxes(inv, 1, 2))
        d = grp.mean - mu_e[grp.idx]
        M = grp.cov + d[:, :, None] * d[:, None, :]
        sm = np.matmul(inv, M)
        tr = np.trace(sm, axis1=1, axis2=2)
        f += float(np.sum(grp.w * (logdet + tr)))
        Gk = grp.w[:, None, None] * (inv - np.matmul(sm, inv))
        np.add.at(G, (grp.rows_ix, grp.cols_ix), Gk)
        if g is not None:
            gk = -2 * grp.w[:, None] * np.matmul(inv, d[:, :, None])[:, :, 0]
            np.add.at(g, grp.idx, gk)
    if not math.isfinite(f):
        return None
    return f, G, g


def _gradient(G, g, jac):
    dS, dM = jac
    out = np.tensordot(dS, G, axes=([1, 2], [0, 1]))
    if g is not None and dM is not None:
        out = out + dM @ g
    return out


def objective_and_gradient(par, data, x):
    mom = par.moments(x)
    if mom is None:
        return None
    ev = evaluate(data, mom[0], mom[1])
    if ev is None:
        return None
    jac = par.jacobian(x)
    if jac is None:
        return None
    return ev[0], _gradient(ev[1], ev[2], jac)


def objective_only(par, data, x):
    mom = par.moments(x)
    if mom is None:
        return None
    ev = evaluate(data, mom[0], mom[1])
    return None if ev is None else ev[0]


def expected_hessian(par, data, x):
    """Expected Hessian of f: sum_k (n_k/N)[tr(S^-1 S_a S^-1 S_b) + 2 mu_a' S^-1 mu_b]."""
    mom = par.moments(x)
    jac = par.jacobian(x)
    if mom is None or jac is None:
        return None
    sigma, mu = mom
    dS, dM = jac
    npar = dS.shape[0]
    H = np.zeros((npar, npar))
    for pat in data.patterns:
        o = pat.obs
        q = o.shape[0]
        res = spd_inverse(_sub(sigma, pat))
        if res is None:
            return None
        inv = res[0]
        w = pat.n / data.n
        dSo = dS if pat.full else dS[:, o][:, :, o]
        Ms = np.matmul(inv[None, :, :], dSo)
        A = Ms.reshape(npar, q * q)
        Bt = Ms.transpose(0, 2, 1).reshape(npar, q * q)
        T = A @ Bt.T
        if dM is not None and mu is not None:
            dMo = dM[:, o]
            Vs = dMo @ inv.T
            T = T + 2 * (dMo @ Vs.T)
        T = np.tril(T)
        T = T + np.tril(T, -1).T
        H += w * T
    return H


def casewise_scores(par, data, x):
    """Casewise scores of logL_i with respect to theta (rows: patterns in order, cases in order)."""
    mom = par.moments(x)
    jac = par.jacobian(x)
    if mom is None or jac is None:
        return None
    sigma, mu_m = mom
    mu = _effective_mu(data, mu_m)
    dS, dM = jac
    npar = dS.shape[0]
    out = []
    for pat in data.patterns:
        o = pat.obs
        res = spd_inverse(_sub(sigma, pat))
        if res is None:
            return None
        inv = res[0]
        dSo = dS if pat.full else dS[:, o][:, :, o]
        Z = pat.rows - mu[o]
        Vv = Z @ inv.T
        # 0.5 * sum_ij (v_i v_j - inv_ij) dS_a,ij + v . dmu_a
        quad = np.einsum("ri,aij,rj->ra", Vv, dSo, Vv, optimize=True)
        tr = np.einsum("ij,aij->a", inv, dSo)
        s = 0.5 * (quad - tr[None, :])
        if dM is not None and mu_m is not None:
            s = s + Vv @ dM[:, o].T
        out.append(s.reshape(-1, npar))
    return np.vstack(out) if out else np.zeros((0, npar))


def first_order_information(par, data, x):
    """B = (1/N) sum_i s_i s_i'."""
    sc = casewise_scores(par, data, x)
    if sc is None:
        return None
    B = (sc.T @ sc) / data.n
    iu = np.triu_indices(B.shape[0], 1)
    B[iu] = B.T[iu]
    return B


def observed_information(par, data, x, h=1e-6):
    """Observed information per case (Hessian of f/2) by Richardson extrapolation of the
    analytic gradient, step h = 1e-6 (lavaan::lav_model_hessian)."""
    k = par.npar
    H = np.zeros((k, k))
    for j in range(k):
        def at(delta):
            y = np.array(x, dtype=float)
            y[j] = y[j] + delta
            r = objective_and_gradient(par, data, y)
            return None if r is None else r[1]

        gl = at(-h)
        gl2 = at(-2 * h)
        gr = at(h)
        gr2 = at(2 * h)
        if gl is None or gl2 is None or gr is None or gr2 is None:
            return None
        H[:, j] = (0.5 * (gl2 - 8 * gl + 8 * gr - gr2)) / (12 * h)
    return 0.5 * (H + H.T)


# =============================================================================================
# optimizer


def minimize(par, data, start, max_iter=500, grad_tol=1e-9, step_tol=1e-12):
    """Fisher scoring with backtracking line search (Sigma kept PD), Newton steps on the
    finite-difference Hessian near the optimum, ridge-regularised fallbacks."""
    x = np.array(start, dtype=float)
    cur = objective_and_gradient(par, data, x)
    if cur is None:
        return {
            "x": x,
            "f": math.nan,
            "grad": np.zeros(par.npar),
            "iterations": 0,
            "converged": False,
            "message": "the starting values imply a non-positive-definite covariance matrix",
        }

    def max_abs(v):
        return float(np.max(np.abs(v))) if v.size else 0.0

    small = 0
    newton = False
    it = 1
    while it <= max_iter:
        H = None
        if newton:
            obs = observed_information(par, data, x)
            if obs is not None:
                H = obs * 2
                if cholesky_lower(H) is None:
                    H = None
        if H is None:
            H = expected_hessian(par, data, x)
        f0, g0 = cur
        directions = [lambda H=H, g0=g0: gj_solve(H, g0) if H is not None else None]
        if H is not None:
            tr = float(np.sum(np.abs(np.diag(H))))
            for kk in range(6):

                def ridge_dir(kk=kk, H=H, g0=g0, tr=tr):
                    ridge = (tr / max(H.shape[0], 1)) * math.pow(10, -8 + 2 * kk)
                    return gj_solve(H + ridge * np.eye(H.shape[0]), g0)

                directions.append(ridge_dir)
        directions.append(lambda g0=g0: np.array(g0, dtype=float))
        accepted = None
        step = g0
        alpha = 1.0
        for dir_fn in directions:
            s = dir_fn()
            if s is None or not np.all(np.isfinite(s)):
                continue
            slope = float(s @ g0)
            if not (slope > 0):
                continue
            alpha = 1.0
            for _ in range(40):
                y = x - alpha * s
                fy = objective_only(par, data, y)
                if (
                    fy is not None
                    and math.isfinite(fy)
                    and fy <= f0 - 1e-4 * alpha * slope + 1e-13 * abs(f0)
                ):
                    accepted = (y, fy)
                    break
                alpha /= 2
            if accepted is not None:
                step = s
                break
        if accepted is None:
            break
        step_size = max_abs(step) * alpha
        x = accepted[0]
        nxt = objective_and_gradient(par, data, x)
        if nxt is None:
            break
        df = f0 - nxt[0]
        cur = nxt
        g = max_abs(cur[1])
        if g < 1e-4:
            newton = True
        if step_size < step_tol or (g < grad_tol and abs(df) < 1e-15):
            small += 1
            if small >= 2 or g < grad_tol * 1e-3:
                break
        else:
            small = 0
        it += 1
    g = max_abs(cur[1])
    return {
        "x": x,
        "f": cur[0],
        "grad": cur[1],
        "iterations": min(it, max_iter),
        "converged": g < max(grad_tol * 100, 1e-7),
    }


# =============================================================================================
# starting values


def start_values(spec, S, mean):
    """Starting values from the (h1) sample moments (triad loadings, remainder residuals,
    cross-block factor covariances, paths 0, intercepts = means)."""
    m, p = spec.m, spec.p
    x = np.zeros(spec.npar)
    blocks = [[] for _ in range(m)]
    for e in spec.entries:
        if e.mat == "lambda":
            blocks[e.col].append(e.row)
    lam = np.zeros((p, m))
    phi = np.zeros(m)
    for j in range(m):
        idx = blocks[j]
        cs = []
        for i in idx:
            sii = S[i, i]
            others = [k for k in idx if k != i]
            if len(others) >= 2:
                s = 0.0
                cnt = 0
                for a in range(len(others)):
                    for b in range(a + 1, len(others)):
                        ka, kb = others[a], others[b]
                        sab = S[ka, kb]
                        if abs(sab) < 1e-8 * math.sqrt(S[ka, ka] * S[kb, kb]):
                            continue
                        s += abs((S[i, ka] * S[i, kb]) / sab)
                        cnt += 1
                c2 = s / cnt if cnt else 0.5 * sii
            elif len(others) == 1:
                c2 = abs(S[i, others[0]])
            else:
                c2 = sii
            c2 = min(max(c2, 0.05 * sii), 0.95 * sii if others else sii)
            cs.append(math.sqrt(c2))
        marker = idx[0]
        signed = [cs[t] if (t == 0 or S[i, marker] >= 0) else -cs[t] for t, i in enumerate(idx)]
        if spec.identification == "marker":
            phi[j] = signed[0] ** 2
            for t, i in enumerate(idx):
                lam[i, j] = signed[t] / signed[0]
        else:
            phi[j] = 1.0
            for t, i in enumerate(idx):
                lam[i, j] = signed[t]

    def latent_cov(a, b):
        s = 0.0
        cnt = 0
        for i in blocks[a]:
            for k in blocks[b]:
                if i == k:
                    continue
                la = lam[i, a]
                lb = lam[k, b]
                if la == 0 or lb == 0:
                    continue
                s += S[i, k] / (la * lb)
                cnt += 1
        if not cnt:
            return 0.0
        lim = 0.9 * math.sqrt(phi[a] * phi[b])
        return max(-lim, min(lim, s / cnt))

    for e in spec.entries:
        if e.free < 0:
            continue
        v = 0.0
        if e.mat == "lambda":
            v = lam[e.row, e.col]
        elif e.mat == "beta":
            v = 0.0
        elif e.mat == "psi":
            v = phi[e.row] if e.row == e.col else latent_cov(e.row, e.col)
        elif e.mat == "theta":
            if e.row == e.col:
                common = 0.0
                for j in range(m):
                    common += lam[e.row, j] ** 2 * phi[j]
                sii = S[e.row, e.row]
                v = max(sii - common, 0.1 * sii)
            else:
                v = 0.0
        elif e.mat == "nu":
            v = mean[e.row] if mean is not None else 0.0
        elif e.mat == "alpha":
            v = 0.0
        x[e.free] = v
    # keep the implied covariance positive definite
    for _ in range(30):
        st = model_state(spec, x)
        if st is not None and cholesky_lower(st.sigma) is not None:
            break
        for e in spec.entries:
            if e.free < 0:
                continue
            if e.row != e.col and e.mat in ("psi", "theta"):
                x[e.free] = x[e.free] * 0.5
            if e.row == e.col and e.mat == "theta":
                x[e.free] = x[e.free] + 0.1 * S[e.row, e.row]
    return x


# =============================================================================================
# inference

_SINGULAR_REL = 1e-10


def information_matrix(par, data, x, kind):
    if kind == "expected":
        H = expected_hessian(par, data, x)
        return None if H is None else H * 0.5
    return observed_information(par, data, x)


def compute_vcov(par, data, x, kind, robust):
    """{'vcov', 'information', 'first_order', 'info_inverse', 'problem'?, 'min_eigen'?}."""
    info = information_matrix(par, data, x, kind)
    if info is None or not np.all(np.isfinite(info)):
        return {"vcov": None, "information": None, "first_order": None, "info_inverse": None,
                "problem": "information-failed"}
    ev = sym_eigenvalues(info)
    max_ev = max(float(np.max(np.abs(ev))) if ev.size else 0.0, 0.0)
    min_ev = float(np.min(ev)) if ev.size else math.inf
    inv = gj_inverse(info)
    if inv is None or not (min_ev > _SINGULAR_REL * max_ev):
        return {"vcov": None, "information": info, "first_order": None, "info_inverse": None,
                "problem": "information-singular", "min_eigen": min_ev}
    n = data.n
    if not robust:
        return {"vcov": inv / n, "information": info, "first_order": None, "info_inverse": inv}
    B = first_order_information(par, data, x)
    if B is None:
        return {"vcov": None, "information": info, "first_order": None, "info_inverse": inv,
                "problem": "information-failed"}
    s = ((inv @ B) @ inv) / n
    return {"vcov": s, "information": info, "first_order": B, "info_inverse": inv}


def _trace_product(B, Ainv):
    """tr(B A^-1) given A^-1."""
    return float(np.sum(B * Ainv.T))


def h1_trace(sat, data, xh1):
    A1 = observed_information(sat, data, xh1)
    B1 = first_order_information(sat, data, xh1)
    if A1 is None or B1 is None:
        return None
    A1inv = gj_inverse(A1)
    if A1inv is None:
        return None
    return _trace_product(B1, A1inv)


def h0_trace(par, data, x, info_inverse=None, first_order=None):
    Iinv = info_inverse
    if Iinv is None:
        I = observed_information(par, data, x)
        Iinv = None if I is None else gj_inverse(I)
    B0 = first_order if first_order is not None else first_order_information(par, data, x)
    if Iinv is None or B0 is None:
        return None
    return _trace_product(B0, Iinv)


# =============================================================================================
# engine


def _err(code, message, details=None):
    e = {"code": code, "message": message}
    if details is not None:
        e["details"] = details
    return {"ok": False, "error": e}


def estimate_cb(model, analysis, data, on_progress=None, is_cancelled=None):
    """Estimate a CB-SEM model. Never raises: failures are {'ok': False, 'error'}."""
    try:
        with np.errstate(all="ignore"):
            return _estimate_cb(model, analysis, data, on_progress, is_cancelled)
    except Exception as e:  # noqa: BLE001 - mirrors estimateCb's catch-all
        return _err("ENGINE_INTERNAL", f"Unexpected error in the CB engine: {e}")


def _max_abs(v):
    return float(np.max(np.abs(v))) if np.size(v) else 0.0


def _estimate_cb(model, analysis, inp, on_progress, is_cancelled):
    started_at = iso_now()

    def cancelled():
        return is_cancelled is not None and is_cancelled() is True

    def progress(stage):
        if on_progress is not None:
            on_progress({"stage": stage})

    progress("validating")
    if analysis.get("method") != "cb-sem":
        return _err("REQUEST_INVALID", f"The CB engine cannot run method {analysis.get('method')}.",
                    {"reason": "method"})
    estimator = str(analysis.get("estimator")).upper()
    if estimator not in ("ML", "MLR"):
        return _err(
            "CAPABILITY_UNSUPPORTED",
            f"Estimator {analysis.get('estimator')} is not implemented in the browser CB engine (ML and MLR are).",
            {"reason": "estimator", "estimator": analysis.get("estimator")},
        )
    if (analysis.get("bootstrap") or {}).get("enabled"):
        return _err("CAPABILITY_UNSUPPORTED",
                    "Bootstrap inference for CB-SEM is not implemented in the browser engine.",
                    {"reason": "cb-bootstrap"})
    robust = estimator == "MLR"
    fiml = analysis.get("missingData") == "fiml"
    cbopt = analysis.get("cb") or {}
    identification = cbopt.get("identification") or "marker"
    requested_means = cbopt.get("meanStructure") or False
    mean_structure = bool(requested_means or fiml)
    warnings = []
    messages = []
    if fiml and not requested_means:
        warnings.append({
            "code": "CB_MEANSTRUCTURE_ADDED",
            "severity": "info",
            "message": 'FIML requires a mean structure; indicator intercepts were added (as lavaan does for missing = "ml").',
        })

    spec, serr = build_cb_spec(model, identification, mean_structure)
    if spec is None:
        return {"ok": False, "error": serr}
    moments = sample_moments(spec.p, mean_structure)
    df = moments - spec.npar
    if df < 0:
        return _err(
            "MODEL_NOT_IDENTIFIED",
            f"The model has more free parameters ({spec.npar}) than sample moments ({moments}); df = {df} < 0 (t-rule).",
            {"reason": "t-rule", "npar": spec.npar, "moments": moments, "df": df},
        )

    prep, perr = cb_prepare_data(model, spec.ov_ids, inp, "fiml" if fiml else "listwise")
    if prep is None:
        return {"ok": False, "error": perr}
    data, n_used, _ = prep
    N = data.n
    if not fiml and not data.complete:
        return _err("ENGINE_INTERNAL", "listwise data are incomplete")

    # h1 (saturated) moments
    if data.complete:
        h1_sigma, h1_mu = data.cov, data.mean
    else:
        em = em_saturated(data)
        if em is None:
            return _err("ENGINE_INADMISSIBLE",
                        "The EM estimate of the saturated covariance matrix became singular.",
                        {"reason": "em-singular"})
        if not em[3]:
            warnings.append({
                "code": "CB_EM_NOT_CONVERGED",
                "severity": "warning",
                "message": f"The EM algorithm for the saturated (h1) moments did not converge in {em[2]} iterations.",
            })
        h1_mu, h1_sigma = em[0], em[1]
    singular = _singular_covariance_error(h1_sigma, spec, model, data.complete)
    if singular is not None:
        return {"ok": False, "error": singular}
    if N <= spec.npar:
        warnings.append({"code": "CB_N_BELOW_PARAMETERS", "severity": "warning",
                         "message": f"Only {N} cases for {spec.npar} free parameters."})

    if cancelled():
        return _err("CANCELLED", "Cancelled.")
    progress("estimating")

    par = ModelParameterization(spec)
    start = start_values(spec, h1_sigma, h1_mu)
    opt = minimize(par, data, start)
    if not opt["converged"]:
        msg = f"; {opt['message']}" if opt.get("message") else ""
        return _err("ENGINE_NONCONVERGENCE",
                    f"The ML optimiser did not converge ({opt['iterations']} iterations{msg}).",
                    {"iterations": opt["iterations"]})
    x = opt["x"]
    st = model_state(spec, x)
    messages.append(
        f"ML by Fisher scoring with Newton refinement: {opt['iterations']} iterations, max |gradient| {js_to_exponential(_max_abs(opt['grad']), 2)}."
    )
    if cancelled():
        return _err("CANCELLED", "Cancelled.")

    # --- inference
    info_kind = "observed" if (robust or fiml) else "expected"
    vc = compute_vcov(par, data, x, info_kind, robust)
    se_problem = None
    if vc["vcov"] is None:
        se_problem = "failed"
        if vc.get("problem") == "information-singular":
            me = vc.get("min_eigen")
            msg = f"The {info_kind} information matrix is singular (smallest eigenvalue {js_to_exponential(me if me is not None else math.nan, 2)}): the model is empirically under-identified at this solution. Standard errors are not reported."
        else:
            msg = "The information matrix could not be computed; standard errors are not reported."
        warnings.append({"code": "CB_INFORMATION_SINGULAR", "severity": "warning", "message": msg})

    progress("summarizing")

    # --- admissibility (lavaan post.check)
    admissible, adm_warnings = _admissibility(spec, st)
    warnings.extend(adm_warnings)

    # --- fit
    fit = _fit_measures(spec, data, x, opt["f"], df, h1_sigma, h1_mu, robust, fiml,
                        vc["info_inverse"], vc["first_order"], warnings)

    # --- parameters
    std = _standardized(spec, st)
    source_method = f"sem-lab:cb-sem:{estimator}"
    parameters = []
    inference_method = "robust-analytic" if robust else "analytic"
    for k, e in enumerate(spec.entries):
        ids = {}
        if e.source_id:
            ids["sourceId"] = e.source_id
        if e.target_id:
            ids["targetId"] = e.target_id
        if e.relationship_id:
            ids["relationshipId"] = e.relationship_id
        est = float(x[e.free]) if e.free >= 0 else float(e.value)
        if e.free < 0:
            parameters.append({
                "key": e.key, "kind": e.kind, **ids, "scale": "unstandardized", "est": est,
                "se": None, "statistic": None, "pValue": None, "ciLower": None, "ciUpper": None,
                "missing": {f: "not-applicable" for f in ("se", "statistic", "pValue", "ciLower", "ciUpper")},
                "status": "fixed", "sourceMethod": source_method, "rawPath": e.where,
            })
        else:
            variance = float(vc["vcov"][e.free, e.free]) if vc["vcov"] is not None else math.nan
            se = math.sqrt(variance) if variance >= 0 else math.nan
            if not math.isfinite(se):
                reason = se_problem or "non-finite"
                parameters.append({
                    "key": e.key, "kind": e.kind, **ids, "scale": "unstandardized", "est": est,
                    "se": None, "statistic": None, "pValue": None, "ciLower": None, "ciUpper": None,
                    "missing": {f: reason for f in ("se", "statistic", "pValue", "ciLower", "ciUpper")},
                    "status": "estimated", "sourceMethod": source_method, "rawPath": e.where,
                })
            else:
                z = est / se if se > 0 else math.nan
                z_ok = math.isfinite(z)
                p = {
                    "key": e.key, "kind": e.kind, **ids, "scale": "unstandardized", "est": est,
                    "se": se, "statistic": z if z_ok else None, "statisticType": "z",
                    "pValue": p_two_sided_normal(z) if z_ok else None,
                    "ciLower": est - Z_975 * se, "ciUpper": est + Z_975 * se,
                }
                if not z_ok:
                    p["missing"] = {"statistic": "non-finite", "pValue": "non-finite"}
                p["status"] = "estimated"
                p["inference"] = {"method": inference_method, "level": 0.95, "sides": "two-sided"}
                p["sourceMethod"] = source_method
                p["rawPath"] = e.where
                parameters.append(p)
        sv = std[k]
        std_missing = {f: "not-supported" for f in ("se", "statistic", "pValue", "ciLower", "ciUpper")}
        if not math.isfinite(sv):
            std_missing["est"] = "non-finite"
        parameters.append({
            "key": e.key, "kind": e.kind, **ids, "scale": "standardized",
            "est": float(sv) if math.isfinite(sv) else None,
            "se": None, "statistic": None, "pValue": None, "ciLower": None, "ciUpper": None,
            "missing": std_missing,
            "status": "fixed" if e.mat == "alpha" else "estimated",
            "sourceMethod": f"{source_method}:std.all",
            "rawPath": f"std.all:{e.where}",
        })

    # --- construct metrics: R² of endogenous factors
    construct_metrics = []
    for j, cid in enumerate(spec.lv_ids):
        if not spec.endogenous[j]:
            construct_metrics.append({"constructId": cid, "metric": "r2", "value": None,
                                      "missing": "not-applicable",
                                      "definition": "R² of an exogenous factor is not defined."})
            continue
        r2 = 1 - st.psi[j, j] / st.phi[j, j]
        definition = "R² = 1 − ψ_jj / Var(η_j), model-implied."
        if math.isfinite(r2):
            construct_metrics.append({"constructId": cid, "metric": "r2", "value": float(r2),
                                      "definition": definition})
        else:
            construct_metrics.append({"constructId": cid, "metric": "r2", "value": None,
                                      "missing": "non-finite", "definition": definition})

    # --- latent correlation matrix
    m = spec.m
    latent_values = []
    for a in range(m):
        row = []
        for b in range(m):
            v = st.phi[a, b] / math.sqrt(st.phi[a, a] * st.phi[b, b]) if st.phi[a, a] * st.phi[b, b] >= 0 else math.nan
            row.append(float(v) if math.isfinite(v) else None)
        latent_values.append(row)
    latent_bad = any(v is None for r in latent_values for v in r)
    lc = {
        "id": "latent-correlations",
        "title": "Model-implied factor correlations",
        "rowIds": list(spec.lv_ids),
        "colIds": list(spec.lv_ids),
        "values": latent_values,
        "symmetric": True,
        "definition": "Φ = (I − B)⁻¹ Ψ (I − B)⁻ᵀ scaled to unit diagonal.",
    }
    if latent_bad:
        lc["missing"] = "non-finite"

    n_rows = inp.get("nRows")
    result = {
        "schemaVersion": 1,
        "sample": {
            "nRows": n_rows,
            "nUsed": n_used,
            "nDropped": n_rows - n_used,
            "missingData": "fiml" if fiml else "listwise",
        },
        "estimation": {
            "converged": True,
            "iterations": opt["iterations"],
            "admissible": admissible,
            "messages": messages,
        },
        "parameters": parameters,
        "constructMetrics": construct_metrics,
        "matrices": [lc],
        "fit": fit,
        "warnings": warnings,
        "provenance": provenance(
            f"estimateCb(estimator={estimator}, missing={'fiml' if fiml else 'listwise'}, identification={identification}, meanStructure={'true' if mean_structure else 'false'})",
            started_at,
            {"@sem-lab/engine/cb": CB_ENGINE_VERSION},
        ),
    }
    return {"ok": True, "result": result}


def _singular_covariance_error(S, spec, model, complete):
    """Non-PD sample covariance: report the near-linear dependency among indicators."""
    if cholesky_lower(S) is not None:
        ev = sym_eigenvalues(S)
        if ev[-1] > 1e-12 * ev[0]:
            return None
    values, vectors = sym_eigen(S)
    k = values.shape[0] - 1
    coef = [float(vectors[i, k]) for i in range(len(spec.ov_ids))]
    mx = max(abs(c) for c in coef)
    big = [(c, i) for i, c in enumerate(coef) if abs(c) > 1e-3 * mx]
    big.sort(key=lambda t: -abs(t[0]))
    names = {i["id"]: i.get("name") for i in model.get("indicators") or []}

    def name(i):
        iid = spec.ov_ids[i]
        nm = names.get(iid)
        return nm if nm is not None else iid

    lead_c, lead_i = big[0]
    parts = []
    for c, i in big[1:]:
        w = -c / lead_c
        parts.append(f"{'+' if w >= 0 else '−'} {js_to_fixed(abs(w), 3)}·{name(i)}")
    rhs = " ".join(parts)
    if rhs.startswith("+ "):
        rhs = rhs[2:]
    return {
        "code": "DATA_ZERO_VARIANCE",
        "message": f"The {'sample' if complete else 'EM-estimated'} covariance matrix of the indicators is singular (smallest eigenvalue {js_to_exponential(float(values[k]), 2)}): a linear combination of indicators has zero variance, {name(lead_i)} ≈ {rhs}. Remove one of the linearly dependent indicators.",
        "details": {
            "reason": "singular-covariance",
            "elementIds": [spec.ov_ids[i] for _, i in big],
            "dependency": {spec.ov_ids[i]: c / lead_c for c, i in big},
            "smallestEigenvalue": float(values[k]),
        },
    }


def _psd_ok(M):
    if M.shape[0] == 0:
        return True
    ev = sym_eigenvalues(M)
    tol = _CB_EPS**0.75 * max(1.0, abs(float(ev[0])))
    return not bool(np.any(ev < -tol))


def _admissibility(spec, st):
    warnings = []
    neg_ov = [iid for i, iid in enumerate(spec.ov_ids) if st.theta[i, i] < 0]
    neg_lv = [cid for j, cid in enumerate(spec.lv_ids) if st.psi[j, j] < 0]
    ok = True
    if neg_ov:
        ok = False
        warnings.append({
            "code": "CB_NEGATIVE_VARIANCE",
            "severity": "warning",
            "message": f"Negative residual variance estimate (Heywood case) for {', '.join(neg_ov)}: the solution is inadmissible; the standardized loading exceeds 1 in absolute value. Do not interpret the fit of this solution.",
            "elementIds": neg_ov,
        })
    if neg_lv:
        ok = False
        warnings.append({
            "code": "CB_NEGATIVE_VARIANCE",
            "severity": "warning",
            "message": f"Negative (residual) variance estimate for factor(s) {', '.join(neg_lv)}: the solution is inadmissible.",
            "elementIds": neg_lv,
        })
    phi_ok = True
    if not neg_lv and spec.m > 0 and not _psd_ok(st.phi):
        ok = phi_ok = False
        warnings.append({
            "code": "CB_LATENT_COV_NOT_PD",
            "severity": "warning",
            "message": "The covariance matrix of the latent variables is not positive definite (e.g. a factor correlation beyond ±1).",
            "elementIds": list(spec.lv_ids),
        })
    theta_ok = True
    if not neg_ov and not _psd_ok(st.theta):
        ok = theta_ok = False
        warnings.append({
            "code": "CB_THETA_NOT_PD",
            "severity": "warning",
            "message": "The residual covariance matrix of the indicators (Θ) is not positive definite.",
            "elementIds": list(spec.ov_ids),
        })
    if not neg_ov and not neg_lv and phi_ok and theta_ok and spec.m > 0 and not _psd_ok(st.psi):
        ok = False
        warnings.append({
            "code": "CB_PSI_NOT_PD",
            "severity": "warning",
            "message": "The covariance matrix of the structural residuals (Ψ) is not positive definite.",
            "elementIds": list(spec.lv_ids),
        })
    return ok, warnings


def _nsqrt(v):
    return math.sqrt(v) if v >= 0 else math.nan


def _ndiv(a, b):
    with np.errstate(all="ignore"):
        return float(np.divide(a, b))


def _standardized(spec, st):
    """lavaan std.all (cov.std = TRUE). Values may be NaN (reported as null)."""
    sd_ov = [_nsqrt(st.sigma[i, i]) for i in range(spec.p)]
    sd_lv = [_nsqrt(st.phi[j, j]) for j in range(spec.m)]
    out = []
    for e in spec.entries:
        r, c = e.row, e.col
        if e.mat == "lambda":
            v = _ndiv(st.lam[r, c] * sd_lv[c], sd_ov[r])
        elif e.mat == "beta":
            v = _ndiv(st.beta[r, c] * sd_lv[c], sd_lv[r])
        elif e.mat == "psi":
            v = (_ndiv(st.psi[r, r], st.phi[r, r]) if r == c
                 else _ndiv(st.psi[r, c], _nsqrt(st.psi[r, r] * st.psi[c, c])))
        elif e.mat == "theta":
            v = (_ndiv(st.theta[r, r], st.sigma[r, r]) if r == c
                 else _ndiv(st.theta[r, c], _nsqrt(st.theta[r, r] * st.theta[c, c])))
        elif e.mat == "nu":
            v = _ndiv(st.nu[r], sd_ov[r])
        elif e.mat == "alpha":
            v = _ndiv(st.alpha[r], sd_lv[r])
        else:
            v = math.nan
        out.append(v)
    return out


def _near_zero(t):
    """|t| below lavaan's all.equal tolerance counts as zero (lav_fit_cfi)."""
    return abs(t) < 1.5e-8


def cfi_value(x2, df, x2n, dfn, c=1.0, cn=1.0):
    use_c = df > 0 and c != 1 and cn != 1
    t1 = max(x2 - (c if use_c else 1) * df, 0)
    t2 = max(x2 - (c if use_c else 1) * df, x2n - (cn if use_c else 1) * dfn, 0)
    if _near_zero(t1) and _near_zero(t2):
        return 1.0
    return 1 - _ndiv(t1, t2)


def tli_value(x2, df, x2n, dfn, c=1.0, cn=1.0):
    use_c = df > 0 and c != 1 and cn != 1
    t1 = (x2 - (c if use_c else 1) * df) * dfn
    t2 = (x2n - (cn if use_c else 1) * dfn) * df
    if df > 0 and abs(t2) > 0:
        return 1 - t1 / t2
    return 1.0


def _js_max(*vals):
    """Math.max: NaN if any argument is NaN."""
    for v in vals:
        if v != v:
            return math.nan
    return max(vals)


def _fit_measures(spec, data, x, f, df, h1_sigma, h1_mu, robust, fiml, info_inverse,
                  first_order, warnings):
    N = data.n
    p = spec.p
    out = []

    def add(name, value, definition, variant=None, ci=None, missing=None, reason="non-finite"):
        ok = value is not None and math.isfinite(value)
        e = {"name": name, "value": (value if isinstance(value, int) else float(value)) if ok else None}
        if not ok:
            e["missing"] = missing or reason
        if variant:
            e["variant"] = variant
        if ci is not None:
            e["ciLower"] = ci[0]
            e["ciUpper"] = ci[1]
        e["definition"] = definition
        out.append(e)

    def na(name, definition, reason, variant=None):
        add(name, None, definition, variant=variant, missing=reason)

    mean_structure = spec.mean_structure
    mu_h1 = h1_mu if mean_structure else None
    f_h1 = evaluate(data, h1_sigma, mu_h1)[0]
    constant = sum(k.n * k.obs.shape[0] for k in data.patterns) * _LOG_2PI
    logl = -0.5 * (N * f + constant)
    logl_h1 = -0.5 * (N * f_h1 + constant)
    x2 = _js_max(N * (f - f_h1), 0)

    # baseline (independence) model: closed form (per-variable available-case ML moments)
    var_b = np.zeros(p)
    mu_b = np.zeros(p)
    cnt = np.zeros(p)
    for pat in data.patterns:
        mu_b[pat.obs] += pat.n * pat.mean
        cnt[pat.obs] += pat.n
    mu_b = mu_b / cnt
    for pat in data.patterns:
        var_b[pat.obs] += pat.n * (np.diag(pat.cov) + (pat.mean - mu_b[pat.obs]) ** 2)
    var_b = var_b / cnt
    sigma_b = np.diag(var_b)
    f_b = evaluate(data, sigma_b, mu_b if mean_structure else None)[0]
    x2_b = N * (f_b - f_h1)
    df_b = (p * (p - 1)) // 2

    just_id = df == 0
    add("chisq", x2, "Likelihood-ratio χ² against the saturated model, N·(F − F_h1) (lavaan: N·F_ML).",
        variant="standard")
    add("df", df, "Degrees of freedom: sample moments − free parameters.")
    if just_id:
        na("pvalue", "p value of the χ² test (not defined for df = 0).", "not-applicable", "standard")
    else:
        add("pvalue", pchisq_upper(x2, df), "p value of the χ² test, P(χ²_df > X²).", variant="standard")
    add("baseline.chisq", x2_b, "χ² of the independence (baseline) model.", variant="standard")
    add("baseline.df", df_b, "Degrees of freedom of the independence model, p(p−1)/2.")
    add("baseline.pvalue", pchisq_upper(x2_b, df_b), "p value of the baseline χ² test.", variant="standard")

    cfi_def = "CFI = 1 − max(X² − df, 0) / max(X² − df, X²_B − df_B, 0)."
    tli_def = "TLI = (X²_B/df_B − X²/df) / (X²_B/df_B − 1)."
    rmsea_def = "RMSEA = sqrt(max(X²/(N·df) − 1/N, 0)); 90% CI by noncentral χ² inversion (lavaan uniroot convention)."
    srmr_def = (
        "SRMR (Bentler): root mean square of covariance residuals scaled by the observed SDs, including the standardized mean residuals (lavaan 'srmr')."
        if mean_structure
        else "SRMR (Bentler): root mean square of the covariance residuals scaled by the observed SDs, diagonal included (lavaan 'srmr')."
    )
    if just_id:
        na("cfi", f"{cfi_def} Not informative for a just-identified model (df = 0).", "not-applicable", "standard")
        na("tli", f"{tli_def} Not informative for df = 0.", "not-applicable", "standard")
        na("rmsea", f"{rmsea_def} Not informative for df = 0.", "not-applicable", "standard")
        na("rmsea.pvalue", "Close-fit test P(RMSEA ≤ .05); not defined for df = 0.", "not-applicable", "standard")
        na("srmr", f"{srmr_def} Zero by construction for df = 0.", "not-applicable", "standard")
        warnings.append({
            "code": "CB_JUST_IDENTIFIED",
            "severity": "info",
            "message": "The model is just-identified (df = 0): it reproduces the sample moments exactly, so χ² = 0 and the global fit indices carry no information about fit.",
        })
    else:
        add("cfi", cfi_value(x2, df, x2_b, df_b), cfi_def, variant="standard")
        add("tli", tli_value(x2, df, x2_b, df_b), tli_def, variant="standard")
        ci = rmsea_ci(x2, df, N)
        add("rmsea", rmsea_point(x2, df, N), rmsea_def, variant="standard", ci=ci)
        add("rmsea.pvalue", rmsea_close_pvalue(x2, df, N, 0.05),
            "Close-fit test P(RMSEA ≤ .05): 1 − F(X²; df, λ = N·df·0.05²).", variant="standard")
        add("srmr", _srmr_bentler(h1_sigma, mu_h1, model_state(spec, x)), srmr_def, variant="standard")
    npar = spec.npar
    add("logl", logl, "Log-likelihood of the fitted model.")
    add("unrestricted.logl", logl_h1, "Log-likelihood of the saturated (h1) model.")
    add("aic", -2 * logl + 2 * npar, "AIC = −2 logL + 2·npar.")
    add("bic", -2 * logl + npar * math.log(N), "BIC = −2 logL + npar·ln N.")
    add("bic2", -2 * logl + npar * math.log((N + 2) / 24), "Sample-size adjusted BIC = −2 logL + npar·ln((N + 2)/24).")
    add("npar", npar, "Number of free parameters.")
    add("ntotal", N, "Number of cases used.")

    if not robust:
        return out

    # ---- MLR: Yuan–Bentler (Mplus variant) scaled test
    scaled_names = ["chisq", "chisq.scaling.factor", "pvalue", "baseline.chisq",
                    "baseline.chisq.scaling.factor", "cfi", "tli", "rmsea"]
    if just_id:
        for nm in scaled_names:
            na(nm, "Yuan–Bentler scaled statistic: not defined for df = 0.", "not-applicable", "scaled")
        for nm in ("cfi", "tli", "rmsea"):
            na(nm, "Robust fit index: not defined for df = 0.", "not-applicable", "robust")
        return out
    sat = SaturatedParameterization(p, mean_structure)
    tr_h1 = h1_trace(sat, data, saturated_theta(h1_sigma, mu_h1))
    par = ModelParameterization(spec)
    tr_h0 = h0_trace(par, data, x, info_inverse, first_order)
    indep = IndependenceParameterization(p, mean_structure)
    x_b = np.concatenate([var_b, mu_b]) if mean_structure else var_b.copy()
    tr_b0 = h0_trace(indep, data, x_b)
    if tr_h1 is None or tr_h0 is None or tr_b0 is None:
        for nm in scaled_names:
            na(nm, "Yuan–Bentler scaled statistic (computation failed).", "failed", "scaled")
        return out
    trace = tr_h1 - tr_h0
    c = _ndiv(trace, df)
    c_b = _ndiv(tr_h1 - tr_b0, df_b)
    x2s = _ndiv(x2, c)
    x2_bs = _ndiv(x2_b, c_b)
    yb_def = 'Yuan–Bentler scaled χ² (Mplus variant, lavaan "yuan.bentler.mplus"): X²/c, c = [tr(B₁A₁⁻¹) − tr(B₀I⁻¹)]/df.'
    add("chisq", x2s, yb_def, variant="scaled")
    add("chisq.scaling.factor", c, "Scaling factor c of the Yuan–Bentler test.", variant="scaled")
    add("pvalue", 1 - pnchisq(x2s, df, 0), "p value of the scaled χ² test.", variant="scaled")
    add("baseline.chisq", x2_bs, "Yuan–Bentler scaled χ² of the independence model.", variant="scaled")
    add("baseline.chisq.scaling.factor", c_b, "Scaling factor of the baseline Yuan–Bentler test.", variant="scaled")
    add("cfi", cfi_value(x2s, df, x2_bs, df_b),
        "CFI computed from the scaled χ² of the model and the baseline (lavaan cfi.scaled).", variant="scaled")
    add("tli", tli_value(x2s, df, x2_bs, df_b),
        "TLI computed from the scaled χ² statistics (lavaan tli.scaled).", variant="scaled")
    ci = rmsea_ci(x2, trace, N)
    add("rmsea", rmsea_point(x2, trace, N),
        "RMSEA with df replaced by tr(UΓ) (lavaan rmsea.scaled), 90% CI likewise.", variant="scaled", ci=ci)
    if fiml:
        why = "Robust (Brosseau-Liard–Savalei / Savalei FIML-corrected) index: not implemented for FIML in the browser engine."
        na("cfi", why, "not-supported", "robust")
        na("tli", why, "not-supported", "robust")
        na("rmsea", why, "not-supported", "robust")
    else:
        add("cfi", cfi_value(x2, df, x2_b, df_b, c, c_b),
            "Robust CFI (Brosseau-Liard & Savalei 2014; lavaan cfi.robust).", variant="robust")
        add("tli", tli_value(x2, df, x2_b, df_b, c, c_b), "Robust TLI (lavaan tli.robust).", variant="robust")
        ci = rmsea_ci(x2s, df, N, c)
        add("rmsea", rmsea_point(x2, df, N, c),
            "Robust RMSEA (Brosseau-Liard, Savalei & Li 2012; lavaan rmsea.robust), 90% CI.",
            variant="robust", ci=ci)
    return out


def _srmr_bentler(S, m, st):
    """Bentler SRMR (lavaan "srmr"; with a mean structure the standardized mean residuals too)."""
    p = S.shape[0]
    d = np.sqrt(np.diag(S))
    R = (S - st.sigma) / np.outer(d, d)
    il = np.tril_indices(p)
    ss = float(np.sum(R[il] ** 2))
    cnt = len(il[0])
    if m is not None and st.mu is not None:
        r = (m - st.mu) / d
        ss += float(np.sum(r * r))
        cnt += p
    return math.sqrt(ss / cnt)


# =============================================================================================
# semlab/api.py
# =============================================================================================
# Public API: one entry point with the TypeScript engine's contract, plus data preparation as
# the web app does it and plain result tables.
#
#     estimate(model, analysis, data) -> {'ok': True, 'result': NormalizedSemResult}
#                                      | {'ok': False, 'error': {'code', 'message', 'details'?}}
#
# ``model`` is the SemModel JSON, ``analysis`` the AnalysisConfiguration JSON (``engine`` must be
# ``"browser"``, the engine the app runs), ``data`` = {columns, rows, nRows, nDropped}: the analysis
# matrix as the web app builds it (``build_analysis_data``). The result is the NormalizedSemResult
# JSON the app shows (same parameter keys, kinds, ids, missing reasons and warnings).

def estimate(model, analysis, data, on_progress=None, is_cancelled=None):
    """Estimate with the method in analysis['method']: PLS-SEM / PLSc -> PLS engine, CB-SEM ->
    CB engine. Never raises for bad input."""
    if analysis.get("method") == "cb-sem":
        return estimate_cb(model, analysis, data, on_progress=on_progress, is_cancelled=is_cancelled)
    return estimate_pls(model, analysis, data, on_progress=on_progress, is_cancelled=is_cancelled)


# ---- data ------------------------------------------------------------------------------------


class AnalysisDataError(ValueError):
    """The data cannot be prepared for the analysis (codes of apps/web buildAnalysisData)."""

    def __init__(self, code, message, columns=None, indicator_ids=None):
        super().__init__(f"{code}: {message}")
        self.code = code
        self.message = message
        self.columns = columns or []
        self.indicator_ids = indicator_ids or []


def _parse_cell(t, missing_tokens):
    t = t.strip()
    if len(t) >= 2 and t[0] == '"' and t[-1] == '"':
        t = t[1:-1]
    if t in missing_tokens:
        return None
    try:
        v = float(t)
    except ValueError:
        return t
    return v if math.isfinite(v) else None


def read_csv(path, missing_tokens=("", "NA"), delimiter=","):
    """Minimal CSV reader (UTF-8, optional BOM, header row, no embedded delimiters): returns
    (columns, rows). Numeric cells become floats, missing tokens None, anything else stays a
    string (and is refused as non-numeric by build_analysis_data if it is an analysis column)."""
    with open(path, encoding="utf-8-sig", newline="") as fh:
        lines = [ln for ln in fh.read().splitlines() if ln]
    header = [h.strip().strip('"') for h in lines[0].split(delimiter)]
    rows = [[_parse_cell(c, missing_tokens) for c in ln.split(delimiter)] for ln in lines[1:]]
    return header, rows


def _is_num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)


def build_analysis_data(model, columns, rows, missing_data="listwise"):
    """The numeric matrix one analysis runs on, exactly as the web app builds it
    (apps/web/src/domain/dataset/analysis-data.ts): one column per bound data column of every
    indicator that belongs to a construct, in model order; reverse scoring min + max - x with
    the ORIGINAL scale endpoints; 'listwise' drops every row with a missing value, 'fiml' keeps
    rows with at least one observed value (missing cells stay None)."""
    by_id = {i["id"]: i for i in model.get("indicators") or []}
    seen = set()
    indicators = []
    for c in model.get("constructs") or []:
        for iid in c.get("indicatorIds") or []:
            ind = by_id.get(iid)
            if ind is not None and iid not in seen:
                seen.add(iid)
                indicators.append(ind)
    unmapped = [i for i in indicators if i.get("column") is None]
    if unmapped:
        raise AnalysisDataError(
            "DATA_INDICATOR_UNMAPPED",
            f"Indicator(s) {', '.join(repr(i.get('name', i['id'])) for i in unmapped)} not mapped to a data column.",
            indicator_ids=[i["id"] for i in unmapped],
        )
    cols = list(columns)
    missing = []
    for i in indicators:
        if i["column"] not in cols and i["column"] not in missing:
            missing.append(i["column"])
    if missing:
        raise AnalysisDataError(
            "DATA_COLUMN_MISSING",
            f"The data have no column(s) {', '.join(repr(c) for c in missing)}.",
            columns=missing,
            indicator_ids=[i["id"] for i in indicators if i["column"] in missing],
        )
    no_scale = [i for i in indicators if i.get("reverseScored") is True and i.get("scale") is None]
    if no_scale:
        raise AnalysisDataError(
            "DATA_SCALE_MISSING",
            "Reverse scoring needs the original scale endpoints (minimum and maximum).",
            columns=list(dict.fromkeys(i["column"] for i in no_scale)),
            indicator_ids=[i["id"] for i in no_scale],
        )
    bindings = []
    conflicts = []
    for ind in indicators:
        col = ind["column"]
        rev = (
            (ind["scale"]["min"], ind["scale"]["max"])
            if ind.get("reverseScored") is True and ind.get("scale")
            else None
        )
        ex = next((b for b in bindings if b["column"] == col), None)
        if ex is None:
            bindings.append({"column": col, "reverse": rev})
        elif ex["reverse"] != rev and col not in conflicts:
            conflicts.append(col)
    if conflicts:
        raise AnalysisDataError(
            "DATA_REVERSE_CONFLICT",
            "Indicators bound to the same column disagree on reverse scoring or scale.",
            columns=conflicts,
        )
    idx = [cols.index(b["column"]) for b in bindings]
    non_numeric = []
    for b, j in zip(bindings, idx):
        if any(r[j] is not None and not _is_num(r[j]) for r in rows):
            non_numeric.append(b["column"])
    if non_numeric:
        raise AnalysisDataError(
            "DATA_NOT_NUMERIC", "Analysis column(s) are not numeric.", columns=non_numeric
        )
    out_of_scale = []
    for b, j in zip(bindings, idx):
        if b["reverse"] is None:
            continue
        lo, hi = b["reverse"]
        if any(r[j] is not None and not (r[j] != r[j]) and (r[j] < lo or r[j] > hi) for r in rows):
            out_of_scale.append(b["column"])
    if out_of_scale:
        raise AnalysisDataError(
            "DATA_OUT_OF_SCALE",
            "Reverse-scored column(s) have values outside the stated scale endpoints.",
            columns=out_of_scale,
        )
    full = []
    complete = []
    for r in rows:
        row = []
        observed = 0
        for b, j in zip(bindings, idx):
            v = r[j] if j < len(r) else None
            if v is None or v != v:
                row.append(None)
                continue
            observed += 1
            rev = b["reverse"]
            row.append(rev[0] + rev[1] - v if rev else v)
        if observed == len(bindings):
            complete.append(row)
        if observed > 0:
            full.append(row)
    n_rows = len(rows)
    use = complete if missing_data == "listwise" else full
    if len(use) < 3:
        raise AnalysisDataError(
            "DATA_INSUFFICIENT_ROWS",
            f"Only {len(use)} usable row(s) remain; at least 3 are needed.",
            columns=[b["column"] for b in bindings],
        )
    return {
        "columns": [b["column"] for b in bindings],
        "rows": use,
        "nRows": n_rows,
        "nDropped": n_rows - len(use),
    }


# ---- tables ----------------------------------------------------------------------------------


def _labels(model):
    lab = {}
    for c in model.get("constructs") or []:
        lab[c["id"]] = c.get("name") or c["id"]
    for i in model.get("indicators") or []:
        lab[i["id"]] = i.get("name") or i["id"]
    return lab


_PARAM_COLS = ("est", "se", "statistic", "pValue", "ciLower", "ciUpper")


def results_tables(result, model):
    """Plain tables (lists of row dicts; no pandas needed) from a NormalizedSemResult:
    parameters (PLS: standardized; CB: unstandardized with the standardized estimate in `std`),
    paths, loadings, weights, effects, constructMetrics (one row per construct), fit, one table
    per matrix (``matrix:<id>``), ipmaConstructs / ipmaIndicators, bootstrap, warnings."""
    lab = _labels(model)
    params = result.get("parameters") or []
    has_unstd = any(p.get("scale") == "unstandardized" for p in params)
    std_by_key = {p["key"]: p.get("est") for p in params if p.get("scale") == "standardized"}
    rows = []
    for p in params:
        if has_unstd and p.get("scale") != "unstandardized":
            continue
        row = {
            "key": p["key"],
            "kind": p["kind"],
            "from": lab.get(p.get("sourceId"), p.get("sourceId")),
            "to": lab.get(p.get("targetId"), p.get("targetId")),
            "via": " -> ".join(lab.get(v, v) for v in p.get("viaIds") or []) or None,
        }
        for c in _PARAM_COLS:
            row[c] = p.get(c)
        if has_unstd:
            row["std"] = std_by_key.get(p["key"])
        row["status"] = p.get("status")
        rows.append(row)
    t = {"parameters": rows}
    t["paths"] = [r for r in rows if r["kind"] == "path"]
    t["loadings"] = [r for r in rows if r["kind"] == "loading"]
    t["weights"] = [r for r in rows if r["kind"] == "weight"]
    t["effects"] = [
        r for r in rows if r["kind"] in ("total-effect", "indirect-effect", "specific-indirect")
    ]
    t["variances"] = [
        r for r in rows if r["kind"] in ("variance", "residual-variance", "covariance", "intercept")
    ]
    cm = {}
    for m in result.get("constructMetrics") or []:
        r = cm.setdefault(m["constructId"], {"construct": lab.get(m["constructId"], m["constructId"])})
        r[m["metric"]] = m.get("value")
    t["constructMetrics"] = list(cm.values())
    t["fit"] = [
        {
            "name": f["name"],
            "variant": f.get("variant", "standard"),
            "value": f.get("value"),
            "ciLower": f.get("ciLower"),
            "ciUpper": f.get("ciUpper"),
            "missing": f.get("missing"),
        }
        for f in result.get("fit") or []
    ]
    for m in result.get("matrices") or []:
        mrows = []
        for rid, vals in zip(m.get("rowIds") or [], m.get("values") or []):
            r = {"": lab.get(rid, rid)}
            for cid, v in zip(m.get("colIds") or [], vals):
                r[lab.get(cid, cid)] = v
            mrows.append(r)
        t[f"matrix:{m['id']}"] = mrows
    ip = result.get("ipma")
    if ip:
        t["ipmaConstructs"] = [
            dict(c, construct=lab.get(c["constructId"], c["constructId"])) for c in ip["constructs"]
        ]
        t["ipmaIndicators"] = [
            dict(c, indicator=lab.get(c["indicatorId"], c["indicatorId"]))
            for c in ip.get("indicators") or []
        ]
    b = result.get("bootstrap")
    if b:
        t["bootstrap"] = [{k: b[k] for k in b if k != "draws"}]
    t["warnings"] = [
        {"code": w["code"], "severity": w["severity"], "message": w["message"]}
        for w in result.get("warnings") or []
    ]
    return t


def pandas_available():
    try:
        import pandas  # noqa: F401
    except ImportError:
        return False
    return True


def to_dataframe(table):
    """One table (list of row dicts) as a pandas DataFrame. Requires pandas (optional)."""
    try:
        import pandas as pd
    except ImportError as e:  # pragma: no cover - depends on the environment
        raise ImportError("to_dataframe needs pandas (pip install pandas); results_tables() does not.") from e
    return pd.DataFrame(table)


def to_dataframes(result, model):
    """Every table of results_tables() as a pandas DataFrame. Requires pandas (optional)."""
    return {name: to_dataframe(rows) for name, rows in results_tables(result, model).items()}


__all__ = ['AnalysisDataError', 'BootstrapProblem', 'CB_ENGINE_VERSION', 'DrawResult', 'ENGINE_ERROR_CODES', 'ENGINE_VERSION', 'EngineError', 'MIRRORS_ENGINE_VERSION', 'RMersenneTwister', '__version__', 'bootstrap_indices', 'build_analysis_data', 'cb_parameter_count', 'core_model', 'correlation', 'estimate', 'estimate_cb', 'estimate_pls', 'extract_theta', 'fit_draw', 'fit_pls', 'gamma_p', 'gamma_q', 'infer_from_draws', 'p_two_sided_normal', 'pandas_available', 'pchisq', 'pchisq_upper', 'pls_prepare_data', 'pnchisq', 'pnorm_upper', 'pt', 'pt_lower_tail_abs', 'quantile7', 'r_set_seed', 'read_csv', 'results_tables', 'rmsea_ci', 'runif', 'sample_sd', 'summarise_bootstrap', 'theta_entries', 'to_dataframe', 'to_dataframes', 'validate_request', 'verify', 'zeroin']

In [ ]:
# Verify the engine file against the SHA-256 of packages/python-engine/dist/semlab_bundle.py
# (the version this notebook was built and tested with), then import it.
import hashlib
import importlib

ENGINE_SHA256 = "0d13368cf8cec0f3e50541b55938a53845b76bb32c67f4ac74cf40a64db2512c"
with open("semlab_bundle.py", encoding="utf-8") as f:
    engine_text = f.read()
if engine_text.startswith("#@title"):  # the Colab form title line of the cell above
    engine_text = engine_text.split("\n", 1)[1]
engine_text = engine_text.rstrip("\n") + "\n"
engine_sha = hashlib.sha256(engine_text.encode("utf-8")).hexdigest()
assert engine_sha == ENGINE_SHA256, (
    f"semlab_bundle.py has sha256 {engine_sha}, expected {ENGINE_SHA256}: the engine cell was edited.")
with open("semlab_bundle.py", "w", encoding="utf-8", newline="\n") as f:
    f.write(engine_text)  # now byte-identical to the committed bundle

sys.modules.pop("semlab_bundle", None)
importlib.invalidate_caches()
import semlab_bundle as semlab

print(f"SEM Lab engine {semlab.__version__} (mirrors @sem-lab/engine {semlab.MIRRORS_ENGINE_VERSION}); "
      f"sha256 {engine_sha[:16]}... verified")

In [ ]:
#@title Helper functions: model figure, charts, tables, export (click to expand)
# SEM Lab notebook helpers: model figure, charts, tables and export.
# Source: notebooks/src/sem_helpers.py (embedded into every notebook by notebooks/build_notebooks.py).
# Pure matplotlib / pandas; nothing here estimates anything (estimation is semlab_bundle.py).
import html as _html
import json as _json
import math as _math
import os as _os
import zipfile as _zipfile

import matplotlib.pyplot as plt
import numpy as _np
import pandas as pd
from matplotlib.patches import Ellipse, FancyArrowPatch, FancyBboxPatch

# Palette (validated reference palette: blue as the single data hue, grey for "not significant").
INK = "#0b0b0b"
INK_2 = "#52514e"
INK_3 = "#8a8985"
RULE = "#d9d8d4"
BLUE = "#2a78d6"
BLUE_DARK = "#184f95"
BLUE_FILL = "#e3eefb"
GREY_FILL = "#f4f3f0"
ORANGE = "#eb6834"
BLUE_RAMP = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
             "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "axes.edgecolor": RULE, "axes.labelcolor": INK_2, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlecolor": INK, "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "legend.frameon": False, "figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight",
})

ARROW = "→"


# ---------------------------------------------------------------------------------------------
# Small utilities
# ---------------------------------------------------------------------------------------------

def names_of(model):
    """id -> display name for constructs and indicators."""
    out = {c["id"]: c["name"] for c in model["constructs"]}
    out.update({i["id"]: i["name"] for i in model["indicators"]})
    return out


def fmt_p(p):
    if p is None or (isinstance(p, float) and _math.isnan(p)):
        return ""
    return "< .001" if p < 0.001 else f"{p:.3f}".replace("0.", ".", 1)


def significant(row):
    """A two-sided percentile CI that excludes 0 (the decision rule used in these notebooks)."""
    lo, hi = row.get("ciLower"), row.get("ciUpper")
    if lo is None or hi is None or pd.isna(lo) or pd.isna(hi):
        return None
    return bool(lo > 0 or hi < 0)


def ensure_dir(path):
    _os.makedirs(path, exist_ok=True)
    return path


# ---------------------------------------------------------------------------------------------
# Data dictionary and descriptives
# ---------------------------------------------------------------------------------------------

def data_dictionary(model, profiles=None):
    """One row per indicator: column, construct, measurement mode and declared scale."""
    rows = []
    owner = {}
    for c in model["constructs"]:
        for iid in c["indicatorIds"]:
            owner[iid] = c
    for ind in model["indicators"]:
        c = owner[ind["id"]]
        sc = ind.get("scale") or {}
        row = {"column": ind["column"], "construct": c["name"], "construct label": c["label"],
               "measurement": c["measurement"], "level": ind.get("level"),
               "scale min": sc.get("min"), "scale max": sc.get("max")}
        if profiles is not None:
            row["generator profile"] = profiles[ind["column"]]["profile"]
        rows.append(row)
    return pd.DataFrame(rows)


def descriptives(df, model):
    cols = [i["column"] for i in model["indicators"]]
    d = df[cols].agg(["count", "mean", "std", "min", "median", "max", "skew"]).T
    d["count"] = d["count"].astype(int)
    d.index.name = "column"
    return d


def likert_distribution_plot(df, model, title="Response distribution per item"):
    """Horizontal 100% stacked bars of response categories (one light-to-dark blue ramp)."""
    inds = model["indicators"]
    lo = min(i["scale"]["min"] for i in inds)
    hi = max(i["scale"]["max"] for i in inds)
    cats = list(range(lo, hi + 1))
    idx = _np.linspace(0, len(BLUE_RAMP) - 1, len(cats)).round().astype(int)
    colors = [BLUE_RAMP[k] for k in idx]
    order = []
    group_breaks = []
    for c in model["constructs"]:
        cols = [i["column"] for i in inds if i["id"] in c["indicatorIds"]]
        order.extend(cols)
        group_breaks.append((c["name"], len(order)))
    n_items = len(order)
    fig, ax = plt.subplots(figsize=(8.6, 0.9 + 0.27 * n_items))
    y = _np.arange(n_items)[::-1]
    left = _np.zeros(n_items)
    for k, cat in enumerate(cats):
        share = _np.array([(df[col] == cat).mean() * 100 for col in order])
        ax.barh(y, share, left=left, color=colors[k], edgecolor="white", linewidth=1.0, height=0.78,
                label=str(cat))
        left += share
    ax.set_yticks(y, order, fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_xlabel("% of respondents")
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)
    prev = 0
    for name, end in group_breaks:
        mid = (prev + end - 1) / 2
        ax.text(101.5, y[0] - mid, name, va="center", ha="left", fontsize=9, color=INK, fontweight="bold")
        if prev > 0:
            ax.axhline(y[0] - prev + 0.5, color=INK_3, linewidth=0.6)
        prev = end
    ax.legend(title="Category", ncol=len(cats), loc="lower center", bbox_to_anchor=(0.5, 1.0),
              fontsize=8, title_fontsize=8, handlelength=1.2, columnspacing=0.8)
    ax.set_title(title, pad=34, loc="left")
    fig.tight_layout()
    return fig


# ---------------------------------------------------------------------------------------------
# Model figure (generated from the canonical SemModel)
# ---------------------------------------------------------------------------------------------

_CW, _CH = 1.55, 0.78          # construct ellipse width / height
_IW, _IH = 0.74, 0.32          # indicator box width / height
_DX = 3.7                      # horizontal distance between layers
_ISTEP_V, _ISTEP_H = 0.44, 0.84  # indicator spacing in a vertical / horizontal fan
_FAN = 1.85                    # distance from construct centre to a side fan
_FAN_V = 1.08                  # distance from construct centre to an above/below fan


def _structural(model):
    return [(r["sourceId"], r["targetId"]) for r in model["relationships"] if r["kind"] == "structural"]


def _layers(model):
    ids = [c["id"] for c in model["constructs"]]
    edges = _structural(model)
    layer = {i: 0 for i in ids}
    for _ in range(len(ids)):
        changed = False
        for s, t in edges:
            if layer[t] < layer[s] + 1:
                layer[t] = layer[s] + 1
                changed = True
        if not changed:
            break
    return layer


def _seg_hits_rect(p, q, cx, cy, w, h, steps=60):
    for k in range(steps + 1):
        t = k / steps
        x = p[0] + (q[0] - p[0]) * t
        y = p[1] + (q[1] - p[1]) * t
        if abs(x - cx) <= w / 2 and abs(y - cy) <= h / 2:
            return True
    return False


def _fan_positions(cx, cy, k, side):
    if side in ("left", "right"):
        x = cx - _FAN if side == "left" else cx + _FAN
        return [(x, cy + (k - 1) / 2 * _ISTEP_V - j * _ISTEP_V) for j in range(k)]
    y = cy + _FAN_V if side == "above" else cy - _FAN_V
    return [(cx - (k - 1) / 2 * _ISTEP_H + j * _ISTEP_H, y) for j in range(k)]


def model_layout(model):
    """Deterministic layered left-to-right layout: construct positions and indicator fan sides."""
    cons = model["constructs"]
    ids = [c["id"] for c in cons]
    k_of = {c["id"]: len(c["indicatorIds"]) for c in cons}
    layer = _layers(model)
    edges = _structural(model)
    preds = {i: [s for s, t in edges if t == i] for i in ids}
    succs = {i: [t for s, t in edges if s == i] for i in ids}
    n_layers = max(layer.values()) + 1
    pos = {}
    # layer 0: stacked clusters, top to bottom in model order
    first = [i for i in ids if layer[i] == 0]
    heights = [max(k_of[i] * _ISTEP_V, _CH) + 0.55 for i in first]
    total = sum(heights)
    y = total / 2
    for i, h in zip(first, heights):
        pos[i] = (0.0, y - h / 2)
        y -= h
    # later layers: barycentre of predecessors, then minimum separation
    for L in range(1, n_layers):
        members = [i for i in ids if layer[i] == L]
        for i in members:
            pos[i] = (L * _DX, sum(pos[p][1] for p in preds[i]) / len(preds[i]))
        members.sort(key=lambda i: -pos[i][1])
        sink_layer = all(not succs[i] for i in members)
        for a, b in zip(members, members[1:]):
            gap = (max(k_of[a], k_of[b]) * _ISTEP_V + 0.6) if sink_layer else 2.6
            if pos[a][1] - pos[b][1] < gap:
                pos[b] = (pos[b][0], pos[a][1] - gap)
    # move constructs that sit on a long edge out of its way (alternating up / down)
    direction = 1
    for _ in range(30):
        moved = False
        for s, t in edges:
            if layer[t] - layer[s] < 2:
                continue
            (x1, y1), (x2, y2) = pos[s], pos[t]
            for c in ids:
                if c in (s, t) or not (layer[s] < layer[c] < layer[t]):
                    continue
                xc, yc = pos[c]
                yseg = y1 + (y2 - y1) * (xc - x1) / (x2 - x1)
                if abs(yc - yseg) < 0.9:
                    pos[c] = (xc, yseg + direction * 1.6)
                    direction = -direction
                    moved = True
        if not moved:
            break
    # indicator fan side: fewest collisions with structural edges and other constructs
    segs = [(pos[s], pos[t]) for s, t in edges]
    sides = {}
    for c in cons:
        i = c["id"]
        cands = []
        if layer[i] == 0:
            cands.append("left")
        if not succs[i]:
            cands.append("right")
        cands += ["above", "below", "right"] if layer[i] > 0 else ["above", "below"]
        best, best_score = None, None
        for rank, side in enumerate(dict.fromkeys(cands)):
            score = 0
            for (fx, fy) in _fan_positions(pos[i][0], pos[i][1], k_of[i], side):
                for p, q in segs:
                    if _seg_hits_rect(p, q, fx, fy, _IW + 0.12, _IH + 0.12):
                        score += 1
                for o in ids:
                    if o != i and abs(pos[o][0] - fx) < (_CW + _IW) / 2 + 0.1 and abs(pos[o][1] - fy) < (_CH + _IH) / 2 + 0.1:
                        score += 5
            if best_score is None or score < best_score:
                best, best_score = side, score
        sides[i] = best
    return pos, sides


def _ellipse_point(cx, cy, dx, dy, a=_CW / 2, b=_CH / 2):
    t = 1.0 / _math.sqrt((dx / a) ** 2 + (dy / b) ** 2)
    return cx + dx * t, cy + dy * t


def _box_point(cx, cy, dx, dy, w=_IW, h=_IH):
    tx = (w / 2) / abs(dx) if dx else _math.inf
    ty = (h / 2) / abs(dy) if dy else _math.inf
    t = min(tx, ty)
    return cx + dx * t, cy + dy * t


def _arrow(ax, p, q, color=INK_2, lw=1.1, style="-|>", ls="-", rad=0.0, z=2):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle=style, mutation_scale=11, color=color, lw=lw,
                                 linestyle=ls, shrinkA=0, shrinkB=0, zorder=z,
                                 connectionstyle=f"arc3,rad={rad}"))


def _label(ax, x, y, text, size=8.5, color=INK, weight="normal"):
    ax.text(x, y, text, ha="center", va="center", fontsize=size, color=color, fontweight=weight, zorder=6,
            bbox={"boxstyle": "round,pad=0.18", "fc": "white", "ec": "none", "alpha": 0.95})


def draw_model(model, title=None, paths=None, r2=None, loadings=None, significant_paths=None,
               show_covariances=True, note=None):
    """Draw the SemModel: constructs as ellipses, indicators as boxes, layered left to right.

    Reflective blocks: arrows construct -> indicator. Composite (formative) blocks: indicator ->
    construct. Optional estimates: paths {(srcId, tgtId): value}, r2 {constructId: value},
    loadings {indicatorId: value}; significant_paths {(srcId, tgtId): bool} draws
    non-significant paths dashed and grey."""
    pos, sides = model_layout(model)
    nm = names_of(model)
    ind_by_id = {i["id"]: i for i in model["indicators"]}
    fig, ax = plt.subplots()
    xs, ys = [], []
    for c in model["constructs"]:
        cid = c["id"]
        cx, cy = pos[cid]
        xs += [cx - _CW / 2, cx + _CW / 2]
        ys += [cy - _CH / 2, cy + _CH / 2]
        ax.add_patch(Ellipse((cx, cy), _CW, _CH, fc=BLUE_FILL, ec=BLUE_DARK, lw=1.3, zorder=4))
        txt = c["name"]
        if r2 and r2.get(cid) is not None:
            ax.text(cx, cy + 0.12, txt, ha="center", va="center", fontsize=10.5, fontweight="bold", color=INK, zorder=5)
            ax.text(cx, cy - 0.17, f"R² = {r2[cid]:.2f}", ha="center", va="center", fontsize=8, color=INK_2, zorder=5)
        else:
            ax.text(cx, cy, txt, ha="center", va="center", fontsize=10.5, fontweight="bold", color=INK, zorder=5)
        fan = _fan_positions(cx, cy, len(c["indicatorIds"]), sides[cid])
        reflective = c["measurement"] == "reflective"
        for iid, (fx, fy) in zip(c["indicatorIds"], fan):
            xs += [fx - _IW / 2, fx + _IW / 2]
            ys += [fy - _IH / 2, fy + _IH / 2]
            ax.add_patch(FancyBboxPatch((fx - _IW / 2, fy - _IH / 2), _IW, _IH, boxstyle="round,pad=0,rounding_size=0.05",
                                        fc=GREY_FILL, ec=INK_3, lw=0.9, zorder=4))
            ax.text(fx, fy, ind_by_id[iid]["name"], ha="center", va="center", fontsize=8, color=INK, zorder=5)
            dx, dy = fx - cx, fy - cy
            pc = _ellipse_point(cx, cy, dx, dy)
            pi = _box_point(fx, fy, -dx, -dy)
            if reflective:
                _arrow(ax, pc, pi, color=INK_3, lw=0.8)
            else:
                _arrow(ax, pi, pc, color=INK_3, lw=0.8)
            if loadings and loadings.get(iid) is not None:
                mx, my = (pc[0] + pi[0]) / 2, (pc[1] + pi[1]) / 2
                _label(ax, mx, my, f"{loadings[iid]:.2f}", size=6.5, color=INK_2)
    covs = []
    placed = []
    seg_list = [((s, t), pos[s], pos[t]) for s, t in _structural(model)]
    for r in model["relationships"]:
        s, t = r["sourceId"], r["targetId"]
        (x1, y1), (x2, y2) = pos[s], pos[t]
        if r["kind"] == "structural":
            p = _ellipse_point(x1, y1, x2 - x1, y2 - y1)
            q = _ellipse_point(x2, y2, x1 - x2, y1 - y2)
            ns = significant_paths is not None and significant_paths.get((s, t)) is False
            _arrow(ax, p, q, color=INK_3 if ns else BLUE_DARK, lw=1.0 if ns else 1.5, ls="--" if ns else "-")
            if paths and paths.get((s, t)) is not None:
                lx, ly = _free_spot(p, q, (s, t), seg_list, placed)
                placed.append((lx, ly))
                _label(ax, lx, ly, f"{paths[(s, t)]:.2f}",
                       size=8.5, color=INK_3 if ns else INK, weight="normal" if ns else "bold")
        elif r["kind"] == "covariance":
            covs.append(f"{nm[s]} \u2194 {nm[t]}")
    pad = 0.45
    has_note = bool(note) or (bool(covs) and show_covariances)
    x0, x1_, y0, y1_ = min(xs) - pad, max(xs) + pad, min(ys) - pad - (0.55 if has_note else 0), max(ys) + pad
    ax.set_xlim(x0, x1_)
    ax.set_ylim(y0, y1_)
    ax.set_aspect("equal")
    ax.axis("off")
    scale = 0.78
    fig.set_size_inches(max(6.5, (x1_ - x0) * scale), max(3.5, (y1_ - y0) * scale + (0.5 if title else 0)))
    if title:
        ax.set_title(title, loc="left", fontsize=11.5, pad=8)
    notes = []
    if covs and show_covariances:
        notes.append("Construct covariances in the model (not drawn): " + ", ".join(covs) + ".")
    if note:
        notes.append(note)
    if notes:
        ax.text(x0 + 0.1, y0 + 0.05, "\n".join(notes), fontsize=8, color=INK_2, ha="left", va="bottom")
    return fig


def _dist_point_segment(x, y, p, q):
    (x1, y1), (x2, y2) = p, q
    dx, dy = x2 - x1, y2 - y1
    t = max(0.0, min(1.0, ((x - x1) * dx + (y - y1) * dy) / (dx * dx + dy * dy)))
    return _math.hypot(x - (x1 + t * dx), y - (y1 + t * dy))


def _free_spot(p, q, key, segs, placed):
    """A point along p-q for a path label, away from other paths and labels."""
    best, best_d = None, -1.0
    for f in (0.5, 0.4, 0.6, 0.3, 0.7, 0.25, 0.75):
        x, y = p[0] + (q[0] - p[0]) * f, p[1] + (q[1] - p[1]) * f
        d = min([_dist_point_segment(x, y, a, b) for k, a, b in segs if k != key] + [9.0])
        d = min([d] + [_math.hypot(x - lx, y - ly) * 0.6 for lx, ly in placed])
        if d >= 0.45:
            return x, y
        if d > best_d:
            best, best_d = (x, y), d
    return best


def _place_labels(ax, xs, ys, labels, fontsize=9):
    """Annotate points, trying a few offsets so that labels do not overlap each other."""
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = []
    offsets = [(7, 4), (7, -13), (-9, 4), (-9, -13), (7, 15), (-9, 15), (7, -24), (-9, -24)]
    for x, y, lab in zip(xs, ys, labels):
        chosen = None
        for ox, oy in offsets:
            ann = ax.annotate(lab, (x, y), xytext=(ox, oy), textcoords="offset points", fontsize=fontsize,
                              color=INK, ha="left" if ox > 0 else "right")
            bb = ann.get_window_extent(renderer).expanded(1.05, 1.15)
            if not any(bb.overlaps(o) for o in boxes):
                chosen = ann
                boxes.append(bb)
                break
            ann.remove()
        if chosen is None:
            ann = ax.annotate(lab, (x, y), xytext=offsets[0], textcoords="offset points", fontsize=fontsize, color=INK)
            boxes.append(ann.get_window_extent(renderer))


def path_dict(paths_df, model, column="est"):
    """{(sourceId, targetId): value} from a paths table that has 'from'/'to' names."""
    by_name = {c["name"]: c["id"] for c in model["constructs"]}
    return {(by_name[r["from"]], by_name[r["to"]]): r[column] for _, r in paths_df.iterrows()}


# ---------------------------------------------------------------------------------------------
# Result charts
# ---------------------------------------------------------------------------------------------

def coefficient_plot(df, label_col="label", title="Path coefficients with 95% bootstrap CIs",
                     xlabel="Standardised path coefficient"):
    """Point estimates with percentile CIs; CIs that include 0 are drawn grey and hollow."""
    d = df.reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(7.2, 0.8 + 0.42 * len(d)))
    y = _np.arange(len(d))[::-1]
    for yy, (_, r) in zip(y, d.iterrows()):
        sig = significant(r)
        col = BLUE if sig else INK_3
        ax.plot([r["ciLower"], r["ciUpper"]], [yy, yy], color=col, lw=2, solid_capstyle="round", zorder=2)
        ax.plot(r["est"], yy, "o", ms=8, color=col if sig else "white", mec=col, mew=2, zorder=3)
        ax.text(r["ciUpper"] + 0.02 * (max(0.0, d["ciUpper"].max()) - min(0.0, d["ciLower"].min())), yy, f"{r['est']:.3f}" + ("" if sig else "  (CI includes 0)"),
                va="center", fontsize=8, color=INK_2)
    ax.axvline(0, color=INK_3, lw=0.8, ls=(0, (3, 2)), zorder=1)
    ax.set_yticks(y, d[label_col])
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)
    lo = min(0.0, d["ciLower"].min())
    hi = max(0.0, d["ciUpper"].max())
    xr = hi - lo
    ax.set_xlim(lo - 0.06 * xr, hi + 0.30 * xr)
    ax.set_xlabel(xlabel)
    ax.xaxis.grid(True, color="#eeeeec", lw=0.6)
    ax.set_axisbelow(True)
    ax.set_title(title, loc="left")
    fig.tight_layout()
    return fig


def bootstrap_histogram(draws, est, ci, label, title=None):
    draws = _np.asarray([d for d in draws if d is not None], dtype=float)
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    ax.hist(draws, bins=50, color=BLUE, edgecolor="white", linewidth=0.6)
    ax.axvline(est, color=INK, lw=1.4, label=f"estimate {est:.3f}")
    ax.axvline(ci[0], color=ORANGE, lw=1.2, ls=(0, (4, 2)), label=f"95% percentile CI [{ci[0]:.3f}, {ci[1]:.3f}]")
    ax.axvline(ci[1], color=ORANGE, lw=1.2, ls=(0, (4, 2)))
    ax.axvline(0, color=INK_3, lw=0.8, label="zero")
    ax.legend(loc="upper left", fontsize=8)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.25)
    ax.set_xlabel(label)
    ax.set_ylabel("bootstrap draws")
    ax.set_title(title or f"Bootstrap distribution: {label}", loc="left")
    fig.tight_layout()
    return fig


def ipma_plot(ipma_df, target_name, label_col="construct", title=None):
    """Importance-performance map: x = importance (unstandardised total effect), y = performance (0-100)."""
    d = ipma_df
    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    ax.scatter(d["importance"], d["performance"], s=70, color=BLUE, edgecolor="white", linewidth=1.5, zorder=3)
    mi, mp = d["importance"].mean(), d["performance"].mean()
    ax.axvline(mi, color=INK_3, lw=0.8, ls=(0, (3, 2)))
    ax.axhline(mp, color=INK_3, lw=0.8, ls=(0, (3, 2)))
    xr = d["importance"].max() - min(0, d["importance"].min())
    ax.set_xlim(min(0, d["importance"].min()) - 0.05 * xr, d["importance"].max() + 0.25 * xr)
    yr = max(5.0, d["performance"].max() - d["performance"].min())
    ax.set_ylim(d["performance"].min() - 0.25 * yr, d["performance"].max() + 0.25 * yr)
    ax.text(ax.get_xlim()[1], ax.get_ylim()[0], "high importance,\nlow performance:\nimprovement priority  ",
            ha="right", va="bottom", fontsize=8, color=INK_2)
    ax.set_xlabel(f"Importance (unstandardised total effect on {target_name})")
    ax.set_ylabel("Performance (0-100)")
    ax.grid(True, color="#eeeeec", lw=0.6)
    ax.set_axisbelow(True)
    ax.set_title(title or f"Importance-performance map, target {target_name}", loc="left")
    fig.tight_layout()
    _place_labels(ax, list(d["importance"]), list(d["performance"]), [str(v) for v in d[label_col]])
    return fig


def heatmap_table(df, title, threshold=None, fmt="{:.3f}"):
    """A small annotated matrix (e.g. HTMT); cells above `threshold` are outlined in orange."""
    m = df.astype(float)
    if m.shape[0] > 1 and m.iloc[0].isna().all() and m.iloc[:, -1].isna().all():
        m = m.iloc[1:, :-1]  # lower triangle without the empty first row and last column
    fig, ax = plt.subplots(figsize=(1.2 + 0.85 * m.shape[1], 0.9 + 0.55 * m.shape[0]))
    vals = m.values
    ax.imshow(_np.where(_np.isnan(vals), _np.nan, vals), cmap=_cmap(), vmin=0, vmax=1)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = vals[i, j]
            if _np.isnan(v):
                continue
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=8.5,
                    color="white" if v > 0.62 else INK)
            if threshold is not None and v > threshold:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=ORANGE, lw=2))
    ax.set_xticks(range(m.shape[1]), m.columns)
    ax.set_yticks(range(m.shape[0]), m.index)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, loc="left")
    fig.tight_layout()
    return fig


def _cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("blue", ["#ffffff"] + BLUE_RAMP[:11])


# ---------------------------------------------------------------------------------------------
# Export: CSV, PNG and a Word-ready table (HTML with inline styles)
# ---------------------------------------------------------------------------------------------

def word_table_html(df, title, note=None, decimals=3, index=False):
    """An APA-style table as HTML with inline styles only. Open the file in a browser, select
    the table, copy, and paste into Word or Google Docs: it arrives as an editable table."""
    d = df.reset_index() if index else df
    cell = "padding:2pt 7pt;font-family:'Times New Roman',Times,serif;font-size:11pt;"
    head = cell + "border-top:1.5pt solid #000;border-bottom:0.75pt solid #000;font-weight:normal;"
    parts = ["<table style=\"border-collapse:collapse;\">",
             (f"<caption style=\"caption-side:top;text-align:left;{cell}padding-left:0;\">"
              f"<b>{_html.escape(title)}</b></caption>"), "<thead><tr>"]
    for k, col in enumerate(d.columns):
        align = "left" if k == 0 else "right"
        parts.append(f"<th style=\"{head}text-align:{align};\">{_html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")
    nrow = len(d)
    for r_i, (_, row) in enumerate(d.iterrows()):
        parts.append("<tr>")
        for k, col in enumerate(d.columns):
            v = row[col]
            if isinstance(v, (float, _np.floating)):
                txt = "" if _np.isnan(v) else f"{v:.{decimals}f}"
            elif v is None:
                txt = ""
            else:
                txt = str(v)
            style = cell + ("text-align:left;" if k == 0 else "text-align:right;")
            if r_i == nrow - 1:
                style += "border-bottom:1.5pt solid #000;"
            parts.append(f"<td style=\"{style}\">{_html.escape(txt)}</td>")
        parts.append("</tr>")
    parts.append("</tbody></table>")
    if note:
        parts.append(f"<p style=\"{cell}font-size:10pt;margin-top:4pt;\"><i>Note.</i> {_html.escape(note)}</p>")
    return "\n".join(parts)


def save_word_tables(path, tables, page_title):
    """Write several Word-ready tables [(title, df, note, decimals)] into one HTML file."""
    body = []
    for title, df, note, decimals in tables:
        body.append(word_table_html(df, title, note, decimals))
        body.append("<p>&nbsp;</p>")
    doc = ("<!doctype html>\n<html><head><meta charset=\"utf-8\"><title>" + _html.escape(page_title) +
           "</title></head>\n<body style=\"background:#ffffff;color:#000000;\">\n" + "\n".join(body) + "\n</body></html>\n")
    with open(path, "w", encoding="utf-8") as f:
        f.write(doc)
    return path


def save_figure(fig, folder, name):
    ensure_dir(folder)
    p = _os.path.join(folder, name)
    fig.savefig(p)
    return p


def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        _json.dump(obj, f, indent=2, ensure_ascii=False)
        f.write("\n")
    return path


def zip_folder(folder, zip_path):
    with _zipfile.ZipFile(zip_path, "w", _zipfile.ZIP_DEFLATED) as z:
        for root, _, files in _os.walk(folder):
            for fn in sorted(files):
                full = _os.path.join(root, fn)
                z.write(full, _os.path.relpath(full, _os.path.dirname(folder) or "."))
    return zip_path


def offer_download(path):
    """In Colab, start a browser download; elsewhere just report where the file is."""
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        print(f"Saved {path} (outside Colab: find it in the notebook's working directory).")
        return
    files.download(path)


# ---------------------------------------------------------------------------------------------
# PLS result tables (readable pandas views of semlab.results_tables)
# ---------------------------------------------------------------------------------------------

def _matrix(frames, key):
    m = frames[key].set_index("")
    m.index.name = None
    return m.astype(float)


def pls_report(result, model, semlab):
    """Readable tables from a NormalizedSemResult of a PLS run."""
    t = semlab.to_dataframes(result, model)
    rep = {}
    load = t["loadings"].rename(columns={"from": "construct", "to": "indicator", "est": "loading"})
    load["loading²"] = load["loading"] ** 2
    rep["loadings"] = load[["construct", "indicator", "loading", "loading²", "se", "ciLower", "ciUpper"]].reset_index(drop=True)
    w = t["weights"].rename(columns={"from": "construct", "to": "indicator", "est": "weight"})
    rep["weights"] = w[["construct", "indicator", "weight", "se", "ciLower", "ciUpper"]].reset_index(drop=True)
    cm = t["constructMetrics"].set_index("construct")
    cm.index.name = None
    cols = [c for c in ["cronbach-alpha", "rho-a", "rho-c", "ave", "r2", "r2-adjusted"] if c in cm.columns]
    rep["quality"] = cm[cols].astype(float).rename(columns={"cronbach-alpha": "alpha", "rho-a": "rho_A", "rho-c": "rho_C",
                                                             "ave": "AVE", "r2": "R²", "r2-adjusted": "R² adj."})
    htmt = _matrix(t, "matrix:htmt")
    rep["htmt"] = htmt.where(_np.tril(_np.ones(htmt.shape, dtype=bool), -1))
    rep["fornell-larcker"] = _matrix(t, "matrix:fornell-larcker")
    paths = t["paths"].copy()
    paths["path"] = paths["from"] + f" {ARROW} " + paths["to"]
    paths["significant (95% CI excludes 0)"] = [significant(r) for _, r in paths.iterrows()]
    rep["paths"] = paths[["path", "key", "from", "to", "est", "se", "statistic", "pValue", "ciLower", "ciUpper",
                          "significant (95% CI excludes 0)"]].reset_index(drop=True)
    rep["f2"] = _matrix(t, "matrix:f2").dropna(how="all").dropna(axis=1, how="all")
    rep["vif"] = _matrix(t, "matrix:vif-structural").dropna(how="all").dropna(axis=1, how="all")
    eff = t["effects"].copy()
    if len(eff):
        eff["effect"] = eff["kind"].map({"total-effect": "total", "indirect-effect": "total indirect",
                                         "specific-indirect": "specific indirect"})
        eff["route"] = [
            f"{r['from']} {ARROW} " + (f"{r['via'].replace(' -> ', f' {ARROW} ')} {ARROW} " if isinstance(r["via"], str) and r["via"] else "") + r["to"]
            for _, r in eff.iterrows()]
        eff["significant (95% CI excludes 0)"] = [significant(r) for _, r in eff.iterrows()]
        rep["effects"] = eff[["effect", "route", "key", "from", "to", "est", "se", "pValue", "ciLower", "ciUpper",
                              "significant (95% CI excludes 0)"]].reset_index(drop=True)
    else:
        rep["effects"] = eff
    rep["fit"] = t["fit"][["name", "value"]].set_index("name")
    rep["warnings"] = t["warnings"]
    return rep


def show(df, decimals=3):
    """Display a DataFrame rounded for reading (the unrounded numbers stay in the variable)."""
    try:
        from IPython.display import display
    except ImportError:  # pragma: no cover
        print(df.round(decimals).to_string())
        return
    display(df.round(decimals))

## 2. Data: synthetic and reproducible

The data are generated from a population model with three standardised common factors
(PEOU, PU, BI). Each respondent's continuous item responses are drawn from the
model-implied multivariate normal distribution and then cut into seven categories (1–7) by
fixed thresholds; the PU items use a "high-agreement" profile, so their answers lean
towards 7. The generator is the one that produced SEM Lab's bundled dataset
(`packages/fixtures/scripts/generate_synthetic.py`): the collapsed cell holds its general functions, the visible cell the
population model of this case. With seed 1103 and N = 500 the notebook reproduces the
bundled CSV byte for byte, which the cell after them checks with a SHA-256 hash.

In [ ]:
#@title Synthetic data generator: shared functions (click to expand)
# Extracted automatically from packages/fixtures/scripts/generate_synthetic.py by notebooks/build_notebooks.py
# (byte-for-byte the code that wrote the fixture; do not edit here).
import csv
import hashlib
import io
import math

import numpy as np


SYMMETRIC_7 = (-1.8, -1.1, -0.4, 0.4, 1.1, 1.8)


HIGH_AGREEMENT_SHIFT = -0.6


class GenerationError(RuntimeError):
    pass


def require(cond: bool, msg: str) -> None:
    if not cond:
        raise GenerationError(msg)


def dot(a, b) -> float:
    return math.fsum(x * y for x, y in zip(a, b))


def transpose(a):
    return [list(r) for r in zip(*a)]


def matmul(a, b):
    bt = transpose(b)
    return [[dot(r, c) for c in bt] for r in a]


def identity(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def mat_sub(a, b):
    return [[x - y for x, y in zip(ra, rb)] for ra, rb in zip(a, b)]


def cholesky(s):
    """Lower Cholesky factor; raises if the matrix is not positive definite."""
    n = len(s)
    lo = [[0.0] * n for _ in range(n)]
    for j in range(n):
        d = s[j][j] - math.fsum(lo[j][k] * lo[j][k] for k in range(j))
        if not d > 0.0:
            raise GenerationError(f"matrix is not positive definite (pivot {j}: {d!r})")
        lo[j][j] = math.sqrt(d)
        for i in range(j + 1, n):
            lo[i][j] = (s[i][j] - math.fsum(lo[i][k] * lo[j][k] for k in range(j))) / lo[j][j]
    return lo


def inverse(a):
    """Gauss-Jordan inverse with partial pivoting (small matrices only)."""
    n = len(a)
    m = [list(map(float, r)) + identity(n)[i] for i, r in enumerate(a)]
    for c in range(n):
        p = max(range(c, n), key=lambda r: abs(m[r][c]))
        require(abs(m[p][c]) > 1e-14, "singular matrix in inverse()")
        m[c], m[p] = m[p], m[c]
        piv = m[c][c]
        m[c] = [x / piv for x in m[c]]
        for r in range(n):
            if r != c and m[r][c] != 0.0:
                f = m[r][c]
                m[r] = [x - f * y for x, y in zip(m[r], m[c])]
    return [r[n:] for r in m]


def reflective(name, label, indicators, loadings):
    require(len(indicators) == len(loadings), f"{name}: loading count")
    return {"name": name, "label": label, "measurement": "reflective",
            "indicators": list(indicators), "loadings": [float(x) for x in loadings]}


def build_population(spec):
    """Compute Phi, Sigma (standardised and raw), mu and derived truths from parameters."""
    cons = spec["constructs"]
    names = [c["name"] for c in cons]
    m = len(cons)
    idx = {n: i for i, n in enumerate(names)}

    # Structural matrix B (row = target, column = source); constructs must be listed in
    # a topological order so that B is strictly lower triangular (recursive model).
    b = [[0.0] * m for _ in range(m)]
    for src, tgt, coef in spec.get("paths", []):
        require(idx[src] < idx[tgt], f"construct order is not topological at {src}->{tgt}")
        b[idx[tgt]][idx[src]] = float(coef)
    endogenous = [any(b[i][j] != 0.0 for j in range(m)) for i in range(m)]

    # Psi: exogenous block = specified correlations, unit variances; endogenous
    # disturbance variances chosen so the latent variance is exactly 1.
    psi = [[0.0] * m for _ in range(m)]
    for i in range(m):
        if not endogenous[i]:
            psi[i][i] = 1.0
    for (a, c), r in spec.get("exoCorrelations", {}).items():
        i, j = idx[a], idx[c]
        require(not endogenous[i] and not endogenous[j], f"correlation {a}~{c} must be exogenous")
        psi[i][j] = psi[j][i] = float(r)
    # sequential computation of the covariance among predecessors
    phi_seq = [[0.0] * m for _ in range(m)]
    for i in range(m):
        if not endogenous[i]:
            for j in range(m):
                if not endogenous[j]:
                    phi_seq[i][j] = psi[i][j]
    for i in range(m):
        if endogenous[i]:
            for k in range(i):
                v = math.fsum(b[i][l] * phi_seq[l][k] for l in range(i))
                phi_seq[i][k] = phi_seq[k][i] = v
            explained = math.fsum(b[i][l] * phi_seq[l][k] * b[i][k] for l in range(i) for k in range(i))
            require(explained < 1.0, f"{names[i]}: explained variance {explained} >= 1")
            psi[i][i] = 1.0 - explained
            phi_seq[i][i] = 1.0
    require(all(psi[i][i] > 0 for i in range(m)), "non-positive disturbance variance")
    cholesky(psi)  # Psi must be positive definite

    inv_ib = inverse(mat_sub(identity(m), b))
    phi = matmul(matmul(inv_ib, psi), transpose(inv_ib))
    for i in range(m):
        require(abs(phi[i][i] - 1.0) < 1e-12, f"latent variance of {names[i]} is {phi[i][i]!r}, not 1")
        phi[i][i] = 1.0
        for j in range(m):
            require(abs(phi[i][j] - phi_seq[i][j]) < 1e-12, "Phi formula and sequential Phi disagree")
    phi = [[(phi[i][j] + phi[j][i]) / 2.0 for j in range(m)] for i in range(m)]

    # indicators: loading vector l (lambda for reflective, R w for composite) and the
    # within-block residual covariance (diag(1-lambda^2) or R - (Rw)(Rw)').
    cols, owner, lvec = [], [], []
    blocks = []
    for ci, c in enumerate(cons):
        k = len(c["indicators"])
        if c["measurement"] == "reflective":
            for lam in c["loadings"]:
                require(0.0 < lam < 1.0, f"{c['name']}: standardised loading {lam} outside (0,1)")
            l_c = list(c["loadings"])
            u = [[(1.0 - l_c[i] * l_c[i]) if i == j else 0.0 for j in range(k)] for i in range(k)]
        else:
            r = c["R"]
            w0 = c["rawWeights"]
            scale = math.sqrt(dot(w0, [dot(row, w0) for row in r]))
            w = [x / scale for x in w0]
            c["weights"] = w
            rw = [dot(row, w) for row in r]
            require(abs(dot(w, rw) - 1.0) < 1e-12, f"{c['name']}: w'Rw != 1")
            c["compositeLoadings"] = rw
            l_c = rw
            u = [[r[i][j] - rw[i] * rw[j] for j in range(k)] for i in range(k)]
        blocks.append((len(cols), k, u))
        cols.extend(c["indicators"])
        owner.extend([ci] * k)
        lvec.extend(l_c)
    p = len(cols)
    sig = [[lvec[i] * phi[owner[i]][owner[j]] * lvec[j] for j in range(p)] for i in range(p)]
    for start, k, u in blocks:
        for i in range(k):
            for j in range(k):
                sig[start + i][start + j] += u[i][j]
    for i in range(p):
        require(abs(sig[i][i] - 1.0) < 1e-12, f"standardised variance of {cols[i]} is {sig[i][i]!r}")
        sig[i][i] = 1.0
    cholesky(sig)  # positive definite (raises otherwise)

    sds = [float(spec.get("sds", {}).get(cn, 1.0)) for cn in cols]
    means = [float(spec.get("means", {}).get(cn, 0.0)) for cn in cols]
    sig_raw = [[sds[i] * sig[i][j] * sds[j] for j in range(p)] for i in range(p)]
    return {"spec": spec, "names": names, "B": b, "Psi": psi, "Phi": phi, "invIB": inv_ib,
            "endogenous": endogenous, "columns": cols, "owner": owner, "l": lvec,
            "SigmaStd": sig, "Sigma": sig_raw, "sds": sds, "means": means}


def rng_for(seed):
    return np.random.Generator(np.random.PCG64(seed))


def draw_sampled(pop, n, seed):
    """Return (standardised rows, raw rows, generator) for N rows of N(mu, Sigma)."""
    rng = rng_for(seed)
    p = len(pop["columns"])
    z = rng.standard_normal((n, p)).tolist()
    lo = cholesky(pop["SigmaStd"])
    std_rows = [[dot(lo[i][: i + 1], zr[: i + 1]) for i in range(p)] for zr in z]
    raw_rows = [[pop["means"][i] + pop["sds"][i] * r[i] for i in range(p)] for r in std_rows]
    return std_rows, raw_rows, rng


def likert(x, thresholds, base):
    return base + sum(1 for t in thresholds if x > t)


def likert_profile(name, thresholds, base, smin, smax):
    return {"profile": name, "thresholds": list(thresholds), "lowestCategory": base,
            "highestCategory": base + len(thresholds), "scaleMin": smin, "scaleMax": smax,
            "rule": "category = lowestCategory + number of thresholds strictly exceeded by the standardised continuous response"}


SYM = likert_profile("symmetric-7", SYMMETRIC_7, 1, 1, 7)


HIGH = likert_profile("high-agreement-7", tuple(round(t + HIGH_AGREEMENT_SHIFT, 10) for t in SYMMETRIC_7), 1, 1, 7)


def discretise(std_rows, columns, profiles):
    out = []
    for r in std_rows:
        out.append([likert(r[j], profiles[c]["thresholds"], profiles[c]["lowestCategory"])
                    for j, c in enumerate(columns)])
    return out


def cell(v):
    if v is None:
        return ""
    if isinstance(v, bool):
        raise GenerationError("boolean cell")
    if isinstance(v, int):
        return str(v)
    require(math.isfinite(v), "non-finite cell")
    return repr(float(v))


def csv_bytes(header, rows):
    buf = io.StringIO()
    w = csv.writer(buf, lineterminator="\n")
    w.writerow(header)
    for r in rows:
        w.writerow([cell(v) for v in r])
    return buf.getvalue().encode("utf-8")

In [ ]:
# The case's population model and data function, extracted from packages/fixtures/scripts/generate_synthetic.py.

TAM_COLS = ["peou1", "peou2", "peou3", "peou4", "pu1", "pu2", "pu3", "pu4", "bi1", "bi2", "bi3"]


def spec_tam(sds=None, means=None):
    return {"constructs": [reflective("PEOU", "Perceived ease of use (synthetic)", TAM_COLS[0:4], [.80, .78, .76, .74]),
                           reflective("PU", "Perceived usefulness (synthetic)", TAM_COLS[4:8], [.85, .82, .80, .78]),
                           reflective("BI", "Behavioural intention (synthetic)", TAM_COLS[8:11], [.88, .85, .82])],
            "paths": [("PEOU", "PU", .60), ("PEOU", "BI", .20), ("PU", "BI", .50)],
            "sds": sds or {}, "means": means or {}}


def data_tam_mediation(seed, n):
    """tam-mediation: population, Likert profile of every item, header and integer rows.
    Also used verbatim by notebooks/colab/01_tam_pls_sem.ipynb (extracted by notebooks/build_notebooks.py)."""
    pop = build_population(spec_tam())
    std, _, _ = draw_sampled(pop, n, seed)
    profiles = {c: (HIGH if c.startswith("pu") else SYM) for c in pop["columns"]}
    rows = discretise(std, pop["columns"], profiles)
    return pop, profiles, pop["columns"], rows

In [ ]:
# Generate the data with the fixture's seed and N, and prove they are byte-identical to the
# dataset bundled with SEM Lab (packages/fixtures/synthetic/tam-mediation/data.csv).
import pandas as pd

SEED, N = 1103, 500
EXPECTED_DATA_SHA256 = "0cf89e16911c5bdc5236a3751d46010eed86a17a36e7565e1fdc50017c89a31f"  # manifest.json of the fixture

pop, profiles, header, rows = data_tam_mediation(SEED, N)
csv_data = csv_bytes(header, rows)
data_sha = hashlib.sha256(csv_data).hexdigest()
assert data_sha == EXPECTED_DATA_SHA256, f"data sha256 {data_sha} differs from the fixture's {EXPECTED_DATA_SHA256}"
with open("data.csv", "wb") as f:
    f.write(csv_data)

df = pd.read_csv("data.csv")
print(f"data.csv written: {len(df)} rows x {df.shape[1]} columns, SYNTHETIC")
print(f"sha256 {data_sha} == fixture tam-mediation: the SEM Lab example uses exactly these bytes")
df.head()

### The population model behind the data

These are the *true* values the generator used. They describe standardised common factors on
the continuous scale, before the responses were cut into categories. A sample estimate will not
reproduce them exactly: there is sampling error (N = 500), discretisation into categories, and
classic PLS-SEM estimates composites, not common factors (its paths are attenuated when
indicators carry measurement error; see the PLSc comparison later).

In [ ]:
pop_loadings = pd.DataFrame(
    [{"construct": c["name"], "indicator": ind, "population loading": lam}
     for c in pop["spec"]["constructs"] for ind, lam in zip(c["indicators"], c["loadings"])])
pop_paths = pd.DataFrame(
    [{"path": f"{s} → {t}", "population coefficient": b} for s, t, b in pop["spec"]["paths"]])
pop_r2 = pd.DataFrame(
    [{"construct": nm, "population R²": 1.0 - pop["Psi"][i][i]}
     for i, nm in enumerate(pop["names"]) if pop["endogenous"][i]])
print("Structural coefficients (standardised):")
display(pop_paths.round(3))
print("Explained variance of the endogenous constructs:")
display(pop_r2.round(3))
print("Standardised loadings:")
display(pop_loadings.pivot_table(index="construct", columns=pop_loadings.groupby("construct").cumcount() + 1,
                                 values="population loading", sort=False).round(2))

## 3. Indicators: mapping, dictionary and descriptives

The model below is SEM Lab's canonical model file for this example
(`packages/fixtures/synthetic/tam-mediation/model.json`), embedded unchanged. Each indicator
is bound to a CSV column and declares its scale endpoints, which IPMA needs to rescale
scores to 0–100.

In [ ]:
import json

MODEL_JSON = r'''
{
  "constructs": [
    {
      "id": "c_peou",
      "indicatorIds": [
        "i_peou1",
        "i_peou2",
        "i_peou3",
        "i_peou4"
      ],
      "label": "Perceived ease of use (synthetic)",
      "measurement": "reflective",
      "name": "PEOU"
    },
    {
      "id": "c_pu",
      "indicatorIds": [
        "i_pu1",
        "i_pu2",
        "i_pu3",
        "i_pu4"
      ],
      "label": "Perceived usefulness (synthetic)",
      "measurement": "reflective",
      "name": "PU"
    },
    {
      "id": "c_bi",
      "indicatorIds": [
        "i_bi1",
        "i_bi2",
        "i_bi3"
      ],
      "label": "Behavioural intention (synthetic)",
      "measurement": "reflective",
      "name": "BI"
    }
  ],
  "indicators": [
    {
      "column": "peou1",
      "id": "i_peou1",
      "level": "ordinal",
      "name": "peou1",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "peou2",
      "id": "i_peou2",
      "level": "ordinal",
      "name": "peou2",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "peou3",
      "id": "i_peou3",
      "level": "ordinal",
      "name": "peou3",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "peou4",
      "id": "i_peou4",
      "level": "ordinal",
      "name": "peou4",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "pu1",
      "id": "i_pu1",
      "level": "ordinal",
      "name": "pu1",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "pu2",
      "id": "i_pu2",
      "level": "ordinal",
      "name": "pu2",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "pu3",
      "id": "i_pu3",
      "level": "ordinal",
      "name": "pu3",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "pu4",
      "id": "i_pu4",
      "level": "ordinal",
      "name": "pu4",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "bi1",
      "id": "i_bi1",
      "level": "ordinal",
      "name": "bi1",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "bi2",
      "id": "i_bi2",
      "level": "ordinal",
      "name": "bi2",
      "scale": {
        "max": 7,
        "min": 1
      }
    },
    {
      "column": "bi3",
      "id": "i_bi3",
      "level": "ordinal",
      "name": "bi3",
      "scale": {
        "max": 7,
        "min": 1
      }
    }
  ],
  "relationships": [
    {
      "id": "r_peou_pu",
      "kind": "structural",
      "sourceId": "c_peou",
      "targetId": "c_pu"
    },
    {
      "id": "r_peou_bi",
      "kind": "structural",
      "sourceId": "c_peou",
      "targetId": "c_bi"
    },
    {
      "id": "r_pu_bi",
      "kind": "structural",
      "sourceId": "c_pu",
      "targetId": "c_bi"
    }
  ]
}
'''
model = json.loads(MODEL_JSON)
assert hashlib.sha256((MODEL_JSON.strip("\n") + "\n").encode("utf-8")).hexdigest() == "9a1315f2573f0889c1f6ed1ae1bf47e55b9056f438b9e9398882cd552da13b72"
with open("model.json", "w", encoding="utf-8") as f:
    f.write(MODEL_JSON.strip("\n") + "\n")

dictionary = data_dictionary(model, profiles)
dictionary

In [ ]:
desc = descriptives(df, model)
show(desc, 2)

In [ ]:
FIGURES = {}  # every figure is collected here for the export section
FIGURES["item_distributions.png"] = likert_distribution_plot(df, model)
plt.show()

**Reading the descriptives.** All items use the full 1–7 range. The PU items have higher means
(about 4.7 against 3.8–4.0) and a slight negative skew because of their high-agreement
profile, a typical pattern for usefulness items. PLS-SEM does not assume normally distributed indicators, but strongly skewed items are
worth noting in a report.

## 4. The model

The figure is drawn from the same canonical model file the app uses. Reflective measurement:
arrows point from each construct to its indicators (the construct is assumed to cause the
item responses). Structural paths connect the constructs.

In [ ]:
FIGURES["model.png"] = draw_model(model, title="TAM: measurement and structural model")
plt.show()

## 5. Estimation with SEM Lab's engine

The analysis settings below are those of the SEM Lab example: classic PLS-SEM (Mode A
weights for reflective blocks), path weighting scheme, at most 300 iterations, tolerance 1e-7,
and a **percentile bootstrap** with a fixed seed so that every run gives identical intervals.

**Bootstrap size.** This notebook uses B = 5,000 draws, a common choice for reporting final
results. The estimation cell took about 4 s on an Apple M-series laptop; a free Colab CPU can be a few times slower (the cell prints its own time). The app's example opens with B = 1,000 for speed; the point
estimates are identical, and setting 5,000 draws with seed 20261103 in the app gives exactly the
intervals shown here, because SEM Lab's bootstrap draws the same resamples in the browser,
in Python and in R.

In [ ]:
analysis = {
    "method": "pls-sem",
    "engine": "browser",          # SEM Lab's engine (here its Python port)
    "estimator": "pls-pm",
    "missingData": "listwise",
    "seed": 20261103,
    "bootstrap": {"enabled": True, "repetitions": 5000, "ci": "percentile",
                  "confidenceLevel": 0.95, "sides": "two-sided"},
    "pls": {"innerWeighting": "path", "maxIterations": 300, "tolerance": 1e-7},
}

In [ ]:
import time

columns, data_rows = semlab.read_csv("data.csv")
data = semlab.build_analysis_data(model, columns, data_rows, analysis["missingData"])
t0 = time.perf_counter()
out = semlab.estimate(model, analysis, data)
elapsed = time.perf_counter() - t0
if not out["ok"]:
    raise RuntimeError(f"{out['error']['code']}: {out['error']['message']}")
result = out["result"]
est = result["estimation"]
boot = result.get("bootstrap") or {}
print(f"N used: {result['sample']['nUsed']} of {result['sample']['nRows']} rows ({result['sample']['missingData']})")
print(f"converged: {est['converged']} after {est['iterations']} iterations; admissible: {est['admissible']}")
if boot:
    print(f"bootstrap: {boot['completed']} of {boot['requested']} draws completed, {boot['failed']} failed, "
          f"{boot['inadmissible']} inadmissible; seed {boot['seed']}; {boot['ciMethod']} CI, level {boot['confidenceLevel']}")
print(f"estimation incl. bootstrap took {elapsed:.1f} s")
for w in result["warnings"]:
    print(f"warning [{w['code']}]: {w['message']}")
assert est["converged"] and est["admissible"]
rep = pls_report(result, model, semlab)

## 6. Measurement model

### Loadings

For reflective indicators we look at the outer loadings (the correlation of each indicator
with its construct score). A common guideline is a loading of at least 0.708, so that the
construct explains at least half of the indicator's variance (loading² ≥ 0.5).

In [ ]:
show(rep["loadings"])
print("Smallest loading:", rep["loadings"].loc[rep["loadings"]["loading"].idxmin(), ["indicator", "loading"]].to_dict())

**Interpretation.** Every loading exceeds 0.708 and every bootstrap interval is far from the
guideline, so all indicators are kept. The population loadings were 0.74–0.88: the estimates
are close, slightly *higher* for most items, because composite loadings also absorb part of each
indicator's own error variance.

### Reliability and convergent validity

| Criterion | Guideline |
|---|---|
| Cronbach's α (lower bound) and ρC (upper bound) | 0.70–0.95 |
| ρA (between them) | 0.70–0.95 |
| AVE | ≥ 0.50 |

In [ ]:
show(rep["quality"][["alpha", "rho_A", "rho_C", "AVE"]])

**Interpretation.** All three constructs are reliable (α, ρA and ρC between 0.83 and 0.93) and
their AVE exceeds 0.50, so each construct explains more than half of its indicators' variance.
None of the values exceeds 0.95, which would suggest redundant item wording.

### Discriminant validity: HTMT

The heterotrait–monotrait ratio estimates the correlation between two constructs as if they
were measured without error. Values below 0.85 (conservative) or 0.90 (liberal, for
conceptually similar constructs) support discriminant validity.

In [ ]:
FIGURES["htmt.png"] = heatmap_table(rep["htmt"], "HTMT (cells above 0.85 are outlined)", threshold=0.85)
plt.show()
show(rep["htmt"])

**Interpretation.** The largest HTMT is well below 0.85, so the three constructs are empirically
distinct. (In the population, the latent correlations are 0.60 for PEOU–PU, 0.50 for PEOU–BI
and 0.62 for PU–BI.)

## 7. Structural model: paths, bootstrap and mediation

In [ ]:
paths = rep["paths"]
show(paths.drop(columns=["key", "from", "to"]), 4)
FIGURES["paths.png"] = coefficient_plot(paths, label_col="path")
plt.show()

**Interpretation.** All three paths are positive and their 95% percentile intervals exclude
zero, so H1–H3 are supported *in this sample*. PEOU → PU is the strongest path. The direct
path PEOU → BI is small (about 0.13). Compare with the population values: PEOU → PU (0.50 vs
0.60) and PEOU → BI (0.13 vs 0.20) are smaller, because classic PLS composites contain
measurement error, which attenuates the correlations between constructs. PU → BI happens to
land on its population value (0.50): attenuation also weakens the control for PEOU in the BI
equation. Sampling error and the discretisation into 7 categories add further differences.
The p values come from a t distribution with B − 1 degrees of freedom and
the bootstrap standard error; the percentile CI is the primary decision rule here.

### R², f² and collinearity

In [ ]:
r2 = rep["quality"][["R²", "R² adj."]].dropna()
show(r2)
print("f² (rows: predictor, columns: outcome):")
show(rep["f2"])
print("Structural VIF (rows: predictor, columns: outcome):")
show(rep["vif"])

**Interpretation.** PEOU explains about a quarter of the variance of PU (R² = 0.25), and PEOU
and PU together explain about a third of BI (R² = 0.33). Effect sizes f² of 0.02, 0.15 and
0.35 are often read as small, medium and large: PEOU → PU (0.33) and PU → BI (0.28) are
medium-to-large, whereas dropping the direct path PEOU → BI would change R²(BI) only
marginally (f² ≈ 0.02). Structural VIFs are far below 3, so collinearity between PEOU and PU
does not distort the BI equation.

### Mediation: direct, indirect and total effects (H4)

In [ ]:
eff = rep["effects"]
show(eff[eff["to"] == "BI"].drop(columns=["key", "from", "to"]), 4)
direct = paths.set_index("path").loc[f"PEOU {ARROW} BI", "est"]
indirect = eff[(eff["effect"] == "total indirect") & (eff["from"] == "PEOU") & (eff["to"] == "BI")].iloc[0]
total = eff[(eff["effect"] == "total") & (eff["from"] == "PEOU") & (eff["to"] == "BI")].iloc[0]
print(f"total {total['est']:.4f} = direct {direct:.4f} + indirect {indirect['est']:.4f} "
      f"(difference {total['est'] - direct - indirect['est']:.1e})")
print(f"VAF (indirect / total) = {indirect['est'] / total['est']:.1%}")

In [ ]:
# The result keeps every bootstrap draw of the paths and of each specific indirect effect.
# Here the only route from PEOU to BI via a mediator is PEOU -> PU -> BI.
specific = eff[(eff["effect"] == "specific indirect") & (eff["from"] == "PEOU") & (eff["to"] == "BI")].iloc[0]
draws = result["bootstrap"]["draws"][specific["key"]]
FIGURES["bootstrap_indirect.png"] = bootstrap_histogram(
    draws, specific["est"], (specific["ciLower"], specific["ciUpper"]), f"indirect effect {specific['route']}")
plt.show()

**Interpretation.** The indirect effect PEOU → PU → BI is the product of the two paths. Its
bootstrap distribution is clearly away from zero and its 95% interval excludes zero, so H4 is
supported: PU mediates the PEOU–BI relationship. Because the direct effect is also significant,
the mediation is *partial* (complementary: direct and indirect effects have the same sign).
The share of the total effect that runs through PU (VAF, variance accounted for) is about two
thirds. The
bootstrap histogram is the reason we test indirect effects by bootstrap: a product of two
coefficients is not normally distributed, so a symmetric "estimate ± 1.96 SE" interval would
be inappropriate.

A causal reading ("ease of use *increases* intention *because* it increases usefulness") needs
more than this design: the data are synthetic and cross-sectional, and alternative models
(e.g. reversed arrows) can fit as well. Report the effects as consistent with mediation, not
as proof of it.

### Estimated model

In [ ]:
sig = {k: v for k, v in zip(path_dict(paths, model).keys(), paths["significant (95% CI excludes 0)"])}
r2_by_id = {c["id"]: rep["quality"].loc[c["name"], "R²"] for c in model["constructs"]
            if pd.notna(rep["quality"].loc[c["name"], "R²"])}
loadings_by_id = {f"i_{r['indicator']}": r["loading"] for _, r in rep["loadings"].iterrows()}
FIGURES["model_estimated.png"] = draw_model(
    model, title="TAM: estimated model (classic PLS-SEM, standardised)", paths=path_dict(paths, model),
    r2=r2_by_id, loadings=loadings_by_id, significant_paths=sig,
    note="Path labels: coefficients; dashed grey = 95% bootstrap CI includes 0. SYNTHETIC data.")
plt.show()

### Global fit (optional in PLS-SEM)

SRMR below 0.08 is a common guideline for an acceptable approximate fit of the *composite*
model. PLS-SEM is primarily a prediction-oriented method, and model fit criteria for PLS are
still debated; report SRMR as supplementary information.

In [ ]:
show(rep["fit"], 4)

## 8. Export

In [ ]:
EXPORT_TABLES = {
    "loadings.csv": rep["loadings"], "reliability_validity.csv": rep["quality"], "htmt.csv": rep["htmt"],
    "paths.csv": rep["paths"], "effects.csv": rep["effects"], "f2.csv": rep["f2"], "fit.csv": rep["fit"],
}
WORD_TABLES = [
    ("Table 1. Reliability and convergent validity",
     rep["quality"][["alpha", "rho_A", "rho_C", "AVE"]].rename_axis("Construct").reset_index(),
     "SYNTHETIC data (N = 500). Classic PLS-SEM, SEM Lab engine.", 3),
    ("Table 2. Structural paths with 95% percentile bootstrap intervals",
     paths[["path", "est", "se", "ciLower", "ciUpper"]].rename(
         columns={"path": "Path", "est": "β", "se": "SE", "ciLower": "2.5%", "ciUpper": "97.5%"}),
     "B = 5,000 bootstrap draws, seed 20261103. SYNTHETIC data.", 3),
    ("Table 3. Effects of PEOU on BI",
     eff[(eff["from"] == "PEOU") & (eff["to"] == "BI")][["effect", "route", "est", "ciLower", "ciUpper"]].rename(
         columns={"effect": "Effect", "route": "Route", "est": "Estimate", "ciLower": "2.5%", "ciUpper": "97.5%"}),
     "Indirect effect = product of the paths PEOU → PU and PU → BI.", 3),
]

## Export: CSV, figures and Word-ready tables

The next cell writes everything to the folder `exports/` and zips it:

- `data.csv` (the synthetic data), `model.json`, `analysis.json` and `result.json` (the full
  result in SEM Lab's result format, the same JSON the app produces);
- one CSV per results table (unrounded numbers);
- every figure as PNG (200 dpi);
- `tables_word.html`: the main tables as APA-style tables with inline styles. Open the file
  in a browser, select a table, copy and paste it into Word or Google Docs: it arrives as an
  editable table. The same tables are shown below the cell, so you can also copy them
  straight from the notebook output.

On Colab, the zip file downloads automatically (allow pop-ups if your browser asks). Outside
Colab, find it in the notebook's working directory.

In [ ]:
import os
import shutil

from IPython.display import HTML

EXPORT_DIR = "exports"
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
ensure_dir(EXPORT_DIR)
shutil.copy("data.csv", EXPORT_DIR)
shutil.copy("model.json", EXPORT_DIR)
write_json(os.path.join(EXPORT_DIR, "analysis.json"), analysis)
write_json(os.path.join(EXPORT_DIR, "result.json"), result)
for name, table in EXPORT_TABLES.items():
    table.to_csv(os.path.join(EXPORT_DIR, name), index=not isinstance(table.index, pd.RangeIndex))
for name, fig in FIGURES.items():
    save_figure(fig, EXPORT_DIR, name)
save_word_tables(os.path.join(EXPORT_DIR, "tables_word.html"), WORD_TABLES, "01_tam_pls_sem: tables")
zip_path = zip_folder(EXPORT_DIR, "01_tam_pls_sem_exports.zip")
print("\n".join(sorted(os.listdir(EXPORT_DIR))))
print(f"\n{zip_path}: {os.path.getsize(zip_path) // 1024} KiB")
offer_download(zip_path)
HTML("".join(word_table_html(df_, title, note, decimals) + "<br>" for title, df_, note, decimals in WORD_TABLES))

## 9. Summary and questions

- The reflective measurement model is sound: loadings > 0.708, reliability 0.80–0.93, AVE > 0.5,
  HTMT < 0.85.
- PEOU → PU, PU → BI and PEOU → BI are positive with bootstrap intervals that exclude zero.
- PU partially mediates the effect of PEOU on BI; most of the total effect is indirect.

**Questions**

1. Why is the bootstrap interval of the indirect effect not symmetric around the estimate?
2. The population path PEOU → PU is 0.60 but the estimate is about 0.50. Name two reasons.
3. Which number tells you that collinearity between PEOU and PU is not a problem for BI?
4. Would you call the mediation "full" or "partial"? Which result decides it?

**Challenges**

1. Re-run the estimation with `"repetitions": 1000` and compare the intervals with B = 5,000.
   Then change the seed. Which numbers change, and by how much? What does that tell you about
   reporting the number of draws and the seed?
2. Change `analysis["method"]` to `"plsc"` (consistent PLS) and re-run Sections 5–7. Compare the
   paths with the population values in Section 2 and explain the difference from classic PLS.